# Aegisub AI KFX Studio

Automate **Anime Romaji Karaoke (KFX)** creation in Google Colab using a clean, two-stage modular pipeline:

| Workflow Path | Who It Is For | Cells to Run |
| :--- | :--- | :--- |
| **Path A: Full Pipeline (Stage 1 -> Stage 2)** | You have raw audio/video (`.mp3`, `.wav`, `.flac`, `.mp4`, `.mkv`) + line-timed romaji lyrics (`.ass`, `.srt`, or pasted timestamps) | **Cell 0** -> **Stage 1 (1.1 - 1.4)** -> **Stage 2 (2.1 - 2.5)** |
| **Path B: KFX Only (Skip Stage 1)** | You **already have** a `\k` / `\kf` / `\ko`-timed `.ass` file (or want to try the built-in instant demo) | **Cell 0** -> **Skip directly to Stage 2 (2.1 - 2.5)** |

### Recommended Google Colab Runtime
- **Path A (Stage 1 + Stage 2):** Enable **GPU** (`Runtime -> Change runtime type -> T4 GPU`) for fast Demucs vocal separation and forced alignment.
- **Path B (Stage 2 Only):** Runs fast on **CPU** (no GPU required).

### Table of Contents
- **0. Core Setup** — Environment detection, optional Google Drive mount (`MOUNT_GOOGLE_DRIVE`), core Python packages (`pillow`, `fonttools`, `google-genai`), and workspace setup.
- **Stage 1 (Optional):**
  - **1.1** Install Demucs & forced-alignment dependencies (`torchaudio` MMS_FA or `whisperx`)
  - **1.2** Upload or pick audio/video from Google Drive (`EXISTING_PATH_OR_MOUNTED_PATH`) + paste/upload/select line-timed romaji lyrics
  - **1.3** Isolate vocals with Meta Demucs (`htdemucs`)
  - **1.4** Word-level forced alignment + Japanese Mora Syllabifier -> `auto_timed_karaoke.ass`
- **Stage 2 (Core KFX Engine):**
  - **2.1** Load timed `.ass` + `.ttf`/`.otf` font file & inspect numbered lyrics in the **Line Inspector Table**
  - **2.2** FreeType / `OS/2` glyph layout & ASS vector math engine (`left`, `center`, `right`, `top`, `middle`, `bottom`, `width`)
  - **2.3** **KFX Rule & Palette Builder** (`ipywidgets` UI + JSON config) supporting **Mode A (Shared KFX Animation + Line-Range Colors)** & **Mode B (Unique KFX Animation + Colors per Line Range)**, 4-color palettes (`c`, `c`, `c`, `\p1`), Gemini AI prompts & 5 offline presets
  - **2.4** Unified multi-section `.ass` synthesis (`final_kfx_karaoke.ass`)
  - **2.5** Render per-section visual keyframe/video preview (`ffmpeg` + `libass`) & download `final_kfx_karaoke.ass`

In [ ]:
#@title 0. Core Setup (Required for Both Stages)
MOUNT_GOOGLE_DRIVE = False #@param {type:"boolean"}
DRIVE_MOUNT_POINT = "/content/drive" #@param {type:"string"}

import os
import sys
import json
import math
import re
import shutil
import subprocess
from pathlib import Path

try:
    import google.colab  # type: ignore
    from google.colab import files as colab_files  # type: ignore
    from google.colab import drive as colab_drive  # type: ignore
    IN_COLAB = True
except ImportError:
    colab_files = None
    colab_drive = None
    IN_COLAB = False

if IN_COLAB:
    print("Installing core Stage 2 packages (pillow, fonttools, google-genai, ipywidgets)...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-U", "pillow", "fonttools", "google-genai>=3.2.0", "ipywidgets"],
        check=True,
    )
    need_ffmpeg = shutil.which("ffmpeg") is None
    need_liberation = not Path("/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf").is_file()
    if need_ffmpeg or need_liberation:
        print("Installing ffmpeg and Liberation/DejaVu fonts...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        pkgs = ["fonts-liberation", "fonts-dejavu-core"] + (["ffmpeg"] if need_ffmpeg else [])
        subprocess.run(["apt-get", "-qq", "install", "-y", *pkgs], check=False)

WORK_DIR = Path(
    os.environ.get("KFX_WORK_DIR", "/content/kfx_studio" if IN_COLAB else "./kfx_studio_work")
).resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)


def check_ffmpeg_libass():
    ffmpeg_bin = shutil.which("ffmpeg")
    if not ffmpeg_bin:
        return False, False
    res = subprocess.run([ffmpeg_bin, "-hide_banner", "-filters"], capture_output=True, text=True)
    has_ass = bool(re.search(r"\bass\b", res.stdout + res.stderr))
    return True, has_ass


def detect_gpu_info():
    if shutil.which("nvidia-smi"):
        res = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            capture_output=True,
            text=True,
        )
        if res.returncode == 0 and res.stdout.strip():
            return True, res.stdout.strip()
    try:
        import torch  # type: ignore
        if torch.cuda.is_available():
            return True, torch.cuda.get_device_name(0)
    except Exception:
        pass
    return False, "CPU only"


def find_system_font(fontname: str = "", bold: bool = False) -> str | None:
    req = (fontname or "").strip().lower()
    win_dir = Path(r"C:\Windows\Fonts")
    if req and win_dir.is_dir():
        win_map = {
            "arial": ("arial.ttf", "arialbd.ttf"),
            "calibri": ("calibri.ttf", "calibrib.ttf"),
            "segoe ui": ("segoeui.ttf", "segoeuib.ttf"),
            "trebuchet ms": ("trebuc.ttf", "trebucbd.ttf"),
            "tahoma": ("tahoma.ttf", "tahomabd.ttf"),
            "georgia": ("georgia.ttf", "georgiab.ttf"),
            "times new roman": ("times.ttf", "timesbd.ttf"),
            "verdana": ("verdana.ttf", "verdanab.ttf"),
            "consolas": ("consola.ttf", "consolab.ttf"),
            "comic sans ms": ("comic.ttf", "comicbd.ttf"),
            "candara": ("candara.ttf", "candarab.ttf"),
            "corbel": ("corbel.ttf", "corbelb.ttf"),
            "impact": ("impact.ttf", "impact.ttf"),
        }
        if req in win_map:
            cand = win_dir / win_map[req][1 if bold else 0]
            if cand.is_file():
                return str(cand)

    if req and shutil.which("fc-match"):
        query_family = "Liberation Sans" if req == "arial" else fontname.strip()
        weight = "bold" if bold else "regular"
        try:
            res = subprocess.run(
                ["fc-match", "-f", "%{file}\n%{family}", f"{query_family}:weight={weight}"],
                capture_output=True,
                text=True,
            )
            lines = [ln.strip() for ln in res.stdout.splitlines() if ln.strip()]
            if len(lines) >= 2:
                f_path, matched_fam = Path(lines[0]), lines[1].lower()
                if f_path.is_file() and f_path.suffix.lower() in (".ttf", ".otf", ".ttc"):
                    if req in matched_fam or (req == "arial" and "liberation sans" in matched_fam):
                        return str(f_path)
        except Exception:
            pass

    if bold:
        candidates = [
            "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf",
            "/usr/share/fonts/truetype/noto/NotoSans-Bold.ttf",
            "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
            "/usr/share/fonts/opentype/noto/NotoSansCJK-Bold.ttc",
            r"C:\Windows\Fonts\arialbd.ttf",
            r"C:\Windows\Fonts\segoeui.ttf",
            "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf",
            "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
            r"C:\Windows\Fonts\arial.ttf",
            "/System/Library/Fonts/Supplemental/Arial Bold.ttf",
            "/System/Library/Fonts/Supplemental/Arial.ttf",
        ]
    else:
        candidates = [
            "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf",
            "/usr/share/fonts/truetype/noto/NotoSans-Regular.ttf",
            "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
            r"C:\Windows\Fonts\arial.ttf",
            r"C:\Windows\Fonts\segoeui.ttf",
            "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf",
            "/usr/share/fonts/truetype/noto/NotoSans-Bold.ttf",
            "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
            r"C:\Windows\Fonts\arialbd.ttf",
            "/System/Library/Fonts/Supplemental/Arial.ttf",
        ]

    for cand in candidates:
        if Path(cand).is_file():
            return cand
    for root in ("/usr/share/fonts", "/usr/local/share/fonts"):
        root_p = Path(root)
        if root_p.exists():
            for ext in ("*.ttf", "*.otf"):
                found = sorted(root_p.rglob(ext))
                if found:
                    return str(found[0])
    return None


def upload_to_dir(target_dir: Path, label: str = "file") -> list[Path]:
    target_dir.mkdir(parents=True, exist_ok=True)
    if IN_COLAB and colab_files is not None:
        print(f"Please choose a {label} to upload...")
        uploaded = colab_files.upload()
        saved = []
        for fname, data in uploaded.items():
            dest = target_dir / Path(fname).name
            dest.write_bytes(data)
            saved.append(dest)
            print(f"Saved: {dest} ({len(data):,} bytes)")
        return saved
    raise RuntimeError(
        f"Not running in Google Colab: set a local file path instead of interactive {label} upload."
    )


def ensure_drive_mounted(mount_point: str | None = None) -> Path:
    mp = Path(mount_point or globals().get("DRIVE_MOUNT_POINT", "/content/drive"))
    mydrive = mp / "MyDrive"
    if IN_COLAB and colab_drive is not None and not mydrive.exists():
        print(f"Mounting Google Drive at {mp} ...")
        colab_drive.mount(str(mp))
    return mydrive if mydrive.exists() else mp


def list_matching_files(
    search_dir: Path,
    exts: set[str] | None = None,
    max_depth: int = 2,
    limit: int = 40,
) -> list[Path]:
    search_dir = Path(search_dir)
    if not search_dir.is_dir():
        return []
    base_parts = len(search_dir.resolve().parts)
    matches: list[Path] = []
    for root, dirs, files in os.walk(search_dir):
        cur = Path(root)
        depth = len(cur.resolve().parts) - base_parts
        dirs[:] = sorted(d for d in dirs if not d.startswith("."))
        if depth >= max_depth:
            dirs[:] = []
        for fn in sorted(files):
            if fn.startswith("."):
                continue
            fp = cur / fn
            if exts is None or fp.suffix.lower() in exts:
                matches.append(fp)
                if len(matches) >= limit:
                    return matches
    return matches


def _show_drive_picker_widget(matches: list[Path], target_field: str, label: str):
    if not IN_COLAB or not matches:
        return
    try:
        import ipywidgets as widgets  # type: ignore
        from IPython.display import display  # type: ignore

        dd = widgets.Dropdown(
            options=[(str(p), str(p)) for p in matches],
            description=f"{label}:",
            layout=widgets.Layout(width="90%"),
        )
        out = widgets.Output()

        def _on_change(change):
            with out:
                out.clear_output()
                print(f"Copy into {target_field}: {change['new']}")

        dd.observe(_on_change, names="value")
        display(dd, out)
        with out:
            print(f"Copy into {target_field}: {dd.value}")
    except Exception:
        pass


def resolve_input_path(
    path_str: str,
    allowed_exts: set[str] | None = None,
    label: str = "file",
    target_field: str = "path",
) -> Path | None:
    raw = (path_str or "").strip().strip('"').strip("'")
    mp_str = str(globals().get("DRIVE_MOUNT_POINT", "/content/drive")).rstrip("/\\")
    drive_prefixes = (mp_str, "/content/drive", "MyDrive", "/MyDrive", "Shared drives", "/Shared drives")

    if raw and any(raw.startswith(pref) for pref in drive_prefixes):
        if IN_COLAB:
            ensure_drive_mounted(mp_str)
        if raw.startswith(("MyDrive", "/MyDrive", "Shared drives", "/Shared drives")):
            raw = str(Path(mp_str) / raw.lstrip("/\\"))

    if not raw:
        search_root = ensure_drive_mounted(mp_str) if IN_COLAB else WORK_DIR
        matches = list_matching_files(search_root, allowed_exts)
        print(f"No {label} path provided in {target_field}.")
        if matches:
            print(f"Matching {label} files found under {search_root} (paste one into {target_field}):")
            for p in matches[:20]:
                print(f"  {p}")
            _show_drive_picker_widget(matches, target_field, label)
        else:
            print(f"No matching {label} files found under {search_root}. Enter a full file or folder path in {target_field}.")
        return None

    p = Path(raw).expanduser()
    if p.is_dir():
        matches = list_matching_files(p, allowed_exts)
        print(f"{target_field} points to a folder: {p}")
        if matches:
            print(f"Matching {label} files inside {p} (paste one into {target_field}):")
            for m in matches[:25]:
                print(f"  {m}")
            _show_drive_picker_widget(matches, target_field, label)
        else:
            print(f"No matching {label} files found inside {p}.")
        return None

    if p.is_file():
        return p

    nearby = list_matching_files(p.parent, allowed_exts, max_depth=1, limit=10) if p.parent.is_dir() else []
    hint = ""
    if nearby:
        hint = "\nNearby matching files:\n  " + "\n  ".join(str(x) for x in nearby)
    raise FileNotFoundError(f"{label} file not found: {p}{hint}")


def download_file_compat(path: Path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Cannot download missing file: {path}")
    if IN_COLAB and colab_files is not None:
        colab_files.download(str(path))
    else:
        print(f"[Local environment] File ready at: {path}")


HAS_FFMPEG, HAS_LIBASS = check_ffmpeg_libass()
HAS_GPU, GPU_DESC = detect_gpu_info()
MOUNTED_DRIVE_PATH = ensure_drive_mounted(DRIVE_MOUNT_POINT) if MOUNT_GOOGLE_DRIVE else None

print(f"Environment   : {'Google Colab' if IN_COLAB else 'Local Python'}")
print(f"Workspace Dir : {WORK_DIR}")
print(f"Compute       : {GPU_DESC}")
print(f"FFmpeg/libass : ffmpeg={HAS_FFMPEG}, libass={HAS_LIBASS}")
if MOUNTED_DRIVE_PATH is not None:
    print(f"Google Drive  : {MOUNTED_DRIVE_PATH}")

---
# Stage 1 (Optional): AI Vocal Separation & Romaji Syllable Auto-Timer

> **Already have a `\k` / `\kf` / `\ko`-timed `.ass` file?**
> Skip Cells **1.1 – 1.4** and jump directly to **Stage 2 (Cell 2.1)**!

### What Stage 1 Does
1. **Extracts clean audio** from `.mp3`, `.wav`, `.flac`, `.mp4`, `.mkv`, etc. via `ffmpeg`.
2. **Isolates the vocal stem** using Meta's **Demucs (`htdemucs`)** so guitars, drums, and synths do not throw off lyric alignment.
3. **Aligns words to the isolated vocal track** using `torchaudio` `MMS_FA` (recommended) or `whisperx`.
4. **Splits Japanese romaji words into exact morae** (`ka/ki/ku/ke/ko`, `sha/chu/ryo`, standalone `n`, double consonants `kk/tt/ss/pp/tch/cch`, long vowels) and distributes word duration proportionally into clean `{\kXX}` tags, saving `auto_timed_karaoke.ass`.

In [ ]:
#@title 1.1 Stage 1 Setup: Install Demucs & Forced Alignment Dependencies
ALIGNMENT_ENGINE = "torchaudio_mms_fa" #@param ["torchaudio_mms_fa", "whisperx"]
INSTALL_STAGE1_DEPS = True #@param {type:"boolean"}

if IN_COLAB and INSTALL_STAGE1_DEPS:
    pkgs = ["demucs", "soundfile", "scipy"]
    if ALIGNMENT_ENGINE == "whisperx":
        pkgs.append("whisperx")
    print(f"Installing Stage 1 dependencies: {', '.join(pkgs)} ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
else:
    print("Skipping pip install (local mode or INSTALL_STAGE1_DEPS=False).")

try:
    import torch  # type: ignore
    import torchaudio  # type: ignore
    STAGE1_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"PyTorch {torch.__version__} | torchaudio {torchaudio.__version__} | Device: {STAGE1_DEVICE}")
    if STAGE1_DEVICE == "cpu":
        print("Tip: Enabling a Colab T4 GPU dramatically speeds up Demucs vocal separation.")
except ImportError as exc:
    STAGE1_DEVICE = "cpu"
    print(f"Note: PyTorch/torchaudio not installed yet ({exc}). Install them before running Cells 1.3–1.4.")

In [ ]:
#@title 1.2 Stage 1 Inputs: Upload / Mount Drive Audio & Line-Timed Romaji Lyrics
AUDIO_SOURCE = "Upload Audio/Video File" #@param ["Upload Audio/Video File", "Use Existing / Mounted Drive Path"]
EXISTING_PATH_OR_MOUNTED_PATH = "" #@param {type:"string"}
LYRICS_INPUT_MODE = "Paste Line-Timed Romaji Below" #@param ["Paste Line-Timed Romaji Below", "Upload Line-Timed .ass or .srt File", "Use Existing / Mounted Drive Lyrics File", "Use Built-in Sample Romaji (Demo Format)"]
LYRICS_FILE_PATH = "" #@param {type:"string"}

# Supported pasted formats:
#   1) [MM:SS.xx - MM:SS.xx] romaji lyric text
#   2) [HH:MM:SS.xx - HH:MM:SS.xx] romaji lyric text
#   3) H:MM:SS.cc,H:MM:SS.cc,romaji lyric text
#   4) MM:SS.xx --> MM:SS.xx | romaji lyric text
#   5) LRC: [MM:SS.xx] romaji lyric text
PASTED_LYRICS = """
[00:01.20 - 00:05.60] Hikari wo sagashite yozora wo kakete yuku
[00:05.80 - 00:10.40] Zutto kawaranai omoi wo dakishimete
[00:10.60 - 00:15.20] Kon'ya hoshi no shita de chikau yo Starlight
[00:15.40 - 00:19.80] Arigatou itsumo soba ni ite kurete
""".strip()

SAMPLE_STAGE1_LYRICS = PASTED_LYRICS
SUPPORTED_MEDIA_EXTS = {".mp3", ".wav", ".flac", ".ogg", ".m4a", ".aac", ".mp4", ".mkv", ".webm", ".mov"}


def parse_timestamp_to_ms(ts: str) -> int:
    ts = ts.strip().replace(",", ".")
    parts = ts.split(":")
    if len(parts) == 2:
        mins = int(parts[0])
        secs = float(parts[1])
        return int(round((mins * 60.0 + secs) * 1000))
    if len(parts) == 3:
        hrs = int(parts[0])
        mins = int(parts[1])
        secs = float(parts[2])
        return int(round((hrs * 3600.0 + mins * 60.0 + secs) * 1000))
    raise ValueError(f"Unrecognized timestamp format: {ts!r}")


def strip_ass_tags(text: str) -> str:
    cleaned = re.sub(r"\{[^}]*\}", "", text)
    cleaned = cleaned.replace("\\N", " ").replace("\\n", " ").replace("\\h", " ")
    return re.sub(r"\s+", " ", cleaned).strip()


def parse_stage1_ass(raw_text: str) -> list[dict]:
    lines = []
    in_events = False
    fields = ["Layer", "Start", "End", "Style", "Name", "MarginL", "MarginR", "MarginV", "Effect", "Text"]
    for raw_line in raw_text.splitlines():
        line = raw_line.strip()
        if not line:
            continue
        if line.startswith("[") and line.endswith("]"):
            in_events = line.lower() == "[events]"
            continue
        if not in_events:
            continue
        if line.lower().startswith("format:"):
            fields = [f.strip() for f in line.split(":", 1)[1].split(",")]
            continue
        if line.startswith("Dialogue:"):
            payload = line.split(":", 1)[1].lstrip()
            cols = payload.split(",", len(fields) - 1)
            if len(cols) < len(fields):
                continue
            row = dict(zip(fields, cols))
            text = strip_ass_tags(row.get("Text", ""))
            if not text:
                continue
            start_ms = parse_timestamp_to_ms(row["Start"])
            end_ms = parse_timestamp_to_ms(row["End"])
            if end_ms > start_ms:
                lines.append({"start_ms": start_ms, "end_ms": end_ms, "text": text, "style": row.get("Style", "Romaji").strip()})
    return lines


def parse_stage1_srt(raw_text: str) -> list[dict]:
    blocks = re.split(r"\r?\n\s*\r?\n", raw_text.strip())
    lines = []
    ts_re = re.compile(r"(\d{1,2}:\d{2}:\d{2}[,.]\d{1,3})\s*-->\s*(\d{1,2}:\d{2}:\d{2}[,.]\d{1,3})")
    for block in blocks:
        rows = [r.strip() for r in block.splitlines() if r.strip()]
        if not rows:
            continue
        match = None
        text_start = 0
        for idx, r in enumerate(rows[:2]):
            m = ts_re.search(r)
            if m:
                match = m
                text_start = idx + 1
                break
        if not match:
            continue
        start_ms = parse_timestamp_to_ms(match.group(1))
        end_ms = parse_timestamp_to_ms(match.group(2))
        text = " ".join(rows[text_start:])
        text = re.sub(r"<[^>]+>", "", text)
        text = strip_ass_tags(text)
        if text and end_ms > start_ms:
            lines.append({"start_ms": start_ms, "end_ms": end_ms, "text": text, "style": "Romaji"})
    return lines


def parse_stage1_pasted(raw_text: str, total_audio_ms: int | None = None) -> list[dict]:
    lines = []
    lrc_pending = []
    range_re = re.compile(
        r"^\[?\s*(\d{1,2}:\d{2}(?::\d{2})?(?:[.,]\d{1,3})?)\s*(?:-|–|->|-->)\s*(\d{1,2}:\d{2}(?::\d{2})?(?:[.,]\d{1,3})?)\s*\]?\s*\|?\s*(.+)$"
    )
    csv_re = re.compile(
        r"^(\d{1,2}:\d{2}:\d{2}(?:[.,]\d{1,3})?)\s*,\s*(\d{1,2}:\d{2}:\d{2}(?:[.,]\d{1,3})?)\s*,\s*(.+)$"
    )
    lrc_re = re.compile(r"^\[(\d{1,2}:\d{2}(?:[.,]\d{1,3})?)\]\s*(.+)$")

    for raw_line in raw_text.splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        m_range = range_re.match(line)
        if m_range:
            s_ms = parse_timestamp_to_ms(m_range.group(1))
            e_ms = parse_timestamp_to_ms(m_range.group(2))
            txt = strip_ass_tags(m_range.group(3))
            if txt and e_ms > s_ms:
                lines.append({"start_ms": s_ms, "end_ms": e_ms, "text": txt, "style": "Romaji"})
            continue
        m_csv = csv_re.match(line)
        if m_csv:
            s_ms = parse_timestamp_to_ms(m_csv.group(1))
            e_ms = parse_timestamp_to_ms(m_csv.group(2))
            txt = strip_ass_tags(m_csv.group(3))
            if txt and e_ms > s_ms:
                lines.append({"start_ms": s_ms, "end_ms": e_ms, "text": txt, "style": "Romaji"})
            continue
        m_lrc = lrc_re.match(line)
        if m_lrc:
            s_ms = parse_timestamp_to_ms(m_lrc.group(1))
            txt = strip_ass_tags(m_lrc.group(2))
            if txt:
                lrc_pending.append((s_ms, txt))
            continue
        raise ValueError(
            f"Could not parse line timestamp format: {line!r}\n"
            "Expected '[MM:SS.xx - MM:SS.xx] text', 'H:MM:SS.cc,H:MM:SS.cc,text', or LRC '[MM:SS.xx] text'."
        )

    if lrc_pending:
        lrc_pending.sort(key=lambda item: item[0])
        for idx, (s_ms, txt) in enumerate(lrc_pending):
            if idx + 1 < len(lrc_pending):
                e_ms = max(s_ms + 300, lrc_pending[idx + 1][0])
            else:
                e_ms = total_audio_ms if (total_audio_ms and total_audio_ms > s_ms) else s_ms + 4500
            lines.append({"start_ms": s_ms, "end_ms": e_ms, "text": txt, "style": "Romaji"})
    return lines


def get_audio_duration_ms(audio_path: Path) -> int:
    import wave
    with wave.open(str(audio_path), "rb") as wf:
        return int(round(wf.getnframes() / float(wf.getframerate()) * 1000.0))


def convert_media_to_wav(src_path: Path, dst_wav: Path) -> Path:
    if not src_path.is_file():
        raise FileNotFoundError(f"Audio/video file not found: {src_path}")
    cmd = [
        "ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
        "-i", str(src_path), "-vn", "-ac", "2", "-ar", "44100", "-c:a", "pcm_s16le", str(dst_wav),
    ]
    subprocess.run(cmd, check=True)
    return dst_wav


def format_ms_clock(ms: int) -> str:
    cs = max(0, int(round(ms / 10.0)))
    h, rem = divmod(cs, 360000)
    m, rem = divmod(rem, 6000)
    s, cc = divmod(rem, 100)
    return f"{h:d}:{m:02d}:{s:02d}.{cc:02d}"


# --- Acquire Audio/Video ---
STAGE1_DIR = WORK_DIR / "stage1"
STAGE1_DIR.mkdir(parents=True, exist_ok=True)
STAGE1_RAW_MEDIA = None
STAGE1_AUDIO_WAV = STAGE1_DIR / "input_audio.wav"
STAGE1_AUDIO_MS = None

if AUDIO_SOURCE == "Upload Audio/Video File":
    if IN_COLAB:
        uploaded_media = upload_to_dir(STAGE1_DIR, label="audio/video file (.mp3, .wav, .flac, .mp4, .mkv)")
        STAGE1_RAW_MEDIA = uploaded_media[0]
    else:
        print("Local mode: set AUDIO_SOURCE='Use Existing / Mounted Drive Path' and EXISTING_PATH_OR_MOUNTED_PATH to load media.")
else:
    STAGE1_RAW_MEDIA = resolve_input_path(
        EXISTING_PATH_OR_MOUNTED_PATH,
        SUPPORTED_MEDIA_EXTS,
        label="audio/video",
        target_field="EXISTING_PATH_OR_MOUNTED_PATH",
    )

if STAGE1_RAW_MEDIA is not None:
    ext = STAGE1_RAW_MEDIA.suffix.lower()
    if ext not in SUPPORTED_MEDIA_EXTS:
        print(f"Warning: {ext} is uncommon; attempting ffmpeg extraction anyway.")
    convert_media_to_wav(STAGE1_RAW_MEDIA, STAGE1_AUDIO_WAV)
    STAGE1_AUDIO_MS = get_audio_duration_ms(STAGE1_AUDIO_WAV)
    print(f"Extracted 44.1kHz stereo WAV: {STAGE1_AUDIO_WAV} ({STAGE1_AUDIO_MS/1000:.2f}s)")

# --- Parse Line-Timed Lyrics ---
if LYRICS_INPUT_MODE == "Upload Line-Timed .ass or .srt File":
    lyric_files = upload_to_dir(STAGE1_DIR, label="line-timed .ass or .srt file")
    lyric_path = lyric_files[0]
    raw_lyrics_text = lyric_path.read_text(encoding="utf-8-sig", errors="replace")
    if lyric_path.suffix.lower() == ".ass":
        STAGE1_LINES = parse_stage1_ass(raw_lyrics_text)
    elif lyric_path.suffix.lower() == ".srt":
        STAGE1_LINES = parse_stage1_srt(raw_lyrics_text)
    else:
        STAGE1_LINES = parse_stage1_pasted(raw_lyrics_text, STAGE1_AUDIO_MS)
elif LYRICS_INPUT_MODE == "Use Existing / Mounted Drive Lyrics File":
    lyric_path = resolve_input_path(
        LYRICS_FILE_PATH,
        {".ass", ".srt", ".lrc", ".txt"},
        label="lyrics",
        target_field="LYRICS_FILE_PATH",
    )
    if lyric_path is not None:
        raw_lyrics_text = lyric_path.read_text(encoding="utf-8-sig", errors="replace")
        if lyric_path.suffix.lower() == ".ass":
            STAGE1_LINES = parse_stage1_ass(raw_lyrics_text)
        elif lyric_path.suffix.lower() == ".srt":
            STAGE1_LINES = parse_stage1_srt(raw_lyrics_text)
        else:
            STAGE1_LINES = parse_stage1_pasted(raw_lyrics_text, STAGE1_AUDIO_MS)
    else:
        STAGE1_LINES = []
elif LYRICS_INPUT_MODE == "Use Built-in Sample Romaji (Demo Format)":
    STAGE1_LINES = parse_stage1_pasted(SAMPLE_STAGE1_LYRICS, STAGE1_AUDIO_MS)
else:
    STAGE1_LINES = parse_stage1_pasted(PASTED_LYRICS, STAGE1_AUDIO_MS)

print(f"\nParsed {len(STAGE1_LINES)} timed romaji lines:")
print(f"{'#':>3}  {'Start':>10}  {'End':>10}  {'Dur(s)':>6}  Text")
print("-" * 72)
for idx, ln in enumerate(STAGE1_LINES):
    dur_s = (ln["end_ms"] - ln["start_ms"]) / 1000.0
    print(f"{idx:>3}  {format_ms_clock(ln['start_ms']):>10}  {format_ms_clock(ln['end_ms']):>10}  {dur_s:>6.2f}  {ln['text']}")

In [ ]:
#@title 1.3 Stage 1: AI Vocal Isolation with Meta Demucs (htdemucs)
DEMUCS_MODEL = "htdemucs" #@param ["htdemucs", "htdemucs_ft"]
FORCE_REEXTRACT_VOCALS = False #@param {type:"boolean"}
PLAY_VOCAL_PREVIEW = True #@param {type:"boolean"}

if not STAGE1_AUDIO_WAV.is_file():
    raise FileNotFoundError(
        "No input audio found at STAGE1_AUDIO_WAV. Upload an audio/video file in Cell 1.2 first "
        "(or skip Stage 1 if you already have a \\k-timed .ass file)."
    )

DEMUCS_OUT_DIR = STAGE1_DIR / "demucs_out"
STAGE1_VOCALS_WAV = STAGE1_DIR / "vocals_44k.wav"
STAGE1_VOCALS_16K = STAGE1_DIR / "vocals_16k.wav"

if FORCE_REEXTRACT_VOCALS or not STAGE1_VOCALS_WAV.is_file():
    _stage1_dev = globals().get("STAGE1_DEVICE", "cuda" if HAS_GPU else "cpu")
    device_flag = "cuda" if _stage1_dev == "cuda" else "cpu"
    print(f"Running Meta Demucs ({DEMUCS_MODEL}) on {device_flag} to isolate vocals...")
    cmd = [
        sys.executable, "-m", "demucs",
        "--two-stems=vocals",
        "-n", DEMUCS_MODEL,
        "-d", device_flag,
        "-o", str(DEMUCS_OUT_DIR),
        str(STAGE1_AUDIO_WAV),
    ]
    subprocess.run(cmd, check=True)
    vocal_matches = sorted((DEMUCS_OUT_DIR / DEMUCS_MODEL).rglob("vocals.wav"))
    if not vocal_matches:
        raise FileNotFoundError(f"Demucs finished, but vocals.wav was not found under {DEMUCS_OUT_DIR}")
    shutil.copy2(vocal_matches[0], STAGE1_VOCALS_WAV)
    print(f"Vocal stem extracted: {STAGE1_VOCALS_WAV}")
else:
    print(f"Using cached vocal stem: {STAGE1_VOCALS_WAV}")

# Resample to 16kHz mono for forced alignment
subprocess.run(
    [
        "ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
        "-i", str(STAGE1_VOCALS_WAV), "-ac", "1", "-ar", "16000", "-c:a", "pcm_s16le", str(STAGE1_VOCALS_16K),
    ],
    check=True,
)
print(f"Alignment-ready 16kHz mono vocals: {STAGE1_VOCALS_16K}")

if PLAY_VOCAL_PREVIEW:
    try:
        from IPython.display import Audio, display  # type: ignore
        preview_wav = STAGE1_DIR / "vocals_preview_20s.wav"
        subprocess.run(
            [
                "ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
                "-i", str(STAGE1_VOCALS_WAV), "-t", "20", str(preview_wav),
            ],
            check=True,
        )
        print("Isolated Vocal Stem Preview (first 20s):")
        display(Audio(str(preview_wav)))
    except Exception as exc:
        print(f"Audio preview skipped: {exc}")

In [ ]:
#@title 1.4 Stage 1: Forced Alignment & Japanese Romaji Mora Syllabifier -> auto_timed_karaoke.ass
SPLIT_SOKUON_AS_MORA = True #@param {type:"boolean"}
SPLIT_LONG_VOWELS = True #@param {type:"boolean"}
MIN_SYLLABLE_CS = 3 #@param {type:"integer"}
ASS_PLAY_RES_X = 1920 #@param {type:"integer"}
ASS_PLAY_RES_Y = 1080 #@param {type:"integer"}
ASS_STYLE_NAME = "Romaji" #@param {type:"string"}
ASS_FONT_NAME = "Arial" #@param {type:"string"}
ASS_FONT_SIZE = 64 #@param {type:"integer"}
DOWNLOAD_STAGE1_ASS = False #@param {type:"boolean"}

# ==============================================================================
# 1. Accurate Japanese Romaji Mora Syllabifier
# ==============================================================================
VOWEL_CHARS = set("aeiouāīūēōâîûêô")
MACRON_CHARS = set("āīūēōâîûêô")
MACRON_TO_ASCII = {
    "ā": "aa", "ī": "ii", "ū": "uu", "ē": "ee", "ō": "ou",
    "â": "aa", "î": "ii", "û": "uu", "ê": "ee", "ô": "ou",
}
LONG_VOWEL_PAIRS = {
    ("a", "a"), ("i", "i"), ("u", "u"), ("e", "e"), ("o", "o"),
    ("o", "u"), ("e", "i"),
}
ONSETS_2 = {
    "sh", "ch", "ts", "dz", "dj", "th", "dh",
    "ky", "kw", "gy", "gw", "sy", "zy", "jy",
    "ty", "cy", "dy", "ny", "hy", "by", "py",
    "my", "ry", "fy", "vy", "wh",
}
ONSETS_1 = set("kgszjtdnhfbpmyrlwv")
GEMINATE_CONSONANTS = set("kstpgdbzjfhcr")


def _try_strict_romaji_morae(
    orig: str,
    split_sokuon_as_mora: bool = True,
    split_long_vowels: bool = True,
) -> list[tuple[str, float]] | None:
    """Strict Japanese romaji mora parser. Returns [(syllable, mora_weight)] or None if non-romaji."""
    low = orig.lower()
    n = len(low)
    i = 0
    tokens: list[tuple[str, float]] = []

    while i < n:
        ch = low[i]

        # 1. Special 3-char sokuon clusters: tch / cch / ssh / tts
        if i + 3 < n and low[i : i + 3] in ("tch", "cch", "ssh", "tts"):
            next_after = low[i + 3]
            if next_after in VOWEL_CHARS or next_after == "y":
                if split_sokuon_as_mora or not tokens:
                    tokens.append((orig[i : i + 1], 1.0))
                else:
                    prev_txt, prev_w = tokens[-1]
                    tokens[-1] = (prev_txt + orig[i : i + 1], prev_w + 1.0)
                i += 1
                continue

        # 2. General double-consonant sokuon (kk, tt, ss, pp, gg, dd, bb, zz, jj, ff, hh, cc)
        if (
            i + 1 < n
            and ch in GEMINATE_CONSONANTS
            and low[i + 1] == ch
            and (
                (i + 2 < n and low[i + 2] in VOWEL_CHARS)
                or (i + 3 < n and low[i + 1 : i + 3] in ONSETS_2 and low[i + 3] in VOWEL_CHARS)
            )
        ):
            if split_sokuon_as_mora or not tokens:
                tokens.append((orig[i : i + 1], 1.0))
            else:
                prev_txt, prev_w = tokens[-1]
                tokens[-1] = (prev_txt + orig[i : i + 1], prev_w + 1.0)
            i += 1
            continue

        # 3. 2-char onset + vowel (sha, chi, tsu, ryo, kya, nya, ...)
        if i + 2 < n and low[i : i + 2] in ONSETS_2 and low[i + 2] in VOWEL_CHARS:
            end_idx = i + 3
            v1 = low[i + 2]
            weight = 2.0 if v1 in MACRON_CHARS else 1.0
            if not split_long_vowels and end_idx < n and (v1, low[end_idx]) in LONG_VOWEL_PAIRS:
                end_idx += 1
                weight += 1.0
            tokens.append((orig[i:end_idx], weight))
            i = end_idx
            continue

        # 4. 1-char onset + vowel (ka, ki, ku, ..., wa, wo)
        if i + 1 < n and ch in ONSETS_1 and low[i + 1] in VOWEL_CHARS:
            end_idx = i + 2
            v1 = low[i + 1]
            weight = 2.0 if v1 in MACRON_CHARS else 1.0
            if not split_long_vowels and end_idx < n and (v1, low[end_idx]) in LONG_VOWEL_PAIRS:
                end_idx += 1
                weight += 1.0
            tokens.append((orig[i:end_idx], weight))
            i = end_idx
            continue

        # 5. Standalone vowel (a, i, u, e, o, ā, ī, ū, ē, ō)
        if ch in VOWEL_CHARS:
            end_idx = i + 1
            weight = 2.0 if ch in MACRON_CHARS else 1.0
            if not split_long_vowels and end_idx < n and (ch, low[end_idx]) in LONG_VOWEL_PAIRS:
                end_idx += 1
                weight += 1.0
            tokens.append((orig[i:end_idx], weight))
            i = end_idx
            continue

        # 6. Moraic 'n' (before consonant, apostrophe, or end of word)
        if ch == "n":
            if i + 1 < n and low[i + 1] in ("'", "\u2019"):
                tokens.append((orig[i : i + 2], 1.0))
                i += 2
                continue
            if i + 1 == n or (low[i + 1] not in VOWEL_CHARS and low[i + 1] != "y"):
                tokens.append((orig[i : i + 1], 1.0))
                i += 1
                continue

        # 7. Moraic 'm' before b / p / m (traditional Hepburn: sempai, shimbun)
        if ch == "m" and i + 1 < n and (low[i + 1] in ("b", "p", "m") or low[i + 1] in ("'", "\u2019")):
            if low[i + 1] in ("'", "\u2019"):
                tokens.append((orig[i : i + 2], 1.0))
                i += 2
            else:
                tokens.append((orig[i : i + 1], 1.0))
                i += 1
            continue

        # Unrecognized consonant cluster or coda -> not pure Japanese romaji
        return None

    return tokens


def _syllabify_english_fallback(core: str) -> list[tuple[str, float]]:
    """Vowel-nucleus syllable splitter for English words embedded in anime lyrics."""
    groups = list(re.finditer(r"[aeiouyAEIOUYāīūēōĀĪŪĒŌ]+", core))
    if len(groups) <= 1:
        return [(core, 1.0)]

    # Drop silent trailing single 'e' when determining syllable count (e.g. 'time', 'shine', 'life')
    low = core.lower()
    if (
        len(groups) > 1
        and groups[-1].end() == len(core)
        and low.endswith("e")
        and not low.endswith(("ee", "ie", "ye", "oe", "ue", "le"))
        and len(groups[-1].group(0)) == 1
    ):
        groups = groups[:-1]

    if len(groups) <= 1:
        return [(core, 1.0)]

    cuts = [0]
    for idx in range(len(groups) - 1):
        v_end = groups[idx].end()
        next_v_start = groups[idx + 1].start()
        inter = next_v_start - v_end
        if inter <= 1:
            cuts.append(v_end)
        else:
            cuts.append(v_end + inter // 2)
    cuts.append(len(core))

    out = []
    for a, b in zip(cuts[:-1], cuts[1:]):
        seg = core[a:b]
        if seg:
            out.append((seg, 1.0))
    return out or [(core, 1.0)]


def syllabify_romaji_word_with_weights(
    word: str,
    split_sokuon_as_mora: bool = True,
    split_long_vowels: bool = True,
) -> list[tuple[str, float]]:
    """
    Split a single word (preserving case & punctuation) into [(syllable, mora_weight)].
    Invariant: ''.join(s for s, _ in result) == word.
    """
    if not word:
        return []

    m = re.match(r"^([^A-Za-z0-9āīūēōâîûêôĀĪŪĒŌÂÎÛÊÔ]*)(.*?)([^A-Za-z0-9āīūēōâîûêôĀĪŪĒŌÂÎÛÊÔ]*)$", word)
    if not m or not m.group(2):
        return [(word, 1.0)]

    lead_p, core, trail_p = m.group(1), m.group(2), m.group(3)

    # Handle internal hyphens by splitting segments while preserving exact text
    if "-" in core:
        pieces = re.split(r"(-)", core)
        combined: list[tuple[str, float]] = []
        idx = 0
        while idx < len(pieces):
            seg = pieces[idx]
            if idx + 1 < len(pieces) and pieces[idx + 1] == "-":
                seg = seg + "-"
                idx += 2
            else:
                idx += 1
            if not seg:
                continue
            seg_base = seg[:-1] if seg.endswith("-") else seg
            sub = (
                _try_strict_romaji_morae(seg_base, split_sokuon_as_mora, split_long_vowels)
                if seg_base
                else None
            )
            if sub is None:
                sub = _syllabify_english_fallback(seg_base) if seg_base else [(seg, 1.0)]
            if seg.endswith("-") and sub:
                sub[-1] = (sub[-1][0] + "-", sub[-1][1])
            combined.extend(sub)
        parsed = combined
    else:
        parsed = _try_strict_romaji_morae(core, split_sokuon_as_mora, split_long_vowels)
        if parsed is None:
            parsed = _syllabify_english_fallback(core)

    if lead_p:
        parsed[0] = (lead_p + parsed[0][0], parsed[0][1])
    if trail_p:
        parsed[-1] = (parsed[-1][0] + trail_p, parsed[-1][1])
    return parsed


def syllabify_romaji_word(
    word: str,
    split_sokuon_as_mora: bool = True,
    split_long_vowels: bool = True,
) -> list[str]:
    return [
        syl
        for syl, _ in syllabify_romaji_word_with_weights(
            word,
            split_sokuon_as_mora=split_sokuon_as_mora,
            split_long_vowels=split_long_vowels,
        )
    ]


def distribute_duration_cs(total_cs: int, weights: list[float], min_cs: int = 1) -> list[int]:
    """Distribute integer centiseconds across weights with exact sum conservation."""
    n = len(weights)
    if n == 0:
        return []
    total_cs = max(n, int(total_cs))
    eff_min = min(max(1, int(min_cs)), total_cs // n)
    alloc = [eff_min] * n
    rem = total_cs - eff_min * n
    if rem <= 0:
        return alloc

    safe_w = [max(0.05, float(w)) for w in weights]
    w_sum = sum(safe_w)
    raw = [rem * (w / w_sum) for w in safe_w]
    floors = [int(math.floor(x)) for x in raw]
    for i, f in enumerate(floors):
        alloc[i] += f
    leftover = rem - sum(floors)
    order = sorted(range(n), key=lambda i: (raw[i] - floors[i], safe_w[i]), reverse=True)
    for idx in order[:leftover]:
        alloc[idx] += 1
    return alloc


# ==============================================================================
# 2. Word-Level Forced Alignment (torchaudio MMS_FA / WhisperX / Energy Fallback)
# ==============================================================================
def normalize_word_for_mms(word: str) -> str:
    s = word.lower()
    for k, v in MACRON_TO_ASCII.items():
        s = s.replace(k, v)
    return re.sub(r"[^a-z']", "", s)


def align_words_energy_fallback(
    waveform_1d,
    sample_rate: int,
    words: list[str],
    line_start_ms: int,
    line_end_ms: int,
) -> list[tuple[int, int]]:
    """Proportional syllable-count fallback when vocal slice is silent or too short for FA."""
    total_ms = max(len(words) * 20, line_end_ms - line_start_ms)
    syl_counts = [max(1, len(syllabify_romaji_word(w, SPLIT_SOKUON_AS_MORA, SPLIT_LONG_VOWELS))) for w in words]

    # Trim leading/trailing silence via RMS energy if detectable
    active_start = line_start_ms
    active_end = line_end_ms
    if waveform_1d is not None and len(waveform_1d) > 320:
        import numpy as np  # type: ignore
        arr = waveform_1d if isinstance(waveform_1d, np.ndarray) else waveform_1d.cpu().numpy()
        hop = max(1, int(sample_rate * 0.02))
        energies = [float(np.sqrt(np.mean(arr[i : i + hop] ** 2))) for i in range(0, len(arr), hop)]
        peak = max(energies) if energies else 0.0
        if peak > 1e-4:
            thresh = peak * 0.08
            above = [idx for idx, e in enumerate(energies) if e >= thresh]
            if above:
                s_off = max(0, int(above[0] * 20 - 40))
                e_off = min(total_ms, int((above[-1] + 1) * 20 + 60))
                if e_off - s_off >= len(words) * 60:
                    active_start = line_start_ms + s_off
                    active_end = line_start_ms + e_off

    active_dur = max(len(words) * 20, active_end - active_start)
    w_sum = float(sum(syl_counts))
    spans = []
    cur = float(active_start)
    for idx, cnt in enumerate(syl_counts):
        nxt = float(active_end) if idx == len(syl_counts) - 1 else cur + active_dur * (cnt / w_sum)
        spans.append((int(round(cur)), int(round(nxt))))
        cur = nxt
    return spans


def align_lines_with_mms_fa(lines: list[dict], vocals_16k: Path) -> list[list[tuple[int, int]]]:
    import torch  # type: ignore
    import torchaudio  # type: ignore

    bundle = torchaudio.pipelines.MMS_FA
    model = bundle.get_model().to(STAGE1_DEVICE)
    model.eval()
    tokenizer = bundle.get_tokenizer()
    aligner = bundle.get_aligner()

    waveform, sr = torchaudio.load(str(vocals_16k))
    if sr != bundle.sample_rate:
        waveform = torchaudio.functional.resample(waveform, sr, bundle.sample_rate)
        sr = bundle.sample_rate
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    all_spans = []
    for ln in lines:
        words = ln["text"].split()
        if not words:
            all_spans.append([])
            continue
        s_samp = max(0, int(round(ln["start_ms"] * sr / 1000.0)))
        e_samp = min(waveform.shape[1], int(round(ln["end_ms"] * sr / 1000.0)))
        seg = waveform[:, s_samp:e_samp]

        norm_tokens = [normalize_word_for_mms(w) for w in words]
        valid_pairs = [(idx, tok) for idx, tok in enumerate(norm_tokens) if tok]

        if seg.shape[1] < int(sr * 0.25) or not valid_pairs:
            all_spans.append(align_words_energy_fallback(seg.squeeze(0), sr, words, ln["start_ms"], ln["end_ms"]))
            continue

        try:
            with torch.inference_mode():
                emission, _ = model(seg.to(STAGE1_DEVICE))
                tokens = tokenizer([tok for _, tok in valid_pairs])
                token_spans = aligner(emission[0], tokens)

            ratio = seg.shape[1] / max(1, emission.shape[1]) / float(sr) * 1000.0
            spans: list[tuple[int, int] | None] = [None] * len(words)
            for (orig_idx, _), word_tok_spans in zip(valid_pairs, token_spans):
                if word_tok_spans:
                    ws = ln["start_ms"] + int(round(word_tok_spans[0].start * ratio))
                    we = ln["start_ms"] + int(round(word_tok_spans[-1].end * ratio))
                    spans[orig_idx] = (ws, max(ws + 30, we))

            # Fill any punctuation-only tokens that had empty normalized strings
            for idx in range(len(words)):
                if spans[idx] is None:
                    prev_e = spans[idx - 1][1] if idx > 0 and spans[idx - 1] else ln["start_ms"]
                    spans[idx] = (prev_e, prev_e + 40)

            all_spans.append([(int(a), int(b)) for a, b in spans])  # type: ignore
        except Exception as exc:
            print(f"MMS_FA fallback on line '{ln['text'][:32]}...': {exc}")
            all_spans.append(align_words_energy_fallback(seg.squeeze(0), sr, words, ln["start_ms"], ln["end_ms"]))

    return all_spans


def align_lines_with_whisperx(lines: list[dict], vocals_16k: Path) -> list[list[tuple[int, int]]]:
    import soundfile as sf  # type: ignore
    import whisperx  # type: ignore

    audio = whisperx.load_audio(str(vocals_16k))
    sr = 16000
    align_model, metadata = whisperx.load_align_model(language_code="ja", device=STAGE1_DEVICE)

    all_spans = []
    for ln in lines:
        words = ln["text"].split()
        if not words:
            all_spans.append([])
            continue
        s_samp = max(0, int(round(ln["start_ms"] * sr / 1000.0)))
        e_samp = min(len(audio), int(round(ln["end_ms"] * sr / 1000.0)))
        seg = audio[s_samp:e_samp]
        if len(seg) < int(sr * 0.25):
            all_spans.append(align_words_energy_fallback(seg, sr, words, ln["start_ms"], ln["end_ms"]))
            continue
        try:
            transcript = [{"text": ln["text"], "start": 0.0, "end": len(seg) / float(sr)}]
            aligned = whisperx.align(transcript, align_model, metadata, seg, STAGE1_DEVICE, return_char_alignments=False)
            w_items = [w for w in aligned.get("word_segments", []) if "start" in w and "end" in w]
            if len(w_items) == len(words):
                all_spans.append([
                    (
                        ln["start_ms"] + int(round(w["start"] * 1000.0)),
                        ln["start_ms"] + int(round(w["end"] * 1000.0)),
                    )
                    for w in w_items
                ])
            else:
                all_spans.append(align_words_energy_fallback(seg, sr, words, ln["start_ms"], ln["end_ms"]))
        except Exception as exc:
            print(f"WhisperX fallback on line '{ln['text'][:32]}...': {exc}")
            all_spans.append(align_words_energy_fallback(seg, sr, words, ln["start_ms"], ln["end_ms"]))

    return all_spans


# ==============================================================================
# 3. Build Clean {\kXX} Karaoke Line & Export auto_timed_karaoke.ass
# ==============================================================================
def build_k_tagged_line(
    words: list[str],
    word_spans_ms: list[tuple[int, int]],
    line_start_ms: int,
    line_end_ms: int,
    split_sokuon_as_mora: bool = True,
    split_long_vowels: bool = True,
    min_syl_cs: int = 3,
) -> str:
    if not words:
        return ""

    line_cs = max(1, int(round((line_end_ms - line_start_ms) / 10.0)))
    word_syllables = [
        syllabify_romaji_word_with_weights(w, split_sokuon_as_mora, split_long_vowels)
        for w in words
    ]
    min_req_after = []
    running = 0
    for syls in reversed(word_syllables):
        min_req_after.append(running)
        running += max(1, len(syls))
    min_req_after.reverse()

    if line_cs < running:
        # Line is extremely short: distribute uniformly across all syllables
        all_items = []
        for w_idx, syls in enumerate(word_syllables):
            for s_idx, (s_txt, w) in enumerate(syls):
                suffix = " " if (s_idx == len(syls) - 1 and w_idx < len(word_syllables) - 1) else ""
                all_items.append((s_txt + suffix, w))
        durs = distribute_duration_cs(line_cs, [w for _, w in all_items], min_cs=1)
        return "".join(f"{{\\k{d}}}{txt}" for (txt, _), d in zip(all_items, durs))

    cursor_cs = 0
    parts: list[str] = []

    for idx, syls in enumerate(word_syllables):
        w_start_ms, w_end_ms = word_spans_ms[idx]
        ideal_start_cs = max(cursor_cs, int(round((w_start_ms - line_start_ms) / 10.0)))
        ideal_end_cs = max(ideal_start_cs + len(syls), int(round((w_end_ms - line_start_ms) / 10.0)))

        max_allowed_end = line_cs - min_req_after[idx]
        actual_end_cs = min(ideal_end_cs, max_allowed_end)
        actual_start_cs = min(ideal_start_cs, actual_end_cs - len(syls))
        actual_start_cs = max(cursor_cs, actual_start_cs)

        gap_cs = actual_start_cs - cursor_cs
        if idx == 0:
            if gap_cs > 0:
                parts.append(f"{{\\k{gap_cs}}}")
        else:
            if gap_cs > 0:
                parts.append(f"{{\\k{gap_cs}}} ")
            else:
                parts[-1] = parts[-1] + " "

        word_dur_cs = max(len(syls), actual_end_cs - actual_start_cs)
        syl_durs = distribute_duration_cs(word_dur_cs, [w for _, w in syls], min_cs=min_syl_cs)
        for (s_txt, _), d_cs in zip(syls, syl_durs):
            parts.append(f"{{\\k{d_cs}}}{s_txt}")

        cursor_cs = actual_start_cs + sum(syl_durs)

    tail_cs = line_cs - cursor_cs
    if tail_cs > 0:
        parts.append(f"{{\\k{tail_cs}}}")

    return "".join(parts)


def build_stage1_ass_document(
    lines: list[dict],
    all_word_spans: list[list[tuple[int, int]]],
    play_res_x: int = 1920,
    play_res_y: int = 1080,
    style_name: str = "Romaji",
    font_name: str = "Arial",
    font_size: int = 64,
    split_sokuon_as_mora: bool = True,
    split_long_vowels: bool = True,
    min_syl_cs: int = 3,
) -> str:
    header = [
        "[Script Info]",
        "Title: Auto-Timed Anime Romaji Karaoke",
        "ScriptType: v4.00+",
        "WrapStyle: 2",
        "ScaledBorderAndShadow: yes",
        f"PlayResX: {play_res_x}",
        f"PlayResY: {play_res_y}",
        "",
        "[V4+ Styles]",
        "Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, "
        "Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, "
        "Shadow, Alignment, MarginL, MarginR, MarginV, Encoding",
        f"Style: {style_name},{font_name},{font_size},&H0000F5FF,&H00FFFFFF,&H00120A1E,&H80000000,"
        "-1,0,0,0,100,100,0,0,1,3,1.5,2,40,40,48,1",
        "",
        "[Events]",
        "Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text",
    ]
    events = []
    for ln, spans in zip(lines, all_word_spans):
        words = ln["text"].split()
        k_text = build_k_tagged_line(
            words,
            spans,
            ln["start_ms"],
            ln["end_ms"],
            split_sokuon_as_mora=split_sokuon_as_mora,
            split_long_vowels=split_long_vowels,
            min_syl_cs=min_syl_cs,
        )
        st_name = ln.get("style") or style_name
        events.append(
            f"Dialogue: 0,{format_ms_clock(ln['start_ms'])},{format_ms_clock(ln['end_ms'])},"
            f"{st_name},,0,0,0,karaoke,{k_text}"
        )
    return "\n".join(header + events) + "\n"


if not globals().get("STAGE1_LINES"):
    raise ValueError("STAGE1_LINES is empty. Run Cell 1.2 first.")

STAGE1_DEVICE = globals().get("STAGE1_DEVICE", "cuda" if HAS_GPU else "cpu")
ALIGNMENT_ENGINE = globals().get("ALIGNMENT_ENGINE", "torchaudio_mms_fa")
_vocals_16k = globals().get("STAGE1_VOCALS_16K")

if _vocals_16k is not None and Path(_vocals_16k).is_file():
    if ALIGNMENT_ENGINE == "whisperx":
        print("Aligning romaji words with WhisperX...")
        STAGE1_WORD_SPANS = align_lines_with_whisperx(STAGE1_LINES, Path(_vocals_16k))
    else:
        print("Aligning romaji words with torchaudio MMS_FA...")
        STAGE1_WORD_SPANS = align_lines_with_mms_fa(STAGE1_LINES, Path(_vocals_16k))
else:
    print("Note: No vocal audio found — generating timing preview with uniform syllable pacing.")
    STAGE1_WORD_SPANS = [
        align_words_energy_fallback(None, 16000, ln["text"].split(), ln["start_ms"], ln["end_ms"])
        for ln in STAGE1_LINES
    ]

STAGE1_ASS_TEXT = build_stage1_ass_document(
    STAGE1_LINES,
    STAGE1_WORD_SPANS,
    play_res_x=ASS_PLAY_RES_X,
    play_res_y=ASS_PLAY_RES_Y,
    style_name=ASS_STYLE_NAME,
    font_name=ASS_FONT_NAME,
    font_size=ASS_FONT_SIZE,
    split_sokuon_as_mora=SPLIT_SOKUON_AS_MORA,
    split_long_vowels=SPLIT_LONG_VOWELS,
    min_syl_cs=MIN_SYLLABLE_CS,
)

AUTO_TIMED_ASS_PATH = WORK_DIR / "auto_timed_karaoke.ass"
AUTO_TIMED_ASS_PATH.write_text(STAGE1_ASS_TEXT, encoding="utf-8")
Path("auto_timed_karaoke.ass").write_text(STAGE1_ASS_TEXT, encoding="utf-8")

print(f"\nSaved Stage 1 output: {AUTO_TIMED_ASS_PATH}")
print("\nGenerated [Events] preview:")
for line in STAGE1_ASS_TEXT.splitlines():
    if line.startswith("Dialogue:"):
        print(" ", line)

if DOWNLOAD_STAGE1_ASS:
    download_file_compat(AUTO_TIMED_ASS_PATH)

---
# Stage 2 (Core Required Feature): Prompt-Driven AI KFX Animation Engine

Welcome to **Stage 2**! You can run this stage directly after Stage 1 **or** start right here by uploading an existing `\k` / `\kf` / `\ko`-timed `.ass` file (or using the built-in instant demo).

### What Stage 2 Does
1. **Loads your timed `.ass` + `.ttf`/`.otf` font** and renders an **Interactive Line Inspector Table** (`Line #1`, `Line #2`, ...) so you can see exact line numbers, timestamps, and style names.
2. **Measures pixel-accurate typography** using **`OS/2`-calibrated Pillow / FreeType** metrics so multi-syllable words never drift or overlap.
3. **Supports two flexible KFX Modes (`ipywidgets` UI + JSON config):**
   - **Mode A — Shared KFX Animation + Line-Range Colors:** Define one **Master KFX Prompt** (or preset) compiled once for consistent physics & timing across the whole song, plus lightweight **Color Palette Rules** (`+ Add Color Range`) specifying `c`, `c`, `c`, and `\p1` particle colors per line range or ASS style.
   - **Mode B — Unique KFX Animation + Colors per Line Range:** Give different line ranges (e.g., `1-3` Verse, `4-6` Chorus) or ASS styles completely distinct animations, glows, particles, and colors.
4. **Compiles all lines chronologically into a single `final_kfx_karaoke.ass`**, renders a per-line visual preview via `ffmpeg` + `libass`, and triggers download.

In [ ]:
#@title 2.1 Stage 2 Inputs: Load Timed .ass, Font (.ttf / .otf) & Line Inspector Table
ASS_SOURCE = "Use Built-in Anime Romaji Demo .ass (Instant Test)" #@param ["Upload existing \\k/\\kf-timed .ass file", "Use Existing / Mounted Drive .ass Path", "Use auto_timed_karaoke.ass from Stage 1", "Use Built-in Anime Romaji Demo .ass (Instant Test)"]
ASS_FILE_PATH = "" #@param {type:"string"}
FONT_SOURCE = "Auto System Font / Fallback" #@param ["Auto System Font / Fallback", "Upload .ttf/.otf Font File", "Use Existing / Mounted Drive Font Path"]
FONT_FILE_PATH = "" #@param {type:"string"}
TARGET_STYLE_FILTER = "" #@param {type:"string"}

DEMO_KFX_ASS = """[Script Info]
Title: Aegisub AI KFX Studio Multi-Section Demo
ScriptType: v4.00+
WrapStyle: 2
ScaledBorderAndShadow: yes
PlayResX: 1920
PlayResY: 1080

[V4+ Styles]
Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding
Style: Romaji_Verse,Arial,66,&H00FFE699,&H00FFFFFF,&H00281810,&H80000000,-1,0,0,0,100,100,1,0,1,3,1.5,2,48,48,54,1
Style: Romaji_Chorus,Arial,70,&H0000D7FF,&H00FFFFFF,&H00120A2A,&H80000000,-1,0,0,0,100,100,1,0,1,3.5,1.5,2,48,48,54,1

[Events]
Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text
Dialogue: 0,0:00:00.50,0:00:04.70,Romaji_Verse,,0,0,0,karaoke,{\\k20}Hi{\\k18}ka{\\k22}ri {\\k20}wo {\\k25}sa{\\k22}ga{\\k26}shi{\\k24}te {\\k28}yo{\\k22}zo{\\k24}ra {\\k18}wo {\\k30}ka{\\k22}ke{\\k24}te {\\k35}yu{\\k40}ku
Dialogue: 0,0:00:04.90,0:00:09.10,Romaji_Verse,,0,0,0,karaoke,{\\k22}Shi{\\k20}zu{\\k24}ka {\\k20}na {\\k24}mi{\\k22}na{\\k24}mo {\\k18}ni {\\k26}yu{\\k24}re{\\k24}ru {\\k24}tsu{\\k24}ki{\\k22}ka{\\k44}ge
Dialogue: 0,0:00:09.30,0:00:13.50,Romaji_Verse,,0,0,0,karaoke,{\\k24}Ko{\\k18}n'{\\k24}ya {\\k28}ho{\\k22}shi {\\k18}no {\\k24}shi{\\k22}ta {\\k20}de {\\k28}chi{\\k26}ka{\\k24}u {\\k22}yo {\\k52}Star{\\k68}light
Dialogue: 0,0:00:13.70,0:00:17.90,Romaji_Chorus,,0,0,0,karaoke,{\\k22}Zut{\\k24}to {\\k22}ka{\\k20}wa{\\k24}ra{\\k20}na{\\k22}i {\\k26}o{\\k22}mo{\\k22}i {\\k18}wo {\\k24}da{\\k22}ki{\\k24}shi{\\k24}me{\\k44}te
Dialogue: 0,0:00:18.10,0:00:22.30,Romaji_Chorus,,0,0,0,karaoke,{\\k24}Mo{\\k22}e{\\k24}a{\\k22}ga{\\k24}ru {\\k26}ki{\\k24}se{\\k24}ki {\\k20}ga {\\k24}mi{\\k24}ra{\\k24}i {\\k22}wo {\\k36}hi{\\k38}ra{\\k42}ku
Dialogue: 0,0:00:22.50,0:00:26.70,Romaji_Chorus,,0,0,0,karaoke,{\\k22}A{\\k20}ri{\\k24}ga{\\k24}to{\\k24}u {\\k26}i{\\k24}tsu{\\k24}mo {\\k26}so{\\k22}ba {\\k18}ni {\\k22}i{\\k20}te {\\k26}ku{\\k24}re{\\k64}te
"""

STAGE2_DIR = WORK_DIR / "stage2"
STAGE2_DIR.mkdir(parents=True, exist_ok=True)


def extract_ttf_family_name(font_path: Path) -> str | None:
    try:
        from fontTools.ttLib import TTFont  # type: ignore
        tt = TTFont(str(font_path), fontNumber=0)
        name_table = tt["name"]
        for name_id in (16, 1):
            for rec in name_table.names:
                if rec.nameID == name_id and rec.platformID in (3, 1, 0):
                    val = rec.toUnicode().strip()
                    if val:
                        return val
    except Exception:
        pass
    try:
        from PIL import ImageFont  # type: ignore
        fam, _ = ImageFont.truetype(str(font_path), 32).getname()
        return fam
    except Exception:
        return None


def inspect_ass_dialogue_lines(ass_text: str, style_filter: str = "") -> tuple[list[dict], list[str]]:
    """Extract 1-based numbered Dialogue lines and unique styles for the Line Inspector Table."""
    in_events = False
    fields = ["Layer", "Start", "End", "Style", "Name", "MarginL", "MarginR", "MarginV", "Effect", "Text"]
    filt = (style_filter or "").strip().lower()
    k_tag_re = re.compile(r"\{[^}]*\\(?:kf|ko|k|K)-?\d+[^}]*\}")
    rows: list[dict] = []
    styles_seen: list[str] = []

    for raw_line in ass_text.splitlines():
        line = raw_line.strip()
        if not line or line.startswith(";"):
            continue
        if line.startswith("[") and line.endswith("]"):
            in_events = line.lower() == "[events]"
            continue
        if not in_events:
            continue
        if line.lower().startswith("format:"):
            fields = [f.strip() for f in line.split(":", 1)[1].split(",")]
            continue
        if line.startswith("Dialogue:"):
            payload = line.split(":", 1)[1].lstrip()
            cols = payload.split(",", len(fields) - 1)
            if len(cols) != len(fields):
                continue
            row = dict(zip(fields, cols))
            st_name = row.get("Style", "Default").strip()
            if filt and st_name.lower() != filt:
                continue
            raw_txt = row.get("Text", "")
            clean_txt = re.sub(r"\{[^}]*\}", "", raw_txt)
            clean_txt = clean_txt.replace("\\N", " ").replace("\\n", " ").replace("\\h", " ")
            clean_txt = re.sub(r"\s+", " ", clean_txt).strip()
            syl_count = len(k_tag_re.findall(raw_txt))
            if st_name not in styles_seen:
                styles_seen.append(st_name)
            rows.append(
                {
                    "line_num": len(rows) + 1,
                    "start": row.get("Start", "").strip(),
                    "end": row.get("End", "").strip(),
                    "style": st_name,
                    "syl_count": syl_count,
                    "clean_text": clean_txt,
                }
            )
    return rows, styles_seen


def display_line_inspector_table(inspected_lines: list[dict], detected_styles: list[str]) -> None:
    import html as _html

    print(f"\n=== Line Inspector Table ({len(inspected_lines)} karaoke lines | Styles: {detected_styles}) ===")
    print(f"{'Line':>6}  {'Start':>10}  {'End':>10}  {'Style':<16}  {'Syls':>4}  Clean Lyrics Text")
    print("-" * 92)
    for item in inspected_lines:
        print(
            f"#{item['line_num']:<5}  {item['start']:>10}  {item['end']:>10}  "
            f"{item['style'][:16]:<16}  {item['syl_count']:>4}  {item['clean_text']}"
        )

    try:
        from IPython import get_ipython  # type: ignore
        if get_ipython() is not None:
            from IPython.display import HTML, display  # type: ignore
            trs = []
            for item in inspected_lines:
                trs.append(
                    "<tr>"
                    f"<td style='padding:6px 10px;font-weight:700;color:#7dd3fc;'>#{item['line_num']}</td>"
                    f"<td style='padding:6px 10px;font-family:monospace;color:#cbd5e1;'>{_html.escape(item['start'])}</td>"
                    f"<td style='padding:6px 10px;font-family:monospace;color:#cbd5e1;'>{_html.escape(item['end'])}</td>"
                    f"<td style='padding:6px 10px;'><span style='background:#334155;color:#f8fafc;padding:2px 8px;border-radius:10px;font-size:12px;'>{_html.escape(item['style'])}</span></td>"
                    f"<td style='padding:6px 10px;text-align:center;color:#94a3b8;'>{item['syl_count']}</td>"
                    f"<td style='padding:6px 10px;color:#f1f5f9;'>{_html.escape(item['clean_text'])}</td>"
                    "</tr>"
                )
            html_doc = (
                "<div style='max-height:320px;overflow-y:auto;border:1px solid #334155;border-radius:8px;background:#0f172a;margin:8px 0;'>"
                "<table style='width:100%;border-collapse:collapse;font-size:13px;text-align:left;'>"
                "<thead style='position:sticky;top:0;background:#1e293b;color:#e2e8f0;border-bottom:1px solid #475569;'>"
                "<tr>"
                "<th style='padding:8px 10px;'>Line #</th>"
                "<th style='padding:8px 10px;'>Start Time</th>"
                "<th style='padding:8px 10px;'>End Time</th>"
                "<th style='padding:8px 10px;'>Style Name</th>"
                "<th style='padding:8px 10px;text-align:center;'>Syls</th>"
                "<th style='padding:8px 10px;'>Clean Lyrics Text</th>"
                "</tr></thead>"
                f"<tbody>{''.join(trs)}</tbody></table></div>"
            )
            display(HTML(html_doc))
    except Exception:
        pass


# --- 1. Load Timed ASS ---
if "Upload existing" in ASS_SOURCE:
    uploaded_ass = upload_to_dir(STAGE2_DIR, label="\\k/\\kf-timed .ass file")
    STAGE2_ASS_PATH = uploaded_ass[0]
    STAGE2_RAW_ASS = STAGE2_ASS_PATH.read_text(encoding="utf-8-sig", errors="replace")
elif "Mounted Drive .ass Path" in ASS_SOURCE:
    resolved_ass = resolve_input_path(
        ASS_FILE_PATH,
        {".ass"},
        label=".ass subtitle",
        target_field="ASS_FILE_PATH",
    )
    if resolved_ass is None:
        raise FileNotFoundError(
            "Select one of the listed .ass files and paste its full path into ASS_FILE_PATH."
        )
    STAGE2_ASS_PATH = resolved_ass
    STAGE2_RAW_ASS = STAGE2_ASS_PATH.read_text(encoding="utf-8-sig", errors="replace")
elif "auto_timed_karaoke.ass" in ASS_SOURCE:
    candidate = WORK_DIR / "auto_timed_karaoke.ass"
    if not candidate.is_file():
        candidate = Path("auto_timed_karaoke.ass")
    if not candidate.is_file():
        raise FileNotFoundError(
            "auto_timed_karaoke.ass not found. Run Stage 1 first or choose Upload / Mounted Drive / Demo in ASS_SOURCE."
        )
    STAGE2_ASS_PATH = candidate
    STAGE2_RAW_ASS = STAGE2_ASS_PATH.read_text(encoding="utf-8-sig", errors="replace")
else:
    STAGE2_ASS_PATH = STAGE2_DIR / "demo_karaoke.ass"
    STAGE2_ASS_PATH.write_text(DEMO_KFX_ASS, encoding="utf-8")
    STAGE2_RAW_ASS = DEMO_KFX_ASS

# --- 2. Load Font (.ttf / .otf) ---
STAGE2_FONT_PATH = None
STAGE2_UPLOADED_FONT_FAMILY = None
STAGE2_FONT_IS_CUSTOM = False

if FONT_SOURCE == "Upload .ttf/.otf Font File" and IN_COLAB:
    uploaded_fonts = upload_to_dir(STAGE2_DIR, label=".ttf / .otf font file")
    if uploaded_fonts:
        STAGE2_FONT_PATH = str(uploaded_fonts[0])
        STAGE2_UPLOADED_FONT_FAMILY = extract_ttf_family_name(uploaded_fonts[0])
        STAGE2_FONT_IS_CUSTOM = True
elif FONT_SOURCE == "Use Existing / Mounted Drive Font Path":
    resolved_font = resolve_input_path(
        FONT_FILE_PATH,
        {".ttf", ".otf", ".ttc"},
        label="font (.ttf/.otf)",
        target_field="FONT_FILE_PATH",
    )
    if resolved_font is not None:
        STAGE2_FONT_PATH = str(resolved_font)
        STAGE2_UPLOADED_FONT_FAMILY = extract_ttf_family_name(resolved_font)
        STAGE2_FONT_IS_CUSTOM = True
    else:
        print("No custom font file resolved yet; using system font fallback.")
        STAGE2_FONT_PATH = find_system_font()
else:
    STAGE2_FONT_PATH = find_system_font()

print(f"Loaded .ass file : {STAGE2_ASS_PATH} ({len(STAGE2_RAW_ASS.splitlines())} lines)")
if STAGE2_FONT_PATH:
    fam_info = f" (family: {STAGE2_UPLOADED_FONT_FAMILY})" if STAGE2_UPLOADED_FONT_FAMILY else ""
    print(f"Font metrics     : {STAGE2_FONT_PATH}{fam_info}")
else:
    print("Font metrics     : No .ttf/.otf found; using calibrated proportional fallback metrics.")

STAGE2_INSPECTED_LINES, STAGE2_DETECTED_STYLES = inspect_ass_dialogue_lines(
    STAGE2_RAW_ASS, style_filter=TARGET_STYLE_FILTER
)
display_line_inspector_table(STAGE2_INSPECTED_LINES, STAGE2_DETECTED_STYLES)

In [ ]:
#@title 2.2 Exact Syllable Layout, FreeType Font Metrics & ASS Vector Math Engine
WIDTH_CALIBRATION_SCALE = 1.0 #@param {type:"number"}
EXTRA_SYLLABLE_SPACING_PX = 0.0 #@param {type:"number"}

from dataclasses import dataclass, field
from PIL import ImageFont  # type: ignore


@dataclass
class ScriptMeta:
    title: str = "Aegisub AI KFX Studio"
    play_res_x: int = 1920
    play_res_y: int = 1080
    wrap_style: int = 2
    scaled_border_and_shadow: str = "yes"
    raw_info_lines: list[str] = field(default_factory=list)


@dataclass
class StyleInfo:
    name: str = "Default"
    fontname: str = "Arial"
    fontsize: float = 64.0
    primary_colour: str = "&H00FFFFFF"
    secondary_colour: str = "&H0000FFFF"
    outline_colour: str = "&H00000000"
    back_colour: str = "&H80000000"
    bold: int = 0
    italic: int = 0
    underline: int = 0
    strikeout: int = 0
    scale_x: float = 100.0
    scale_y: float = 100.0
    spacing: float = 0.0
    angle: float = 0.0
    border_style: int = 1
    outline: float = 2.0
    shadow: float = 1.0
    alignment: int = 2
    margin_l: int = 40
    margin_r: int = 40
    margin_v: int = 40
    encoding: int = 1
    raw_line: str = ""


@dataclass
class Syllable:
    index: int
    word_index: int
    text: str
    raw_text: str
    k_type: str
    start_ms: int
    end_ms: int
    duration_ms: int
    rel_start_ms: int
    rel_end_ms: int
    left: float
    center: float
    right: float
    top: float
    middle: float
    bottom: float
    width: float
    height: float
    x: float
    y: float
    is_space: bool = False


@dataclass
class KaraokeLine:
    index: int
    layer: int
    start_ms: int
    end_ms: int
    duration_ms: int
    style: str
    actor: str
    margin_l: int
    margin_r: int
    margin_v: int
    effect: str
    raw_text: str
    plain_text: str
    alignment: int
    fontsize: float
    scale_x: float
    scale_y: float
    spacing: float
    left: float
    center: float
    right: float
    top: float
    middle: float
    bottom: float
    width: float
    height: float
    x: float
    y: float
    syllables: list[Syllable] = field(default_factory=list)


class FontMeasurer:
    """Measures exact prefix advances and line heights via Pillow / FreeType with OS/2 GDI calibration."""

    def __init__(self, font_path: str | None = None):
        self.font_path = font_path
        self._cache: dict[tuple[str, int, int], tuple[ImageFont.FreeTypeFont | None, float, float]] = {}

    @staticmethod
    def _read_os2_em_ratio(font_path: str) -> float | None:
        """Compute unitsPerEm / (usWinAscent + usWinDescent) matching libass/VSFilter GDI scaling."""
        try:
            from fontTools.ttLib import TTFont  # type: ignore
            tt = TTFont(font_path, fontNumber=0, lazy=True)
            upm = float(tt["head"].unitsPerEm)
            win_h = 0.0
            if "OS/2" in tt:
                os2 = tt["OS/2"]
                win_h = float(getattr(os2, "usWinAscent", 0) + getattr(os2, "usWinDescent", 0))
            if win_h <= 0.0 and "hhea" in tt:
                hhea = tt["hhea"]
                win_h = float(getattr(hhea, "ascent", 0) - getattr(hhea, "descent", 0))
            tt.close()
            if upm > 0.0 and win_h > 0.0:
                return upm / win_h
        except Exception:
            pass
        return None

    def _load_calibrated(
        self,
        fontsize: float,
        scale_y: float = 100.0,
        font_path_override: str | None = None,
    ):
        eff_path = font_path_override or self.font_path
        key = (eff_path or "", int(round(fontsize * 10)), int(round(scale_y * 10)))
        if key in self._cache:
            return self._cache[key]

        target_h = max(8.0, float(fontsize) * (float(scale_y) / 100.0))
        if eff_path and Path(eff_path).is_file():
            try:
                em_ratio = self._read_os2_em_ratio(eff_path)
                if em_ratio is None:
                    ref_px = 200
                    ref_font = ImageFont.truetype(eff_path, ref_px)
                    asc, desc = ref_font.getmetrics()
                    em_ratio = ref_px / max(1.0, float(asc + desc))
                ideal_em = max(8.0, target_h * em_ratio)
                cal_px = max(8, int(round(ideal_em)))
                font = ImageFont.truetype(eff_path, cal_px)
                subpixel_scale = ideal_em / float(cal_px)
                res = (font, subpixel_scale, target_h)
                self._cache[key] = res
                return res
            except Exception as exc:
                print(f"FreeType load warning ({eff_path}): {exc}; falling back to proportional metrics.")

        res = (None, 1.0, target_h)
        self._cache[key] = res
        return res

    @staticmethod
    def _fallback_advance(text: str, fontsize: float) -> float:
        total = 0.0
        for ch in text:
            if ch == " ":
                total += 0.28 * fontsize
            elif ch in "ilIjft.,!|:;'\"`":
                total += 0.27 * fontsize
            elif ch in "mwMW@%":
                total += 0.78 * fontsize
            elif ch.isupper():
                total += 0.62 * fontsize
            elif ch.isdigit():
                total += 0.55 * fontsize
            else:
                total += 0.51 * fontsize
        return total

    def line_height(
        self,
        fontsize: float,
        scale_y: float = 100.0,
        font_path_override: str | None = None,
    ) -> float:
        _, _, line_h = self._load_calibrated(fontsize, scale_y, font_path_override=font_path_override)
        return line_h

    def measure_prefix_widths(
        self,
        full_text: str,
        fontsize: float,
        scale_x: float = 100.0,
        scale_y: float = 100.0,
        spacing: float = 0.0,
        font_path_override: str | None = None,
    ) -> list[float]:
        """Return cumulative x positions [0..len(full_text)] including exact kerning and ASS spacing."""
        sx = float(scale_x) / 100.0
        font, sub_scale, _ = self._load_calibrated(fontsize, scale_y, font_path_override=font_path_override)

        # In ASS, ScaleY scales uniformly; ScaleX scales horizontally relative to ScaleY.
        horiz_ratio = (float(scale_x) / max(1.0, float(scale_y))) if font is not None else sx
        raw_prefixes = [0.0]
        if font is not None:
            for k in range(1, len(full_text) + 1):
                w = float(font.getlength(full_text[:k])) * sub_scale * horiz_ratio
                raw_prefixes.append(w)
        else:
            for k in range(1, len(full_text) + 1):
                raw_prefixes.append(self._fallback_advance(full_text[:k], fontsize) * sx)

        return raw_prefixes


def _parse_ass_time_ms(ts: str) -> int:
    parts = ts.strip().split(":")
    h = int(parts[0])
    m = int(parts[1])
    s = float(parts[2])
    return int(round((h * 3600.0 + m * 60.0 + s) * 1000.0))


def _extract_line_overrides(raw_text: str, style: StyleInfo) -> dict:
    an = style.alignment
    fs = style.fontsize
    sx = style.scale_x
    sy = style.scale_y
    sp = style.spacing
    pos = None

    for blk in re.findall(r"\{([^}]*)\}", raw_text):
        m_an = re.search(r"\\an([1-9])", blk)
        if m_an:
            an = int(m_an.group(1))
        m_fs = re.search(r"\\fs(\d+(?:\.\d+)?)", blk)
        if m_fs:
            fs = float(m_fs.group(1))
        m_sx = re.search(r"\\fscx(\d+(?:\.\d+)?)", blk)
        if m_sx:
            sx = float(m_sx.group(1))
        m_sy = re.search(r"\\fscy(\d+(?:\.\d+)?)", blk)
        if m_sy:
            sy = float(m_sy.group(1))
        m_sp = re.search(r"\\fsp(-?\d+(?:\.\d+)?)", blk)
        if m_sp:
            sp = float(m_sp.group(1))
        m_pos = re.search(r"\\pos\(\s*(-?\d+(?:\.\d+)?)\s*,\s*(-?\d+(?:\.\d+)?)\s*\)", blk)
        if m_pos:
            pos = (float(m_pos.group(1)), float(m_pos.group(2)))

    return {"an": an, "fs": fs, "sx": sx, "sy": sy, "sp": sp, "pos": pos}


def _tokenize_karaoke_syllables(raw_text: str, total_dur_ms: int) -> list[dict]:
    """Parse {\\kXX}, {\\kfXX}, {\\KXX}, {\\koXX} blocks into raw syllable tokens."""
    tag_re = re.compile(r"(\{[^}]*\})")
    tokens = tag_re.split(raw_text)
    k_re = re.compile(r"\\(kf|ko|k|K)(-?\d+)")

    items = []
    pending_k: tuple[str, int] | None = None
    saw_any_k = False

    def clean_visible(s: str) -> str:
        return s.replace("\\N", " ").replace("\\n", " ").replace("\\h", " ")

    for tok in tokens:
        if not tok:
            continue
        if tok.startswith("{") and tok.endswith("}"):
            matches = k_re.findall(tok)
            if matches:
                saw_any_k = True
                # If an earlier k-tag had empty text, flush it as a pause syllable
                if pending_k is not None:
                    items.append({"k_type": pending_k[0], "dur_ms": pending_k[1], "raw_text": ""})
                # Handle multiple \k tags inside a single {...} block
                for k_t, val_s in matches[:-1]:
                    items.append({"k_type": k_t.lower(), "dur_ms": max(0, int(val_s) * 10), "raw_text": ""})
                last_t, last_v = matches[-1]
                pending_k = ("kf" if last_t == "K" else last_t.lower(), max(0, int(last_v) * 10))
        else:
            vis = clean_visible(tok)
            if pending_k is not None:
                items.append({"k_type": pending_k[0], "dur_ms": pending_k[1], "raw_text": vis})
                pending_k = None
            elif vis:
                items.append({"k_type": "k", "dur_ms": 0, "raw_text": vis})

    if pending_k is not None:
        items.append({"k_type": pending_k[0], "dur_ms": pending_k[1], "raw_text": ""})

    if not saw_any_k:
        plain = clean_visible(re.sub(r"\{[^}]*\}", "", raw_text))
        return [{"k_type": "k", "dur_ms": max(1, total_dur_ms), "raw_text": plain}]

    return items


def parse_ass_file(
    ass_text: str,
    font_path: str | None = None,
    style_filter: str = "",
    width_scale: float = 1.0,
    extra_spacing_px: float = 0.0,
    font_is_custom: bool = False,
) -> tuple[ScriptMeta, dict[str, StyleInfo], list[KaraokeLine]]:
    meta = ScriptMeta()
    styles: dict[str, StyleInfo] = {}
    measurer = FontMeasurer(font_path)
    style_font_cache: dict[tuple[str, bool], str | None] = {}
    w_scale = max(0.5, float(width_scale))
    sp_extra = float(extra_spacing_px)

    section = ""
    style_fields = [
        "Name", "Fontname", "Fontsize", "PrimaryColour", "SecondaryColour", "OutlineColour", "BackColour",
        "Bold", "Italic", "Underline", "StrikeOut", "ScaleX", "ScaleY", "Spacing", "Angle",
        "BorderStyle", "Outline", "Shadow", "Alignment", "MarginL", "MarginR", "MarginV", "Encoding",
    ]
    event_fields = ["Layer", "Start", "End", "Style", "Name", "MarginL", "MarginR", "MarginV", "Effect", "Text"]
    raw_events = []

    for raw_line in ass_text.splitlines():
        line = raw_line.strip()
        if not line or line.startswith(";"):
            continue
        if line.startswith("[") and line.endswith("]"):
            section = line.lower()
            continue

        if section == "[script info]":
            meta.raw_info_lines.append(line)
            if ":" in line:
                k, v = [x.strip() for x in line.split(":", 1)]
                kl = k.lower()
                if kl == "title":
                    meta.title = v
                elif kl == "playresx":
                    meta.play_res_x = int(float(v))
                elif kl == "playresy":
                    meta.play_res_y = int(float(v))
                elif kl == "wrapstyle":
                    meta.wrap_style = int(float(v))
                elif kl == "scaledborderandshadow":
                    meta.scaled_border_and_shadow = v

        elif section in ("[v4+ styles]", "[v4 styles]"):
            if line.lower().startswith("format:"):
                style_fields = [f.strip() for f in line.split(":", 1)[1].split(",")]
            elif line.lower().startswith("style:"):
                payload = line.split(":", 1)[1].lstrip()
                cols = [c.strip() for c in payload.split(",", len(style_fields) - 1)]
                row = dict(zip(style_fields, cols))
                st = StyleInfo(
                    name=row.get("Name", "Default"),
                    fontname=row.get("Fontname", "Arial"),
                    fontsize=float(row.get("Fontsize", 64)),
                    primary_colour=row.get("PrimaryColour", "&H00FFFFFF"),
                    secondary_colour=row.get("SecondaryColour", "&H0000FFFF"),
                    outline_colour=row.get("OutlineColour", "&H00000000"),
                    back_colour=row.get("BackColour", "&H80000000"),
                    bold=int(float(row.get("Bold", 0))),
                    italic=int(float(row.get("Italic", 0))),
                    underline=int(float(row.get("Underline", 0))),
                    strikeout=int(float(row.get("StrikeOut", 0))),
                    scale_x=float(row.get("ScaleX", 100)),
                    scale_y=float(row.get("ScaleY", 100)),
                    spacing=float(row.get("Spacing", 0)),
                    angle=float(row.get("Angle", 0)),
                    border_style=int(float(row.get("BorderStyle", 1))),
                    outline=float(row.get("Outline", 2)),
                    shadow=float(row.get("Shadow", 1)),
                    alignment=int(float(row.get("Alignment", 2))),
                    margin_l=int(float(row.get("MarginL", 40))),
                    margin_r=int(float(row.get("MarginR", 40))),
                    margin_v=int(float(row.get("MarginV", 40))),
                    encoding=int(float(row.get("Encoding", 1))),
                    raw_line=line,
                )
                styles[st.name] = st

        elif section == "[events]":
            if line.lower().startswith("format:"):
                event_fields = [f.strip() for f in line.split(":", 1)[1].split(",")]
            elif line.startswith("Dialogue:"):
                payload = line.split(":", 1)[1].lstrip()
                cols = payload.split(",", len(event_fields) - 1)
                if len(cols) == len(event_fields):
                    raw_events.append(dict(zip(event_fields, cols)))

    if not styles:
        styles["Default"] = StyleInfo()

    parsed_lines: list[KaraokeLine] = []
    filt = style_filter.strip().lower()

    for idx, ev in enumerate(raw_events):
        st_name = ev.get("Style", "Default").strip()
        if filt and st_name.lower() != filt:
            continue
        style = styles.get(st_name) or next(iter(styles.values()))
        start_ms = _parse_ass_time_ms(ev["Start"])
        end_ms = _parse_ass_time_ms(ev["End"])
        dur_ms = max(1, end_ms - start_ms)
        raw_text = ev.get("Text", "")

        m_l = int(float(ev.get("MarginL", 0) or 0)) or style.margin_l
        m_r = int(float(ev.get("MarginR", 0) or 0)) or style.margin_r
        m_v = int(float(ev.get("MarginV", 0) or 0)) or style.margin_v

        ov = _extract_line_overrides(raw_text, style)
        an = ov["an"]
        fs = ov["fs"]
        sx = ov["sx"]
        sy = ov["sy"]
        sp = ov["sp"]
        pos = ov["pos"]

        toks = _tokenize_karaoke_syllables(raw_text, dur_ms)
        plain_text = "".join(t["raw_text"] for t in toks)

        if font_is_custom and font_path:
            eff_font_path = font_path
        else:
            f_key = (style.fontname.lower(), bool(style.bold))
            if f_key not in style_font_cache:
                finder = globals().get("find_system_font")
                matched = finder(style.fontname, bold=bool(style.bold)) if callable(finder) else None
                style_font_cache[f_key] = matched or font_path
            eff_font_path = style_font_cache[f_key]

        raw_prefixes = measurer.measure_prefix_widths(
            plain_text, fs, sx, sy, sp, font_path_override=eff_font_path
        )
        raw_prefixes = [w * w_scale for w in raw_prefixes]
        sx_factor = sx / 100.0
        eff_sp = sp + sp_extra
        n_chars = len(plain_text)

        def char_start_x(c_idx: int) -> float:
            return raw_prefixes[c_idx] + c_idx * eff_sp * sx_factor

        def char_end_x(c_idx_excl: int) -> float:
            if c_idx_excl <= 0:
                return 0.0
            return raw_prefixes[c_idx_excl] + max(0, c_idx_excl - 1) * eff_sp * sx_factor

        line_w = char_end_x(n_chars)
        line_h = measurer.line_height(fs, sy, font_path_override=eff_font_path)

        if pos is not None:
            px, py = pos
            h_an = (an - 1) % 3
            v_an = (an - 1) // 3
            line_left = px if h_an == 0 else (px - line_w / 2.0 if h_an == 1 else px - line_w)
            line_bottom = py if v_an == 0 else (py + line_h / 2.0 if v_an == 1 else py + line_h)
            line_top = line_bottom - line_h
        else:
            h_an = (an - 1) % 3
            v_an = (an - 1) // 3
            if h_an == 0:
                line_left = float(m_l)
            elif h_an == 1:
                line_left = float(m_l) + (meta.play_res_x - m_l - m_r - line_w) / 2.0
            else:
                line_left = float(meta.play_res_x - m_r) - line_w

            if v_an == 0:
                line_bottom = float(meta.play_res_y - m_v)
                line_top = line_bottom - line_h
            elif v_an == 1:
                line_top = (meta.play_res_y - line_h) / 2.0
                line_bottom = line_top + line_h
            else:
                line_top = float(m_v)
                line_bottom = line_top + line_h

        line_right = line_left + line_w
        line_center = (line_left + line_right) / 2.0
        line_middle = (line_top + line_bottom) / 2.0

        syllables: list[Syllable] = []
        char_cursor = 0
        time_cursor = 0
        word_idx = 0

        for s_idx, tok in enumerate(toks):
            r_txt = tok["raw_text"]
            s_len = len(r_txt)
            c_start = char_cursor
            c_end = char_cursor + s_len
            char_cursor = c_end

            s_dur = tok["dur_ms"]
            rel_s = time_cursor
            rel_e = time_cursor + s_dur
            time_cursor = rel_e

            stripped = r_txt.strip()
            is_space = len(stripped) == 0

            if is_space:
                vis_l = line_left + char_start_x(c_start)
                vis_r = line_left + (char_end_x(c_end) if s_len > 0 else char_start_x(c_start))
            else:
                lead_sp = len(r_txt) - len(r_txt.lstrip())
                trail_sp = len(r_txt) - len(r_txt.rstrip())
                v_start = c_start + lead_sp
                v_end = c_end - trail_sp
                vis_l = line_left + char_start_x(v_start)
                vis_r = line_left + char_end_x(v_end)

            vis_w = max(0.0, vis_r - vis_l)
            vis_c = (vis_l + vis_r) / 2.0

            syllables.append(
                Syllable(
                    index=s_idx,
                    word_index=word_idx,
                    text=stripped,
                    raw_text=r_txt,
                    k_type=tok["k_type"],
                    start_ms=start_ms + rel_s,
                    end_ms=start_ms + rel_e,
                    duration_ms=s_dur,
                    rel_start_ms=rel_s,
                    rel_end_ms=rel_e,
                    left=round(vis_l, 2),
                    center=round(vis_c, 2),
                    right=round(vis_r, 2),
                    top=round(line_top, 2),
                    middle=round(line_middle, 2),
                    bottom=round(line_bottom, 2),
                    width=round(vis_w, 2),
                    height=round(line_h, 2),
                    x=round(vis_c, 2),
                    y=round(line_middle, 2),
                    is_space=is_space,
                )
            )

            if r_txt.endswith(" ") or is_space:
                word_idx += 1

        parsed_lines.append(
            KaraokeLine(
                index=len(parsed_lines),
                layer=int(float(ev.get("Layer", 0) or 0)),
                start_ms=start_ms,
                end_ms=end_ms,
                duration_ms=dur_ms,
                style=style.name,
                actor=ev.get("Name", "").strip(),
                margin_l=m_l,
                margin_r=m_r,
                margin_v=m_v,
                effect=ev.get("Effect", "").strip(),
                raw_text=raw_text,
                plain_text=plain_text,
                alignment=an,
                fontsize=fs,
                scale_x=sx,
                scale_y=sy,
                spacing=sp,
                left=round(line_left, 2),
                center=round(line_center, 2),
                right=round(line_right, 2),
                top=round(line_top, 2),
                middle=round(line_middle, 2),
                bottom=round(line_bottom, 2),
                width=round(line_w, 2),
                height=round(line_h, 2),
                x=round(line_center, 2),
                y=round(line_middle, 2),
                syllables=syllables,
            )
        )

    return meta, styles, parsed_lines


# ==============================================================================
# ASS Helper Library & Centered \p1 Vector Shapes
# ==============================================================================
SHAPES = {
    "star4": "m 0 -12 l 3 -3 l 12 0 l 3 3 l 0 12 l -3 3 l -12 0 l -3 -3",
    "star5": "m 0 -12 l 3 -4 l 11 -4 l 5 2 l 7 10 l 0 5 l -7 10 l -5 2 l -11 -4 l -3 -4",
    "petal": "m 0 -10 b 7 -6 8 4 0 10 b -8 4 -7 -6 0 -10",
    "circle": "m 0 -10 b 6 -10 10 -6 10 0 b 10 6 6 10 0 10 b -6 10 -10 6 -10 0 b -10 -6 -6 -10 0 -10",
    "diamond": "m 0 -11 l 8 0 l 0 11 l -8 0",
    "heart": "m 0 10 b -12 2 -12 -9 -5 -9 b -1 -9 0 -4 0 -4 b 0 -4 1 -9 5 -9 b 12 -9 12 2 0 10",
    "music_note": "m -3 8 b -7 8 -9 5 -7 3 b -5 1 -1 2 -1 4 l -1 -10 l 9 -12 l 9 -7 l 2 -5 l 2 4 b 2 7 -1 8 -3 8",
}


class KFXHelpers:
    @staticmethod
    def ms_to_ass(ms: int | float) -> str:
        cs = max(0, int(round(float(ms) / 10.0)))
        h, rem = divmod(cs, 360000)
        m, rem = divmod(rem, 6000)
        s, cc = divmod(rem, 100)
        return f"{h:d}:{m:02d}:{s:02d}.{cc:02d}"

    @staticmethod
    def ass_color(hex_rgb: str) -> str:
        s = hex_rgb.strip().lstrip("#")
        if len(s) != 6:
            return "&HFFFFFF&"
        r, g, b = s[0:2].upper(), s[2:4].upper(), s[4:6].upper()
        return f"&H{b}{g}{r}&"

    @staticmethod
    def ass_alpha(val: int) -> str:
        v = max(0, min(255, int(round(val))))
        return f"&H{v:02X}&"

    @staticmethod
    def clamp(val: float, lo: float, hi: float) -> float:
        return max(lo, min(hi, float(val)))

    @staticmethod
    def lerp(a: float, b: float, t: float) -> float:
        return float(a) + (float(b) - float(a)) * float(t)

    @staticmethod
    def lerp_color(hex1: str, hex2: str, t: float) -> str:
        t = max(0.0, min(1.0, float(t)))
        s1 = hex1.strip().lstrip("#")
        s2 = hex2.strip().lstrip("#")
        r1, g1, b1 = int(s1[0:2], 16), int(s1[2:4], 16), int(s1[4:6], 16)
        r2, g2, b2 = int(s2[0:2], 16), int(s2[2:4], 16), int(s2[4:6], 16)
        r = int(round(r1 + (r2 - r1) * t))
        g = int(round(g1 + (g2 - g1) * t))
        b = int(round(b1 + (b2 - b1) * t))
        return f"&H{b:02X}{g:02X}{r:02X}&"

    @staticmethod
    def make_event(
        layer: int,
        start_ms: int | float,
        end_ms: int | float,
        style: str,
        text: str,
        effect: str = "fx",
        actor: str = "",
    ) -> str:
        s_ms = max(0, int(round(start_ms)))
        e_ms = max(s_ms + 10, int(round(end_ms)))
        return (
            f"Dialogue: {int(layer)},{KFXHelpers.ms_to_ass(s_ms)},{KFXHelpers.ms_to_ass(e_ms)},"
            f"{style},{actor},0,0,0,{effect},{text}"
        )

    @staticmethod
    def make_rect(w: float, h: float) -> str:
        hw, hh = int(round(w / 2.0)), int(round(h / 2.0))
        return f"m {-hw} {-hh} l {hw} {-hh} l {hw} {hh} l {-hw} {hh}"

    @staticmethod
    def make_shard(w: float, h: float, rng) -> str:
        """Generate a random centered triangular/quad crystalline shard path for \\p1."""
        sides = rng.choice([3, 4])
        pts = []
        base_angles = sorted(rng.uniform(0, 2 * math.pi) for _ in range(sides))
        for ang in base_angles:
            rx = rng.uniform(0.35, 0.75) * w
            ry = rng.uniform(0.35, 0.75) * h
            pts.append((int(round(math.cos(ang) * rx)), int(round(math.sin(ang) * ry))))
        cmds = [f"m {pts[0][0]} {pts[0][1]}"] + [f"l {px} {py}" for px, py in pts[1:]]
        return " ".join(cmds)


KFX_META, KFX_STYLES, KFX_LINES = parse_ass_file(
    STAGE2_RAW_ASS,
    font_path=STAGE2_FONT_PATH,
    style_filter=TARGET_STYLE_FILTER,
    width_scale=WIDTH_CALIBRATION_SCALE,
    extra_spacing_px=EXTRA_SYLLABLE_SPACING_PX,
    font_is_custom=globals().get("STAGE2_FONT_IS_CUSTOM", False),
)

print(f"Script PlayRes : {KFX_META.play_res_x} x {KFX_META.play_res_y}")
print(f"Styles Loaded  : {list(KFX_STYLES.keys())}")
print(f"Karaoke Lines  : {len(KFX_LINES)}")

if KFX_LINES:
    ln0 = KFX_LINES[0]
    st0 = KFX_STYLES.get(ln0.style) or next(iter(KFX_STYLES.values()))
    resolved_0 = (
        STAGE2_FONT_PATH
        if globals().get("STAGE2_FONT_IS_CUSTOM", False) and STAGE2_FONT_PATH
        else (find_system_font(st0.fontname, bold=bool(st0.bold)) or STAGE2_FONT_PATH)
    )
    measured_fam = extract_ttf_family_name(Path(resolved_0)) if resolved_0 and Path(resolved_0).is_file() else None
    print(f"Style Font     : {st0.fontname} (Bold={bool(st0.bold)}) | Measured Font: {measured_fam or 'Proportional Fallback'} ({resolved_0 or 'none'})")
    if measured_fam:
        req_l = st0.fontname.strip().lower()
        got_l = measured_fam.strip().lower()
        if req_l not in got_l and not (req_l == "arial" and "liberation sans" in got_l):
            print(
                f"  [Warning] Measured font '{measured_fam}' differs from style font '{st0.fontname}'. "
                "For exact syllable tightness, upload/mount that exact .ttf/.otf in Cell 2.1, "
                "or tune WIDTH_CALIBRATION_SCALE / EXTRA_SYLLABLE_SPACING_PX above."
            )
    print(f"\nLayout Check — Line 0: {ln0.plain_text!r}")
    print(
        f"  Line Box: left={ln0.left}, center={ln0.center}, right={ln0.right}, "
        f"top={ln0.top}, middle={ln0.middle}, bottom={ln0.bottom}, w={ln0.width}, h={ln0.height}"
    )
    print(f"  {'Idx':>3}  {'Syl':<8}  {'Dur(ms)':>7}  {'Left':>7}  {'Center':>7}  {'Right':>7}  {'Width':>6}")
    for s in ln0.syllables[:10]:
        print(
            f"  {s.index:>3}  {s.text or '<sp>':<8}  {s.duration_ms:>7}  "
            f"{s.left:>7.1f}  {s.center:>7.1f}  {s.right:>7.1f}  {s.width:>6.1f}"
        )

In [ ]:
#@title 2.3 KFX Mode Toggle, Rule & Palette Builder (UI + JSON) & AI / Preset Generator
import ast
import colorsys
import inspect
import json
import random

MODE_A_LABEL = "Mode A: Shared KFX Animation + Line-Range Colors"
MODE_B_LABEL = "Mode B: Unique KFX Animation + Colors per Line Range"
KFX_RULE_MODES = [MODE_A_LABEL, MODE_B_LABEL]

KFX_RULE_MODE = "Mode A: Shared KFX Animation + Line-Range Colors" #@param ["Mode A: Shared KFX Animation + Line-Range Colors", "Mode B: Unique KFX Animation + Colors per Line Range"]
GENERATION_MODE = "Offline Built-in Presets (No API Key Needed)" #@param ["Offline Built-in Presets (No API Key Needed)", "AI Prompt-Driven Generator (Gemini API)"]
CONFIG_MODE = "Interactive UI (ipywidgets Rule Builder)" #@param ["Interactive UI (ipywidgets Rule Builder)", "JSON Config (Edit JSON Below)"]

# Mode A Master Animation Controls (used when KFX_RULE_MODE == Mode A)
MASTER_PRESET = "neon_glow_bounce" #@param ["neon_glow_bounce", "sakura_petals", "gradient_clip_wipe", "shatter_disintegrate", "sparkle_trail"]
MASTER_KFX_PROMPT = "Soft neon glow on syllable hit, energetic upward bounce, and spinning 4-point star vector particles \\p1 bursting from each syllable center and fading out" #@param {type:"string"}

GEMINI_API_KEY = "" #@param {type:"string"}
GEMINI_MODEL = "gemini-4-hydrogen" #@param ["gemini-4-hydrogen", "gemini-4-argon", "gemini-4-carbon"]
GEMINI_THINKING_LEVEL = "low" #@param ["low", "medium", "high"]
PARTICLE_DENSITY = 1.0 #@param {type:"number"}
MOTION_INTENSITY = 1.0 #@param {type:"number"}
RANDOM_SEED = 42 #@param {type:"integer"}

TARGET_MODES = [
    "Line Number Range",
    "ASS Style Name",
    "All Remaining Unassigned Lines",
]

# Mode A: Lightweight Color-Only Rules (applied to a single shared Master KFX function)
DEFAULT_COLOR_PALETTE_RULES = [
    {
        "name": "Verse Palette (Cool Cyan & Ice)",
        "target_mode": "Line Number Range",
        "target_value": "1-3",
        "primary_hex": "#67E8F9",
        "highlight_hex": "#E0F2FE",
        "glow_hex": "#0284C7",
        "particle_hex": "#A5F3FC",
    },
    {
        "name": "Chorus Palette (Gold, Crimson & Ember)",
        "target_mode": "All Remaining Unassigned Lines",
        "target_value": "",
        "primary_hex": "#FFD700",
        "highlight_hex": "#FF2E54",
        "glow_hex": "#B91C1C",
        "particle_hex": "#FDE047",
    },
]

# Mode B: Full Independent KFX Section Rules (unique animation + colors per range)
DEFAULT_MULTI_SECTION_RULES = [
    {
        "name": "Verse (Calm Blue Glow & Bubbles)",
        "target_mode": "Line Number Range",
        "target_value": "1-3",
        "primary_hex": "#67E8F9",
        "secondary_hex": "#E0F2FE",
        "outline_hex": "#0284C7",
        "particle_hex": "#A5F3FC",
        "preset": "sparkle_trail",
        "prompt": "Soft calm blue glow, slow fade-in, gentle floating water bubbles \\p1 rising from each syllable and fading out",
    },
    {
        "name": "Chorus (Gold & Crimson Bounce + Sparks)",
        "target_mode": "All Remaining Unassigned Lines",
        "target_value": "",
        "primary_hex": "#FFD700",
        "secondary_hex": "#FF2E54",
        "outline_hex": "#B91C1C",
        "particle_hex": "#FDE047",
        "preset": "neon_glow_bounce",
        "prompt": "High-energy chorus: explosive gold and crimson flash, sharp syllable bounce \\fscx130\\fscy130, and fast diagonal spark particles \\p1",
    },
]

# Used directly when CONFIG_MODE == "JSON Config (Edit JSON Below)" or when ipywidgets is unavailable
COLOR_PALETTES_JSON_CONFIG = json.dumps(DEFAULT_COLOR_PALETTE_RULES, indent=2)
RULES_JSON_CONFIG = json.dumps(DEFAULT_MULTI_SECTION_RULES, indent=2)


# ==============================================================================
# Dynamic 4-Color Palette System (\1c, \2c, \3c, \p1) & Function Dispatcher
# ==============================================================================
def _normalize_hex(val: str, fallback: str) -> str:
    s = (val or "").strip()
    if re.fullmatch(r"#[0-9a-fA-F]{6}", s):
        return s.upper()
    if re.fullmatch(r"[0-9a-fA-F]{6}", s):
        return f"#{s.upper()}"
    return fallback


def build_palette_dict(
    primary_hex: str,
    highlight_hex: str,
    glow_hex: str,
    particle_hex: str | None = None,
) -> dict:
    """Build a dynamic 4-color palette dictionary in both ASS BGR (&HBBGGRR&) and HTML #RRGGBB."""
    p_hex = _normalize_hex(primary_hex, "#00F5FF")
    h_hex = _normalize_hex(highlight_hex, "#FF4FD8")
    g_hex = _normalize_hex(glow_hex, "#0284C7")
    pt_hex = _normalize_hex(particle_hex or "", p_hex)
    return {
        "primary_1c": KFXHelpers.ass_color(p_hex),
        "highlight_2c": KFXHelpers.ass_color(h_hex),
        "glow_3c": KFXHelpers.ass_color(g_hex),
        "particle_c": KFXHelpers.ass_color(pt_hex),
        "primary_hex": p_hex,
        "highlight_hex": h_hex,
        "glow_hex": g_hex,
        "particle_hex": pt_hex,
    }


def resolve_palette(config: dict | None = None, palette: dict | None = None) -> dict:
    """Resolve dynamic 4-color palette dict from explicit `palette` or legacy `config` fields."""
    cfg = config or {}
    src = palette if isinstance(palette, dict) else (cfg.get("palette") if isinstance(cfg.get("palette"), dict) else cfg)

    p_hex = src.get("primary_hex") or cfg.get("primary_hex") or "#00F5FF"
    h_hex = src.get("highlight_hex") or src.get("secondary_hex") or cfg.get("highlight_hex") or cfg.get("secondary_hex") or "#FF4FD8"
    g_hex = src.get("glow_hex") or src.get("outline_hex") or cfg.get("glow_hex") or cfg.get("outline_hex") or h_hex
    pt_hex = src.get("particle_hex") or cfg.get("particle_hex") or p_hex
    out = build_palette_dict(p_hex, h_hex, g_hex, pt_hex)

    # Preserve direct ASS overrides if caller supplied raw primary_1c / highlight_2c / glow_3c / particle_c
    if isinstance(src.get("primary_1c"), str) and src["primary_1c"].startswith("&H"):
        out["primary_1c"] = src["primary_1c"]
    if isinstance(src.get("highlight_2c"), str) and src["highlight_2c"].startswith("&H"):
        out["highlight_2c"] = src["highlight_2c"]
    if isinstance(src.get("glow_3c"), str) and src["glow_3c"].startswith("&H"):
        out["glow_3c"] = src["glow_3c"]
    if isinstance(src.get("particle_c"), str) and src["particle_c"].startswith("&H"):
        out["particle_c"] = src["particle_c"]
    return out


def invoke_kfx_function(
    kfx_fn,
    line,
    style,
    meta,
    helpers,
    rng,
    config: dict,
    palette: dict | None = None,
) -> list[str]:
    """Invoke either 7-arg `(..., config, palette)` or legacy 6-arg `(..., config)` KFX functions cleanly."""
    pal = resolve_palette(config, palette)
    cfg = dict(config or {})
    cfg["palette"] = pal
    if palette is not None or "primary_hex" not in cfg:
        cfg["primary_hex"] = pal["primary_hex"]
        cfg["highlight_hex"] = pal["highlight_hex"]
        cfg["secondary_hex"] = pal["highlight_hex"]
        cfg["glow_hex"] = pal["glow_hex"]
        cfg["outline_hex"] = pal["glow_hex"]
        cfg["particle_hex"] = pal["particle_hex"]
        cfg["primary_ass"] = pal["primary_1c"]
        cfg["secondary_ass"] = pal["highlight_2c"]
        cfg["outline_ass"] = pal["glow_3c"]
        cfg["particle_ass"] = pal["particle_c"]

    try:
        sig = inspect.signature(kfx_fn)
        params = list(sig.parameters.values())
        has_varargs = any(p.kind == inspect.Parameter.VAR_POSITIONAL for p in params)
        pos_or_kw = [
            p for p in params
            if p.kind in (inspect.Parameter.POSITIONAL_ONLY, inspect.Parameter.POSITIONAL_OR_KEYWORD)
        ]
        if has_varargs or len(pos_or_kw) >= 7:
            return kfx_fn(line, style, meta, helpers, rng, cfg, pal)
        if any(p.name == "palette" and p.kind == inspect.Parameter.KEYWORD_ONLY for p in params) or any(
            p.kind == inspect.Parameter.VAR_KEYWORD for p in params
        ):
            return kfx_fn(line, style, meta, helpers, rng, cfg, palette=pal)
    except (TypeError, ValueError):
        pass
    return kfx_fn(line, style, meta, helpers, rng, cfg)


# ==============================================================================
# 5 Rich Multi-Layer Built-In Offline KFX Presets (Dynamic 4-Color Palette Aware)
# ==============================================================================
def preset_neon_glow_bounce(line, style, meta, helpers, rng, config, palette=None):
    events = []
    pal = resolve_palette(config, palette)
    c1 = pal["primary_1c"]
    c2 = pal["highlight_2c"]
    c3 = pal["glow_3c"]
    cp = pal["particle_c"]
    white = "&HFFFFFF&"
    dark = "&H1A0810&"
    density = max(0.2, float(config["particle_density"]))
    intensity = max(0.2, float(config["motion_intensity"]))
    amp = 16.0 * intensity
    bsx = float(getattr(line, "scale_x", 100.0))
    bsy = float(getattr(line, "scale_y", 100.0))

    line_in = max(0, line.start_ms - 240)
    line_out = line.end_ms + 280
    fade_in = line.start_ms - line_in
    fade_out = line_out - line.end_ms

    for syl in line.syllables:
        if syl.is_space or not syl.text:
            continue
        cx, cy = syl.center, syl.middle
        dur = max(60, syl.duration_ms)

        # Layer 0: Base crisp shadow border across whole line lifetime
        events.append(
            helpers.make_event(
                0, line_in, line_out, line.style,
                f"{{\\an5\\pos({cx:.2f},{cy:.2f})\\fscx{bsx:g}\\fscy{bsy:g}\\fad({fade_in},{fade_out})\\bord3.5\\shad1.5\\1a&HFF&\\3c{dark}\\4c{dark}}}{syl.text}",
                effect="fx_shadow",
            )
        )

        # Layer 0: Pulsing neon aura during syllable hit using glow_3c + primary_1c
        aura_s = max(0, syl.start_ms - 60)
        aura_e = min(line_out, syl.end_ms + 300)
        peak_t = min(140, max(50, dur // 2))
        aura_dur = aura_e - aura_s
        events.append(
            helpers.make_event(
                0, aura_s, aura_e, line.style,
                f"{{\\an5\\pos({cx:.2f},{cy:.2f})\\fscx{bsx:g}\\fscy{bsy:g}\\blur10\\bord6\\shad0\\1c{c1}\\3c{c3}\\alpha&H85&"
                f"\\t(0,{peak_t},\\fscx{bsx * 1.16:.1f}\\fscy{bsy * 1.16:.1f}\\alpha&H30&)"
                f"\\t({peak_t},{aura_dur},\\fscx{bsx:g}\\fscy{bsy:g}\\alpha&HFF&)}}{syl.text}",
                effect="fx_aura",
            )
        )

        # Layer 1: Pre-hit idle syllable (tinted with highlight_2c)
        if syl.start_ms > line_in:
            events.append(
                helpers.make_event(
                    1, line_in, syl.start_ms, line.style,
                    f"{{\\an5\\pos({cx:.2f},{cy:.2f})\\fscx{bsx:g}\\fscy{bsy:g}\\fad({fade_in},0)\\bord2\\shad0\\1c{c2}\\3c{dark}}}{syl.text}",
                    effect="fx_pre",
                )
            )

        # Layer 1: Active upward bounce + return (2 continuous \move segments)
        up_ms = max(40, int(round(dur * 0.45)))
        down_ms = max(40, dur - up_ms)
        mid_ms = syl.start_ms + up_ms
        end_hit_ms = mid_ms + down_ms

        events.append(
            helpers.make_event(
                1, syl.start_ms, mid_ms, line.style,
                f"{{\\an5\\move({cx:.2f},{cy:.2f},{cx:.2f},{cy - amp:.2f})\\bord2.5\\shad0\\blur1.5"
                f"\\1c{c2}\\3c{c3}\\fscx{bsx:g}\\fscy{bsy:g}"
                f"\\t(0,{up_ms},\\fscx{bsx * 1.16:.1f}\\fscy{bsy * 1.16:.1f}\\1c{c1}\\3c{c3})}}{syl.text}",
                effect="fx_bounce_up",
            )
        )
        events.append(
            helpers.make_event(
                1, mid_ms, end_hit_ms, line.style,
                f"{{\\an5\\move({cx:.2f},{cy - amp:.2f},{cx:.2f},{cy:.2f})\\bord2.5\\shad0\\blur1"
                f"\\1c{c1}\\3c{c3}\\fscx{bsx * 1.16:.1f}\\fscy{bsy * 1.16:.1f}"
                f"\\t(0,{down_ms},\\fscx{bsx:g}\\fscy{bsy:g}\\3c{dark})}}{syl.text}",
                effect="fx_bounce_down",
            )
        )

        # Layer 1: Post-hit lit syllable hold
        if line_out > end_hit_ms:
            events.append(
                helpers.make_event(
                    1, end_hit_ms, line_out, line.style,
                    f"{{\\an5\\pos({cx:.2f},{cy:.2f})\\fscx{bsx:g}\\fscy{bsy:g}\\fad(0,{fade_out})\\bord2\\shad0\\1c{c1}\\3c{c3}}}{syl.text}",
                    effect="fx_post",
                )
            )

        # Layer 2: Bursting 4-point star vector particles (\p1) driven by particle_c
        count = max(3, int(round((5 + syl.width / 22.0) * density)))
        for p_idx in range(count):
            ang = rng.uniform(0.0, 2.0 * math.pi)
            dist = rng.uniform(32.0, 95.0) * intensity
            sx = cx + rng.uniform(-syl.width * 0.35, syl.width * 0.35)
            sy = cy + rng.uniform(-syl.height * 0.25, syl.height * 0.25)
            ex = sx + math.cos(ang) * dist
            ey = sy + math.sin(ang) * dist - rng.uniform(10.0, 32.0) * intensity
            life = rng.randint(360, 720)
            p_start = syl.start_ms + rng.randint(0, min(80, max(10, dur // 3)))
            p_end = p_start + life
            scale0 = rng.randint(45, 95)
            spin = rng.randint(-260, 260)
            r_col = rng.choice([cp, c1, c2, white])
            col = cp if p_idx % 2 == 0 else r_col
            shape = SHAPES["star4"] if rng.random() < 0.78 else SHAPES["circle"]
            events.append(
                helpers.make_event(
                    2, p_start, p_end, line.style,
                    f"{{\\an5\\move({sx:.1f},{sy:.1f},{ex:.1f},{ey:.1f})\\bord0\\shad0\\blur1.5"
                    f"\\1c{col}\\fscx{scale0}\\fscy{scale0}\\frz{rng.randint(0, 180)}\\alpha&H10&"
                    f"\\t(0,{life},\\frz{spin}\\fscx10\\fscy10\\alpha&HFF&)\\p1}}{shape}{{\\p0}}",
                    effect="fx_star_particle",
                )
            )

    return events


def preset_sakura_petals(line, style, meta, helpers, rng, config, palette=None):
    events = []
    pal = resolve_palette(config, palette)
    c1 = pal["primary_1c"]
    c2 = pal["highlight_2c"]
    c3 = pal["glow_3c"]
    cp = pal["particle_c"]
    white = "&HFFFFFF&"
    plum = "&H3A152E&"
    density = max(0.2, float(config["particle_density"]))
    intensity = max(0.2, float(config["motion_intensity"]))
    bsx = float(getattr(line, "scale_x", 100.0))
    bsy = float(getattr(line, "scale_y", 100.0))

    line_in = max(0, line.start_ms - 300)
    line_out = line.end_ms + 360
    fade_in = line.start_ms - line_in
    fade_out = line_out - line.end_ms

    for syl in line.syllables:
        if syl.is_space or not syl.text:
            continue
        cx, cy = syl.center, syl.middle
        dur = max(80, syl.duration_ms)

        # Layer 0: Warm shadow + soft blossom aura (glow_3c)
        events.append(
            helpers.make_event(
                0, line_in, line_out, line.style,
                f"{{\\an5\\pos({cx:.2f},{cy:.2f})\\fscx{bsx:g}\\fscy{bsy:g}\\fad({fade_in},{fade_out})\\bord3\\shad2\\blur1\\1a&HFF&\\3c{plum}\\4c{plum}}}{syl.text}",
                effect="fx_sakura_shadow",
            )
        )
        events.append(
            helpers.make_event(
                0, syl.start_ms, min(line_out, syl.end_ms + 360), line.style,
                f"{{\\an5\\pos({cx:.2f},{cy:.2f})\\fscx{bsx:g}\\fscy{bsy:g}\\blur9\\bord6\\shad0\\1c{c1}\\3c{c3}\\alpha&H90&"
                f"\\t(0,{min(160, dur)},\\fscx{bsx * 1.10:.1f}\\fscy{bsy * 1.10:.1f}\\alpha&H45&)"
                f"\\t({min(160, dur)},{min(line_out, syl.end_ms + 360) - syl.start_ms},\\fscx{bsx:g}\\fscy{bsy:g}\\alpha&HFF&)}}{syl.text}",
                effect="fx_sakura_glow",
            )
        )

        # Layer 1: Pre-hit -> Blossom swell -> Post-hit
        if syl.start_ms > line_in:
            events.append(
                helpers.make_event(
                    1, line_in, syl.start_ms, line.style,
                    f"{{\\an5\\pos({cx:.2f},{cy:.2f})\\fscx{bsx:g}\\fscy{bsy:g}\\fad({fade_in},0)\\bord2\\shad0\\1c{c2}\\3c{plum}}}{syl.text}",
                    effect="fx_pre",
                )
            )
        half = max(50, dur // 2)
        events.append(
            helpers.make_event(
                1, syl.start_ms, syl.end_ms, line.style,
                f"{{\\an5\\pos({cx:.2f},{cy:.2f})\\fscx{bsx:g}\\fscy{bsy:g}\\bord2.5\\shad0\\1c{c2}\\3c{c3}"
                f"\\t(0,{half},\\fscx{bsx * 1.08:.1f}\\fscy{bsy * 1.08:.1f}\\1c{c1})"
                f"\\t({half},{dur},\\fscx{bsx:g}\\fscy{bsy:g}\\3c{plum})}}{syl.text}",
                effect="fx_active",
            )
        )
        if line_out > syl.end_ms:
            events.append(
                helpers.make_event(
                    1, syl.end_ms, line_out, line.style,
                    f"{{\\an5\\pos({cx:.2f},{cy:.2f})\\fscx{bsx:g}\\fscy{bsy:g}\\fad(0,{fade_out})\\bord2\\shad0\\1c{c1}\\3c{c3}}}{syl.text}",
                    effect="fx_post",
                )
            )

        # Layer 2: Drifting tumbling sakura petals (\p1) driven by particle_c
        count = max(3, int(round((4 + syl.width / 25.0) * density)))
        for p_idx in range(count):
            sx = rng.uniform(syl.left - 8.0, syl.right + 8.0)
            sy = rng.uniform(syl.top, syl.bottom)
            dx = rng.uniform(35.0, 110.0) * intensity
            dy = rng.uniform(-35.0, 55.0) * intensity
            life = rng.randint(650, 1200)
            p_start = syl.start_ms + rng.randint(0, min(140, dur))
            p_end = p_start + life
            sc = rng.randint(55, 115)
            r_col = rng.choice([cp, c1, c2, white])
            col = cp if p_idx % 2 == 0 else r_col
            events.append(
                helpers.make_event(
                    2, p_start, p_end, line.style,
                    f"{{\\an5\\move({sx:.1f},{sy:.1f},{sx + dx:.1f},{sy + dy:.1f})\\bord0\\shad0\\blur0.8"
                    f"\\1c{col}\\fscx{sc}\\fscy{sc}"
                    f"\\frx{rng.randint(-40, 40)}\\fry{rng.randint(-40, 40)}\\frz{rng.randint(0, 360)}"
                    f"\\fad(80,{int(life * 0.45)})"
                    f"\\t(0,{life},\\frx{rng.randint(90, 260)}\\fry{rng.randint(90, 260)}\\frz{rng.randint(180, 540)})"
                    f"\\p1}}{SHAPES['petal']}{{\\p0}}",
                    effect="fx_sakura_petal",
                )
            )
    return events


def preset_gradient_clip_wipe(line, style, meta, helpers, rng, config, palette=None):
    events = []
    pal = resolve_palette(config, palette)
    hex1 = pal["primary_hex"]
    hex2 = pal["highlight_hex"]
    c3 = pal["glow_3c"]
    cp = pal["particle_c"]
    white = "&HFFFFFF&"
    dark = "&H14101C&"
    density = max(0.2, float(config["particle_density"]))
    bsx = float(getattr(line, "scale_x", 100.0))
    bsy = float(getattr(line, "scale_y", 100.0))

    line_in = max(0, line.start_ms - 220)
    line_out = line.end_ms + 260
    fade_in = line.start_ms - line_in
    fade_out = line_out - line.end_ms
    bands = 5

    for syl in line.syllables:
        if syl.is_space or not syl.text:
            continue
        cx, cy = syl.center, syl.middle
        x1 = int(math.floor(syl.left - 8))
        x2 = int(math.ceil(syl.right + 8))
        y1 = int(math.floor(syl.top - 10))
        y2 = int(math.ceil(syl.bottom + 10))
        t1 = max(0, syl.start_ms - line_in)
        t2 = max(t1 + 20, syl.end_ms - line_in)

        # Layer 0: Clean shadow & border
        events.append(
            helpers.make_event(
                0, line_in, line_out, line.style,
                f"{{\\an5\\pos({cx:.2f},{cy:.2f})\\fscx{bsx:g}\\fscy{bsy:g}\\fad({fade_in},{fade_out})\\bord3\\shad2\\1a&HFF&\\3c{dark}\\4c{dark}}}{syl.text}",
                effect="fx_grad_shadow",
            )
        )
        # Layer 1: Inactive base fill
        events.append(
            helpers.make_event(
                1, line_in, line_out, line.style,
                f"{{\\an5\\pos({cx:.2f},{cy:.2f})\\fscx{bsx:g}\\fscy{bsy:g}\\fad({fade_in},{fade_out})\\bord2\\shad0\\1c{white}\\3c{dark}}}{syl.text}",
                effect="fx_grad_base",
            )
        )
        # Layer 2: Multi-band vertical gradient \clip karaoke sweep (primary->highlight, glow_3c outline)
        band_h = (y2 - y1) / float(bands)
        for b in range(bands):
            by1 = int(round(y1 + b * band_h))
            by2 = int(round(y1 + (b + 1) * band_h)) if b < bands - 1 else y2
            band_col = helpers.lerp_color(hex1, hex2, b / max(1, bands - 1))
            events.append(
                helpers.make_event(
                    2, line_in, line_out, line.style,
                    f"{{\\an5\\pos({cx:.2f},{cy:.2f})\\fscx{bsx:g}\\fscy{bsy:g}\\fad({fade_in},{fade_out})\\bord2\\shad0\\1c{band_col}\\3c{c3}"
                    f"\\clip({x1},{by1},{x1},{by2})\\t({t1},{t2},\\clip({x1},{by1},{x2},{by2}))}}{syl.text}",
                    effect=f"fx_grad_band_{b}",
                )
            )

        # Layer 3: Leading-edge shimmer glint gliding across the wipe using particle_c
        glint_n = max(1, int(round(2 * density)))
        for g_idx in range(glint_n):
            gy = cy + (g_idx - (glint_n - 1) / 2.0) * (syl.height * 0.25)
            events.append(
                helpers.make_event(
                    3, syl.start_ms, syl.end_ms + 120, line.style,
                    f"{{\\an5\\move({syl.left:.1f},{gy:.1f},{syl.right:.1f},{gy:.1f},0,{max(20, syl.duration_ms)})"
                    f"\\bord0\\shad0\\blur2\\1c{cp}\\fscx75\\fscy75\\fad(20,120)"
                    f"\\t(0,{max(40, syl.duration_ms)},\\frz180)\\p1}}{SHAPES['star4']}{{\\p0}}",
                    effect="fx_wipe_glint",
                )
            )
    return events


def preset_shatter_disintegrate(line, style, meta, helpers, rng, config, palette=None):
    events = []
    pal = resolve_palette(config, palette)
    c1 = pal["primary_1c"]
    c2 = pal["highlight_2c"]
    c3 = pal["glow_3c"]
    cp = pal["particle_c"]
    dark = "&H101218&"
    density = max(0.2, float(config["particle_density"]))
    intensity = max(0.2, float(config["motion_intensity"]))
    bsx = float(getattr(line, "scale_x", 100.0))
    bsy = float(getattr(line, "scale_y", 100.0))

    line_in = max(0, line.start_ms - 180)
    fade_in = line.start_ms - line_in

    for syl in line.syllables:
        if syl.is_space or not syl.text:
            continue
        cx, cy = syl.center, syl.middle
        dur = max(60, syl.duration_ms)
        shatter_t = syl.end_ms
        dissolve_end = shatter_t + 180

        # Layer 0: Ice-sharp shadow before shatter
        events.append(
            helpers.make_event(
                0, line_in, dissolve_end, line.style,
                f"{{\\an5\\pos({cx:.2f},{cy:.2f})\\fscx{bsx:g}\\fscy{bsy:g}\\fad({fade_in},140)\\bord3\\shad1.5\\1a&HFF&\\3c{dark}\\4c{dark}}}{syl.text}",
                effect="fx_shatter_shadow",
            )
        )
        # Layer 1: Pre-hit
        if syl.start_ms > line_in:
            events.append(
                helpers.make_event(
                    1, line_in, syl.start_ms, line.style,
                    f"{{\\an5\\pos({cx:.2f},{cy:.2f})\\fscx{bsx:g}\\fscy{bsy:g}\\fad({fade_in},0)\\bord2\\shad0\\1c{c2}\\3c{dark}}}{syl.text}",
                    effect="fx_pre",
                )
            )
        # Layer 1: High-impact punch + rapid fracture flash
        punch_t = min(90, max(40, dur // 3))
        events.append(
            helpers.make_event(
                1, syl.start_ms, shatter_t, line.style,
                f"{{\\an5\\pos({cx:.2f},{cy:.2f})\\bord3\\shad0\\blur2\\1c{c2}\\3c{c3}\\fscx{bsx * 1.22:.1f}\\fscy{bsy * 1.22:.1f}"
                f"\\t(0,{punch_t},\\fscx{bsx * 1.03:.1f}\\fscy{bsy * 1.03:.1f}\\1c{c1}\\3c{c3}\\blur0.5)}}{syl.text}",
                effect="fx_impact",
            )
        )
        # Layer 1: Quick dissolve as shards burst out
        events.append(
            helpers.make_event(
                1, shatter_t, dissolve_end, line.style,
                f"{{\\an5\\pos({cx:.2f},{cy:.2f})\\fscx{bsx:g}\\fscy{bsy:g}\\bord2\\shad0\\1c{c1}\\3c{c3}"
                f"\\t(0,180,\\fscx{bsx * 1.15:.1f}\\fscy{bsy * 0.88:.1f}\\blur12\\alpha&HFF&)}}{syl.text}",
                effect="fx_dissolve",
            )
        )

        # Layer 2: Crystalline \p1 shards driven by particle_c
        cols = max(3, int(round(max(2.0, syl.width / 16.0) * math.sqrt(density))))
        rows = max(2, int(round(3 * math.sqrt(density))))
        cell_w = max(8.0, syl.width / cols)
        cell_h = max(8.0, syl.height / rows)
        for r in range(rows):
            for c_i in range(cols):
                sx = syl.left + (c_i + 0.5) * cell_w + rng.uniform(-3.0, 3.0)
                sy = syl.top + (r + 0.5) * cell_h + rng.uniform(-3.0, 3.0)
                dx = (sx - cx) * rng.uniform(1.8, 4.2) * intensity + rng.uniform(-30, 30)
                dy = (sy - cy) * rng.uniform(1.2, 2.8) * intensity + rng.uniform(15, 95) * intensity
                life = rng.randint(420, 780)
                p_start = shatter_t + rng.randint(0, 40)
                p_end = p_start + life
                shard_path = helpers.make_shard(max(6.0, cell_w * 0.75), max(6.0, cell_h * 0.75), rng)
                r_col = rng.choice([cp, c1, c2, c3])
                col = cp if (r + c_i) % 2 == 0 else r_col
                spin = rng.randint(-520, 520)
                events.append(
                    helpers.make_event(
                        2, p_start, p_end, line.style,
                        f"{{\\an5\\move({sx:.1f},{sy:.1f},{sx + dx:.1f},{sy + dy:.1f})\\bord0.8\\shad0\\blur0.5"
                        f"\\1c{col}\\3c{dark}\\frz{rng.randint(0, 180)}\\fad(0,{int(life * 0.55)})"
                        f"\\t(0,{life},\\frz{spin}\\fscx35\\fscy35)\\p1}}{shard_path}{{\\p0}}",
                        effect="fx_shard",
                    )
                )
    return events


def preset_sparkle_trail(line, style, meta, helpers, rng, config, palette=None):
    events = []
    pal = resolve_palette(config, palette)
    c1 = pal["primary_1c"]
    c2 = pal["highlight_2c"]
    c3 = pal["glow_3c"]
    cp = pal["particle_c"]
    white = "&HFFFFFF&"
    dark = "&H120C24&"
    density = max(0.2, float(config["particle_density"]))
    intensity = max(0.2, float(config["motion_intensity"]))
    bsx = float(getattr(line, "scale_x", 100.0))
    bsy = float(getattr(line, "scale_y", 100.0))

    line_in = max(0, line.start_ms - 250)
    line_out = line.end_ms + 300
    fade_in = line.start_ms - line_in
    fade_out = line_out - line.end_ms

    for syl in line.syllables:
        if syl.is_space or not syl.text:
            continue
        cx, cy = syl.center, syl.middle
        dur = max(60, syl.duration_ms)
        x1 = int(math.floor(syl.left - 8))
        x2 = int(math.ceil(syl.right + 8))
        y1 = int(math.floor(syl.top - 10))
        y2 = int(math.ceil(syl.bottom + 10))
        t1 = max(0, syl.start_ms - line_in)
        t2 = max(t1 + 20, syl.end_ms - line_in)

        # Layer 0: Glowing aura halo (glow_3c)
        events.append(
            helpers.make_event(
                0, line_in, line_out, line.style,
                f"{{\\an5\\pos({cx:.2f},{cy:.2f})\\fscx{bsx:g}\\fscy{bsy:g}\\fad({fade_in},{fade_out})\\bord4.5\\shad1.5\\blur4\\1a&HFF&\\3c{c3}\\4c{dark}}}{syl.text}",
                effect="fx_trail_shadow",
            )
        )
        # Layer 1: Base + smooth luminous clip wipe
        events.append(
            helpers.make_event(
                1, line_in, line_out, line.style,
                f"{{\\an5\\pos({cx:.2f},{cy:.2f})\\fscx{bsx:g}\\fscy{bsy:g}\\fad({fade_in},{fade_out})\\bord2\\shad0\\1c{c2}\\3c{dark}}}{syl.text}",
                effect="fx_trail_base",
            )
        )
        events.append(
            helpers.make_event(
                2, line_in, line_out, line.style,
                f"{{\\an5\\pos({cx:.2f},{cy:.2f})\\fscx{bsx:g}\\fscy{bsy:g}\\fad({fade_in},{fade_out})\\bord2\\shad0\\blur0.8\\1c{c1}\\3c{c3}"
                f"\\clip({x1},{y1},{x1},{y2})\\t({t1},{t2},\\clip({x1},{y1},{x2},{y2}))}}{syl.text}",
                effect="fx_trail_lit",
            )
        )

        # Layer 3: Progressive comet sparkle/bubble trail driven by particle_c
        count = max(4, int(round((5 + syl.width / 18.0) * density)))
        for idx in range(count):
            frac = idx / max(1, count - 1) if count > 1 else 0.5
            sx = helpers.lerp(syl.left, syl.right, frac) + rng.uniform(-4.0, 4.0)
            sy = cy + rng.uniform(-syl.height * 0.3, syl.height * 0.3)
            dx = rng.uniform(-24.0, 14.0) * intensity
            dy = rng.uniform(-52.0, -16.0) * intensity
            life = rng.randint(420, 820)
            p_start = syl.start_ms + int(round(frac * dur))
            p_end = p_start + life
            sc_peak = rng.randint(65, 115)
            shape = SHAPES["circle"] if "bubble" in config.get("prompt", "").lower() else (
                SHAPES["star4"] if idx % 2 == 0 else SHAPES["diamond"]
            )
            r_col = rng.choice([cp, c1, c2, white])
            col = cp if idx % 2 == 0 else r_col
            events.append(
                helpers.make_event(
                    3, p_start, p_end, line.style,
                    f"{{\\an5\\move({sx:.1f},{sy:.1f},{sx + dx:.1f},{sy + dy:.1f})\\bord0\\shad0\\blur1.2"
                    f"\\1c{col}\\fscx20\\fscy20\\frz{rng.randint(0, 90)}\\fad(30,{int(life * 0.55)})"
                    f"\\t(0,{int(life * 0.35)},\\fscx{sc_peak}\\fscy{sc_peak})"
                    f"\\t({int(life * 0.35)},{life},\\fscx15\\fscy15\\frz{rng.randint(120, 320)})"
                    f"\\p1}}{shape}{{\\p0}}",
                    effect="fx_sparkle_trail",
                )
            )
    return events


BUILTIN_PRESETS = {
    "neon_glow_bounce": preset_neon_glow_bounce,
    "sakura_petals": preset_sakura_petals,
    "gradient_clip_wipe": preset_gradient_clip_wipe,
    "shatter_disintegrate": preset_shatter_disintegrate,
    "sparkle_trail": preset_sparkle_trail,
}


# ==============================================================================
# Safe AST Sandbox & Gemini API (google-genai) Code Generator
# ==============================================================================
KFX_SYSTEM_PROMPT = r"""You are an expert Aegisub ASS Karaoke Effects (KFX) engineer writing pure Python 3 code.

Write ONLY a single Python function with this exact signature (plus any small pure-Python helper functions if needed, no prose or explanations):
def generate_line_kfx(line, style, meta, helpers, rng, config, palette) -> list[str]:

Available objects passed into your function:
- line (KaraokeLine): index, start_ms, end_ms, duration_ms, style, alignment, left, center, right, top, middle, bottom, width, height, x, y, scale_x, scale_y, plain_text, syllables
- syl (each item in line.syllables): index, word_index, text, raw_text, k_type, start_ms, end_ms, duration_ms, rel_start_ms, rel_end_ms, left, center, right, top, middle, bottom, width, height, x, y, is_space
  - ALWAYS skip whitespace syllables: `if syl.is_space or not syl.text: continue`
- style (StyleInfo): name, fontname, fontsize, primary_colour, secondary_colour, outline_colour, back_colour, outline, shadow, alignment
- meta (ScriptMeta): play_res_x, play_res_y
- helpers (KFXHelpers):
  - helpers.ass_color("#RRGGBB") -> "&HBBGGRR&"  (ASS uses BGR order! NEVER format RGB directly!)
  - helpers.ass_alpha(0..255) -> "&HXX&"
  - helpers.lerp(a, b, t) -> float
  - helpers.lerp_color(hex1, hex2, t) -> "&HBBGGRR&"
  - helpers.clamp(val, lo, hi) -> float
  - helpers.make_event(layer, start_ms, end_ms, style_name, text, effect="fx", actor="") -> str
  - helpers.make_rect(w, h) -> str (for \\p1)
  - helpers.make_shard(w, h, rng) -> str (for \\p1)
- rng: seeded random.Random instance
- palette (dict): Dynamic 4-color palette dictionary passed per line:
  - palette["primary_1c"]: ASS BGR string "&HBBGGRR&" for Primary fill (\\1c)
  - palette["highlight_2c"]: ASS BGR string "&HBBGGRR&" for Pre-hit / Highlight / Secondary fill (\\1c or \\2c)
  - palette["glow_3c"]: ASS BGR string "&HBBGGRR&" for Outline / Neon Glow aura (\\3c)
  - palette["particle_c"]: ASS BGR string "&HBBGGRR&" for \\p1 Vector Particles (\\1c)
  - Also contains HTML hex counterparts: palette["primary_hex"], palette["highlight_hex"], palette["glow_hex"], palette["particle_hex"] (useful with helpers.lerp_color).
- config (dict):
  - "particle_density": float
  - "motion_intensity": float
  - "rule_name": str
  - "prompt": str
  - "palette": same palette dict as the `palette` argument
- Global dicts/modules already available in scope: math, random, colorsys, SHAPES
  SHAPES keys (all centered at (0,0) around [-12,12]): "star4", "star5", "petal", "circle", "diamond", "heart", "music_note".

CRITICAL ASS / LIBASS / PALETTE RULES:
1. NEVER hardcode &H...& or #RRGGBB colors for syllable fills, highlights, glows, or \\p1 particles. ALWAYS read them dynamically from `palette["primary_1c"]`, `palette["highlight_2c"]`, `palette["glow_3c"]`, and `palette["particle_c"]` so the exact same function switches colors smoothly across different line ranges!
2. Never put both \\pos(...) and \\move(...) in the same event. Use \\an5\\pos(syl.center, syl.middle) OR \\an5\\move(x1,y1,x2,y2).
3. In \\t(t1, t2, ...), times t1 and t2 are ALWAYS in milliseconds RELATIVE to that single Dialogue event's own Start time (starting at 0), NOT absolute timestamps.
4. For \\p1 vector drawings: wrap with `{\\an5\\move(...)\\bord0\\shad0\\1c{palette['particle_c']}...\\p1}{SHAPES['star4']}{\\p0}`.
5. Generate rich multi-layer output: Layer 0 shadow/glow, Layer 1+ syllable animations/transforms/clips, Layer 2+ randomized \\p1 vector particles.
6. Output ONLY valid Python code (you may enclose in ```python ... ``` or output raw Python). Do not import os, sys, subprocess, or touch files.
"""

BLOCKED_AST_NAMES = {
    "eval", "exec", "compile", "open", "__import__", "getattr", "setattr",
    "delattr", "globals", "locals", "vars", "dir", "breakpoint", "input",
    "memoryview", "object", "type",
}
ALLOWED_IMPORTS = {"math", "random", "colorsys"}


def extract_python_code(text: str) -> str:
    text = (text or "").strip()
    if not text:
        return ""

    blocks: list[str] = []
    in_fence = False
    cur_lines: list[str] = []
    for ln in text.splitlines():
        s = ln.strip()
        if not in_fence:
            if s.startswith("```"):
                in_fence = True
                cur_lines = []
        else:
            if s == "```":
                blocks.append("\n".join(cur_lines).strip())
                in_fence = False
                cur_lines = []
            else:
                cur_lines.append(ln)
    if in_fence and cur_lines:
        blocks.append("\n".join(cur_lines).strip())

    non_empty = [b for b in blocks if b]
    if non_empty:
        for blk in non_empty:
            if "def generate_line_kfx" in blk:
                return blk
        return max(non_empty, key=len)

    cleaned = re.sub(r"^```[A-Za-z0-9_-]*\s*", "", text)
    cleaned = re.sub(r"\s*```$", "", cleaned).strip()
    m_code = re.search(r"^(?:import |from |def generate_line_kfx\b)", cleaned, flags=re.MULTILINE)
    if m_code and not re.match(r"^(?:import |from |def |#)", cleaned):
        cleaned = cleaned[m_code.start():].strip()
    return cleaned


def validate_kfx_ast(code: str) -> None:
    tree = ast.parse(code)
    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for alias in node.names:
                if alias.name.split(".")[0] not in ALLOWED_IMPORTS:
                    raise ValueError(f"Disallowed import in generated KFX code: {alias.name}")
        elif isinstance(node, ast.ImportFrom):
            mod = (node.module or "").split(".")[0]
            if mod not in ALLOWED_IMPORTS:
                raise ValueError(f"Disallowed from-import in generated KFX code: {node.module}")
        elif isinstance(node, ast.Name):
            if node.id in BLOCKED_AST_NAMES or node.id.startswith("__"):
                raise ValueError(f"Disallowed identifier in generated KFX code: {node.id}")
        elif isinstance(node, ast.Attribute):
            if node.attr.startswith("_"):
                raise ValueError(f"Disallowed private/dunder attribute access: .{node.attr}")


def compile_kfx_function(
    code: str,
    sample_line,
    sample_style,
    meta,
    seed: int = 42,
    rule_cfg: dict | None = None,
    verify_palette_switch: bool = False,
):
    validate_kfx_ast(code)

    def safe_import(name, globals=None, locals=None, fromlist=(), level=0):
        root = name.split(".")[0]
        if root not in ALLOWED_IMPORTS:
            raise ImportError(f"Blocked import: {name}")
        return __import__(name, globals, locals, fromlist, level)

    safe_builtins = {
        "abs": abs, "all": all, "any": any, "bool": bool, "dict": dict,
        "enumerate": enumerate, "float": float, "int": int, "len": len,
        "list": list, "max": max, "min": min, "pow": pow, "range": range,
        "reversed": reversed, "round": round, "set": set, "sorted": sorted,
        "str": str, "sum": sum, "tuple": tuple, "zip": zip,
        "__import__": safe_import,
    }
    ns = {
        "__builtins__": safe_builtins,
        "math": math,
        "random": random,
        "colorsys": colorsys,
        "SHAPES": SHAPES,
    }
    exec(compile(code, "<ai_generated_kfx>", "exec"), ns, ns)
    fn = ns.get("generate_line_kfx")
    if not callable(fn):
        raise ValueError("Generated code did not define callable generate_line_kfx(line, style, meta, helpers, rng, config, palette).")

    if sample_line is not None:
        pal_default = build_palette_dict("#00F5FF", "#FF4FD8", "#0284C7", "#A5F3FC")
        dry_cfg = dict(rule_cfg) if rule_cfg else {
            "primary_hex": "#00F5FF",
            "highlight_hex": "#FF4FD8",
            "secondary_hex": "#FF4FD8",
            "glow_hex": "#0284C7",
            "outline_hex": "#0284C7",
            "particle_hex": "#A5F3FC",
            "primary_ass": pal_default["primary_1c"],
            "secondary_ass": pal_default["highlight_2c"],
            "outline_ass": pal_default["glow_3c"],
            "particle_ass": pal_default["particle_c"],
            "palette": pal_default,
            "particle_density": PARTICLE_DENSITY,
            "motion_intensity": MOTION_INTENSITY,
            "rule_name": "dry_run",
            "prompt": "",
        }
        if verify_palette_switch:
            pal_a = build_palette_dict("#00F5FF", "#E0F2FE", "#0284C7", "#A5F3FC")
            pal_b = build_palette_dict("#FFD700", "#FF2E54", "#B91C1C", "#FDE047")
            out_a = invoke_kfx_function(fn, sample_line, sample_style, meta, KFXHelpers, random.Random(seed), dry_cfg, pal_a)
            if not isinstance(out_a, list) or not all(isinstance(x, str) and x.startswith("Dialogue:") for x in out_a):
                raise ValueError("generate_line_kfx must return list[str] where every item starts with 'Dialogue:'.")
            out_b = invoke_kfx_function(fn, sample_line, sample_style, meta, KFXHelpers, random.Random(seed), dry_cfg, pal_b)
            if not isinstance(out_b, list) or not all(isinstance(x, str) and x.startswith("Dialogue:") for x in out_b):
                raise ValueError("generate_line_kfx must return list[str] where every item starts with 'Dialogue:'.")
            if out_a == out_b:
                raise ValueError(
                    "Generated function produced identical ASS output for two distinct color palettes. "
                    "Do not hardcode colors; read palette['primary_1c'], palette['highlight_2c'], palette['glow_3c'], and palette['particle_c']."
                )
        else:
            out = invoke_kfx_function(fn, sample_line, sample_style, meta, KFXHelpers, random.Random(seed), dry_cfg, dry_cfg.get("palette"))
            if not isinstance(out, list) or not all(isinstance(x, str) and x.startswith("Dialogue:") for x in out):
                raise ValueError("generate_line_kfx must return list[str] where every item starts with 'Dialogue:'.")
    return fn


def resolve_gemini_api_key(manual_key: str) -> str:
    if manual_key and manual_key.strip():
        return manual_key.strip()
    if os.environ.get("GEMINI_API_KEY"):
        return os.environ["GEMINI_API_KEY"].strip()
    if IN_COLAB:
        try:
            from google.colab import userdata  # type: ignore
            secret = userdata.get("GEMINI_API_KEY")
            if secret:
                return secret.strip()
        except Exception:
            pass
    return ""


def generate_kfx_with_gemini(
    prompt: str,
    api_key: str,
    model_name: str,
    sample_line,
    sample_style,
    meta,
    thinking_level: str = "low",
    rule_cfg: dict | None = None,
    shared_palette_mode: bool = False,
):
    from google import genai  # type: ignore

    pal_default = build_palette_dict("#00F5FF", "#FF4FD8", "#0284C7", "#A5F3FC")
    cfg = rule_cfg or {
        "primary_hex": "#00F5FF",
        "highlight_hex": "#FF4FD8",
        "secondary_hex": "#FF4FD8",
        "glow_hex": "#0284C7",
        "outline_hex": "#0284C7",
        "particle_hex": "#A5F3FC",
        "primary_ass": pal_default["primary_1c"],
        "secondary_ass": pal_default["highlight_2c"],
        "outline_ass": pal_default["glow_3c"],
        "particle_ass": pal_default["particle_c"],
        "palette": pal_default,
        "particle_density": PARTICLE_DENSITY,
        "motion_intensity": MOTION_INTENSITY,
        "rule_name": "Shared Master KFX" if shared_palette_mode else "Section Rule",
        "prompt": prompt,
    }
    pal = resolve_palette(cfg)

    client = genai.Client(api_key=api_key)
    if shared_palette_mode:
        user_payload = (
            f"Mode: Shared Master Song-Wide KFX Animation (Dynamic Per-Range Color Palettes)\n"
            f"Design a single reusable multi-layer Aegisub karaoke effect matching this creative description:\n"
            f"{prompt}\n\n"
            f"Style context: PlayRes={meta.play_res_x}x{meta.play_res_y}, Font={sample_style.fontname}, "
            f"Fontsize={sample_line.fontsize if sample_line else sample_style.fontsize}, "
            f"ParticleDensity={cfg['particle_density']}, MotionIntensity={cfg['motion_intensity']}.\n"
            f"IMPORTANT REQUIREMENT: This single function will be called across different song sections with DIFFERENT `palette` dicts. "
            f"You MUST read ALL colors dynamically from `palette['primary_1c']`, `palette['highlight_2c']`, `palette['glow_3c']`, and `palette['particle_c']` "
            f"(or `palette['primary_hex']` / `palette['highlight_hex']` with `helpers.lerp_color`), and NEVER hardcode any fixed lyric/glow/particle color hex literals."
        )
    else:
        user_payload = (
            f"Section Rule Name: {cfg.get('rule_name', 'Section Rule')}\n"
            f"Design a multi-layer Aegisub karaoke effect matching this creative description:\n"
            f"{prompt}\n\n"
            f"Style context: PlayRes={meta.play_res_x}x{meta.play_res_y}, Font={sample_style.fontname}, "
            f"Fontsize={sample_line.fontsize if sample_line else sample_style.fontsize}, "
            f"PrimaryHex(\\1c)={pal['primary_hex']} ({pal['primary_1c']}), "
            f"HighlightHex(\\2c)={pal['highlight_hex']} ({pal['highlight_2c']}), "
            f"GlowHex(\\3c)={pal['glow_hex']} ({pal['glow_3c']}), "
            f"ParticleHex(\\p1)={pal['particle_hex']} ({pal['particle_c']}), "
            f"ParticleDensity={cfg['particle_density']}, MotionIntensity={cfg['motion_intensity']}.\n"
            f"Read colors via `palette['primary_1c']`, `palette['highlight_2c']`, `palette['glow_3c']`, and `palette['particle_c']`."
        )

    label = cfg.get("rule_name", "Shared Master KFX" if shared_palette_mode else "Rule")
    print(f"Calling Gemini API ({model_name}, thinking_level={thinking_level}) for '{label}'...")
    interaction = client.interactions.create(
        model=model_name,
        system_instruction=KFX_SYSTEM_PROMPT,
        input=user_payload,
        generation_config={
            "thinking_level": thinking_level,
            "max_output_tokens": 32768,
        },
    )
    status = getattr(interaction, "status", "completed")
    code = extract_python_code(interaction.output_text or "")

    try:
        if not code:
            raise ValueError(f"Empty model output (status={status})")
        fn = compile_kfx_function(
            code,
            sample_line,
            sample_style,
            meta,
            seed=RANDOM_SEED,
            rule_cfg=cfg,
            verify_palette_switch=shared_palette_mode,
        )
        return fn, code
    except Exception as first_err:
        print(f"Initial generated KFX failed validation (status={status}, error={first_err}); requesting 1 repair pass...")
        if status == "completed" and getattr(interaction, "id", None):
            repair = client.interactions.create(
                model=model_name,
                previous_interaction_id=interaction.id,
                system_instruction=KFX_SYSTEM_PROMPT,
                input=f"Your code failed validation with error: {first_err}. Fix it and return ONLY valid Python code.",
                generation_config={
                    "thinking_level": "low",
                    "max_output_tokens": 32768,
                },
            )
        else:
            # Never pass previous_interaction_id for incomplete/failed interactions (API returns 400).
            repair = client.interactions.create(
                model=model_name,
                system_instruction=KFX_SYSTEM_PROMPT,
                input=(
                    f"{user_payload}\n\n"
                    f"Important: A previous attempt returned incomplete/invalid output (status={status}, error={first_err}). "
                    "Keep helper logic concise and return the complete, syntactically valid Python generate_line_kfx function."
                ),
                generation_config={
                    "thinking_level": "low",
                    "max_output_tokens": 65536,
                },
            )
        repair_status = getattr(repair, "status", "completed")
        repaired_code = extract_python_code(repair.output_text or "")
        if not repaired_code:
            raise ValueError(f"Empty repair output (status={repair_status})")
        fn = compile_kfx_function(
            repaired_code,
            sample_line,
            sample_style,
            meta,
            seed=RANDOM_SEED,
            rule_cfg=cfg,
            verify_palette_switch=shared_palette_mode,
        )
        return fn, repaired_code


# ==============================================================================
# Rule Normalization, Line Mapping, Mode A (Shared) & Mode B (Per-Section) Engines
# ==============================================================================
def normalize_rule(raw: dict, idx: int, default_prefix: str = "Section Rule") -> dict:
    p_hex = _normalize_hex(raw.get("primary_hex", ""), "#00F5FF")
    h_hex = _normalize_hex(raw.get("highlight_hex") or raw.get("secondary_hex") or "", "#FF4FD8")
    g_hex = _normalize_hex(raw.get("glow_hex") or raw.get("outline_hex") or "", "#0284C7")
    pt_hex = _normalize_hex(raw.get("particle_hex", ""), p_hex)
    pal = build_palette_dict(p_hex, h_hex, g_hex, pt_hex)

    mode = raw.get("target_mode", "Line Number Range")
    if mode not in TARGET_MODES:
        mode = "All Remaining Unassigned Lines"
    preset = raw.get("preset", "neon_glow_bounce")
    if preset not in BUILTIN_PRESETS:
        preset = "neon_glow_bounce"

    return {
        "id": str(raw.get("id") or f"rule_{idx + 1}"),
        "name": str(raw.get("name") or f"{default_prefix} #{idx + 1}").strip(),
        "target_mode": mode,
        "target_value": str(raw.get("target_value") or "").strip(),
        "primary_hex": pal["primary_hex"],
        "highlight_hex": pal["highlight_hex"],
        "secondary_hex": pal["highlight_hex"],
        "glow_hex": pal["glow_hex"],
        "outline_hex": pal["glow_hex"],
        "particle_hex": pal["particle_hex"],
        "primary_ass": pal["primary_1c"],
        "secondary_ass": pal["highlight_2c"],
        "outline_ass": pal["glow_3c"],
        "particle_ass": pal["particle_c"],
        "palette": pal,
        "preset": preset,
        "prompt": str(raw.get("prompt") or "").strip(),
    }


def normalize_color_palette_rule(raw: dict, idx: int) -> dict:
    return normalize_rule(raw, idx, default_prefix="Color Range")


def _rule_to_config(rule: dict, particle_density: float, motion_intensity: float) -> dict:
    return {
        "primary_hex": rule["primary_hex"],
        "highlight_hex": rule["highlight_hex"],
        "secondary_hex": rule["secondary_hex"],
        "glow_hex": rule["glow_hex"],
        "outline_hex": rule["outline_hex"],
        "particle_hex": rule["particle_hex"],
        "primary_ass": rule["primary_ass"],
        "secondary_ass": rule["secondary_ass"],
        "outline_ass": rule["outline_ass"],
        "particle_ass": rule["particle_ass"],
        "palette": dict(rule["palette"]),
        "particle_density": float(particle_density),
        "motion_intensity": float(motion_intensity),
        "rule_name": rule["name"],
        "prompt": rule.get("prompt", ""),
    }


def parse_line_range_spec(spec: str, max_line: int) -> tuple[set[int], list[str]]:
    """Parse 1-based line range specs such as '1-5', '1-3, 7, 9-end'."""
    nums: set[int] = set()
    warnings: list[str] = []
    raw = (spec or "").strip()
    if not raw:
        return nums, ["Empty Line Number Range specification."]

    for tok in [t.strip() for t in raw.split(",") if t.strip()]:
        m_range = re.fullmatch(r"(\d+)\s*-\s*(\d+|end|\*)", tok, flags=re.IGNORECASE)
        if m_range:
            a = int(m_range.group(1))
            b_raw = m_range.group(2).lower()
            b = max_line if b_raw in ("end", "*") else int(b_raw)
            lo, hi = (a, b) if a <= b else (b, a)
            if hi < 1 or lo > max_line:
                warnings.append(f"Range '{tok}' is outside available lines 1..{max_line}.")
                continue
            for n in range(max(1, lo), min(max_line, hi) + 1):
                nums.add(n)
        elif tok.isdigit():
            n = int(tok)
            if 1 <= n <= max_line:
                nums.add(n)
            else:
                warnings.append(f"Line #{n} is outside available lines 1..{max_line}.")
        else:
            warnings.append(f"Unrecognized range token '{tok}' (expected e.g. '1-5' or '1, 3, 6-10').")
    return nums, warnings


def resolve_line_rule_assignments(lines: list[KaraokeLine], rules: list[dict]) -> dict:
    """Map every 1-based karaoke line to one rule, detecting overlaps & auto-assigning fallbacks."""
    norm_rules = [normalize_rule(r, i) for i, r in enumerate(rules or [])]
    max_line = len(lines)
    rule_by_id = {r["id"]: r for r in norm_rules}

    specific_matches: dict[int, list[dict]] = {i + 1: [] for i in range(max_line)}
    spec_warnings: list[str] = []
    catchall_rules: list[dict] = []

    for r in norm_rules:
        if r["target_mode"] == "All Remaining Unassigned Lines":
            catchall_rules.append(r)
        elif r["target_mode"] == "Line Number Range":
            matched_nums, w_list = parse_line_range_spec(r["target_value"], max_line)
            for w in w_list:
                spec_warnings.append(f"[{r['name']}] {w}")
            for n in sorted(matched_nums):
                specific_matches[n].append(r)
        elif r["target_mode"] == "ASS Style Name":
            wanted = {s.strip().lower() for s in r["target_value"].split(",") if s.strip()}
            if not wanted:
                spec_warnings.append(f"[{r['name']}] Empty ASS Style Name target.")
            for i, ln in enumerate(lines):
                if ln.style.strip().lower() in wanted:
                    specific_matches[i + 1].append(r)

    overlap_warnings: list[str] = list(spec_warnings)
    if len(catchall_rules) > 1:
        overlap_warnings.append(
            f"Multiple 'All Remaining Unassigned Lines' rules defined; using '{catchall_rules[0]['name']}'."
        )

    assignments: dict[int, dict] = {}
    unassigned_nums: list[int] = []

    for i in range(max_line):
        line_num = i + 1
        hits = specific_matches[line_num]
        if hits:
            if len(hits) > 1:
                names = ", ".join(f"'{h['name']}'" for h in hits)
                overlap_warnings.append(
                    f"Overlap on Line #{line_num} (matched {names}) -> applying first match '{hits[0]['name']}'."
                )
            assignments[line_num] = hits[0]
        else:
            unassigned_nums.append(line_num)

    auto_fallback_used = False
    fallback_rule = None
    if unassigned_nums:
        if catchall_rules:
            fallback_rule = catchall_rules[0]
        else:
            auto_fallback_used = True
            fallback_rule = normalize_rule(
                {
                    "id": "auto_fallback",
                    "name": "Auto Fallback (Unassigned Lines)",
                    "target_mode": "All Remaining Unassigned Lines",
                    "target_value": "",
                    "primary_hex": "#00F5FF",
                    "highlight_hex": "#FF4FD8",
                    "glow_hex": "#0284C7",
                    "particle_hex": "#A5F3FC",
                    "preset": "neon_glow_bounce",
                    "prompt": "Clean cyan neon glow on syllable hit",
                },
                len(norm_rules),
            )
            norm_rules.append(fallback_rule)
            rule_by_id[fallback_rule["id"]] = fallback_rule
        for line_num in unassigned_nums:
            assignments[line_num] = fallback_rule

    used_rule_ids: list[str] = []
    for i in range(max_line):
        rid = assignments[i + 1]["id"]
        if rid not in used_rule_ids:
            used_rule_ids.append(rid)

    return {
        "rules": norm_rules,
        "rule_by_id": rule_by_id,
        "assignments": assignments,
        "overlap_warnings": overlap_warnings,
        "unassigned_lines": unassigned_nums,
        "auto_fallback_used": auto_fallback_used,
        "used_rule_ids": used_rule_ids,
    }


def build_unified_ass_document(
    lines: list[KaraokeLine],
    styles: dict[str, StyleInfo],
    meta: ScriptMeta,
    line_assignments: dict[int, dict],
    rule_executors: dict[str, tuple],
    seed: int = 42,
) -> tuple[str, dict[int, int], dict[str, int]]:
    """Synthesize all lines chronologically across per-rule executors into a single unified .ass script."""
    rng = random.Random(seed)
    out_header = [
        "[Script Info]",
        f"Title: {meta.title} — AI KFX Studio",
        "ScriptType: v4.00+",
        f"WrapStyle: {meta.wrap_style}",
        f"ScaledBorderAndShadow: {meta.scaled_border_and_shadow}",
        f"PlayResX: {meta.play_res_x}",
        f"PlayResY: {meta.play_res_y}",
        "",
        "[V4+ Styles]",
        "Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, "
        "Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, "
        "Shadow, Alignment, MarginL, MarginR, MarginV, Encoding",
    ]
    for st in styles.values():
        if st.raw_line:
            out_header.append(st.raw_line)
        else:
            out_header.append(
                f"Style: {st.name},{st.fontname},{int(round(st.fontsize))},{st.primary_colour},{st.secondary_colour},"
                f"{st.outline_colour},{st.back_colour},{st.bold},{st.italic},{st.underline},{st.strikeout},"
                f"{st.scale_x:g},{st.scale_y:g},{st.spacing:g},{st.angle:g},{st.border_style},"
                f"{st.outline:g},{st.shadow:g},{st.alignment},{st.margin_l},{st.margin_r},{st.margin_v},{st.encoding}"
            )

    out_header += [
        "",
        "[Events]",
        "Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text",
    ]

    comments: list[str] = []
    fx_events: list[str] = []
    layer_counts: dict[int, int] = {}
    per_rule_counts: dict[str, int] = {}

    ordered_lines = sorted(lines, key=lambda ln: (ln.start_ms, ln.index))
    for line in ordered_lines:
        line_num = line.index + 1
        comments.append(
            f"Comment: {line.layer},{KFXHelpers.ms_to_ass(line.start_ms)},{KFXHelpers.ms_to_ass(line.end_ms)},"
            f"{line.style},{line.actor},{line.margin_l},{line.margin_r},{line.margin_v},karaoke,{line.raw_text}"
        )
        rule = line_assignments[line_num]
        kfx_fn, rule_cfg, _src, _code = rule_executors[rule["id"]]
        st = styles.get(line.style) or next(iter(styles.values()))
        generated = invoke_kfx_function(kfx_fn, line, st, meta, KFXHelpers, rng, rule_cfg, rule["palette"])
        for ev in generated:
            if not ev.startswith("Dialogue:"):
                continue
            fx_events.append(ev)
            m_layer = re.match(r"Dialogue:\s*(-?\d+)\s*,", ev)
            ly = int(m_layer.group(1)) if m_layer else 0
            layer_counts[ly] = layer_counts.get(ly, 0) + 1
            r_name = rule["name"]
            per_rule_counts[r_name] = per_rule_counts.get(r_name, 0) + 1

    def sort_key(ev_line: str):
        parts = ev_line.split(":", 1)[1].lstrip().split(",", 3)
        return (int(parts[0]), _parse_ass_time_ms(parts[1]), _parse_ass_time_ms(parts[2]))

    fx_events.sort(key=sort_key)
    full_text = "\n".join(out_header + comments + fx_events) + "\n"
    return full_text, layer_counts, per_rule_counts


def _save_final_ass(ass_text: str, out_dir: Path | None) -> Path:
    save_root = Path(out_dir) if out_dir is not None else WORK_DIR
    save_root.mkdir(parents=True, exist_ok=True)
    ass_path = save_root / "final_kfx_karaoke.ass"
    ass_path.write_text(ass_text, encoding="utf-8")
    if Path("final_kfx_karaoke.ass").resolve() != ass_path.resolve():
        Path("final_kfx_karaoke.ass").write_text(ass_text, encoding="utf-8")
    return ass_path


def compile_shared_animation_kfx(
    lines: list[KaraokeLine],
    styles: dict[str, StyleInfo],
    meta: ScriptMeta,
    master_prompt: str,
    master_preset: str,
    palette_rules: list[dict],
    generation_mode: str,
    gemini_api_key: str,
    gemini_model: str,
    thinking_level: str,
    particle_density: float,
    motion_intensity: float,
    seed: int = 42,
    out_dir: Path | None = None,
) -> dict:
    """Mode A: Compile the master KFX function ONCE, then apply per-range 4-color palettes across lines 1..N."""
    if not lines:
        raise ValueError("KFX_LINES is empty. Run Cells 2.1 and 2.2 first.")

    norm_palettes = [normalize_color_palette_rule(r, i) for i, r in enumerate(palette_rules or [])]
    plan = resolve_line_rule_assignments(lines, norm_palettes)

    safe_preset = master_preset if master_preset in BUILTIN_PRESETS else "neon_glow_bounce"
    prompt_str = (master_prompt or "").strip()

    use_ai = str(generation_mode).startswith("AI Prompt-Driven")
    resolved_key = resolve_gemini_api_key(gemini_api_key) if use_ai else ""
    if use_ai and not resolved_key:
        print(
            "No GEMINI_API_KEY found (checked input box, GEMINI_API_KEY env var, and Colab Secrets). "
            f"Using Master Preset '{safe_preset}' instead."
        )
        use_ai = False

    sample_ln = lines[0]
    sample_st = styles.get(sample_ln.style) or next(iter(styles.values()))
    first_rule = plan["rule_by_id"][plan["used_rule_ids"][0]]
    master_cfg = _rule_to_config(first_rule, particle_density, motion_intensity)
    master_cfg["rule_name"] = "Master Shared KFX"
    master_cfg["prompt"] = prompt_str

    master_fn = None
    master_code = None
    master_src = f"Offline Master Preset ({safe_preset})"
    if use_ai and prompt_str:
        try:
            master_fn, master_code = generate_kfx_with_gemini(
                prompt_str,
                resolved_key,
                gemini_model,
                sample_ln,
                sample_st,
                meta,
                thinking_level=thinking_level,
                rule_cfg=master_cfg,
                shared_palette_mode=True,
            )
            master_src = f"Gemini AI Shared Master ({gemini_model})"
        except Exception as exc:
            print(f"Gemini generation failed for Master KFX ({exc}); falling back to preset '{safe_preset}'.")
            master_fn = None
            master_code = None

    if master_fn is None:
        master_fn = BUILTIN_PRESETS[safe_preset]

    rule_executors: dict[str, tuple] = {}
    for rid in plan["used_rule_ids"]:
        rule = plan["rule_by_id"][rid]
        rule_cfg = _rule_to_config(rule, particle_density, motion_intensity)
        rule_cfg["prompt"] = prompt_str or rule_cfg.get("prompt", "")
        rule["preset"] = safe_preset
        rule_executors[rid] = (master_fn, rule_cfg, master_src, master_code)

    ass_text, layer_counts, per_rule_counts = build_unified_ass_document(
        lines,
        styles,
        meta,
        plan["assignments"],
        rule_executors,
        seed=seed,
    )
    ass_path = _save_final_ass(ass_text, out_dir)

    return {
        "kfx_mode": MODE_A_LABEL,
        "ass_text": ass_text,
        "ass_path": ass_path,
        "line_assignments": plan["assignments"],
        "overlap_warnings": plan["overlap_warnings"],
        "unassigned_lines": plan["unassigned_lines"],
        "auto_fallback_used": plan["auto_fallback_used"],
        "used_rule_ids": plan["used_rule_ids"],
        "rule_by_id": plan["rule_by_id"],
        "rule_executors": rule_executors,
        "layer_counts": layer_counts,
        "per_rule_counts": per_rule_counts,
        "master_engine": master_src,
        "master_code": master_code,
        "master_preset": safe_preset,
        "master_prompt": prompt_str,
    }


def compile_multi_section_kfx(
    lines: list[KaraokeLine],
    styles: dict[str, StyleInfo],
    meta: ScriptMeta,
    rules: list[dict],
    generation_mode: str,
    gemini_api_key: str,
    gemini_model: str,
    thinking_level: str,
    particle_density: float,
    motion_intensity: float,
    seed: int = 42,
    out_dir: Path | None = None,
) -> dict:
    """Mode B: Compile an independent KFX animation function + 4-color palette per matched section rule."""
    if not lines:
        raise ValueError("KFX_LINES is empty. Run Cells 2.1 and 2.2 first.")

    plan = resolve_line_rule_assignments(lines, rules)
    use_ai = str(generation_mode).startswith("AI Prompt-Driven")
    resolved_key = resolve_gemini_api_key(gemini_api_key) if use_ai else ""
    if use_ai and not resolved_key:
        print(
            "No GEMINI_API_KEY found (checked input box, GEMINI_API_KEY env var, and Colab Secrets). "
            "Using each rule's fallback preset instead."
        )
        use_ai = False

    rule_executors: dict[str, tuple] = {}
    for rid in plan["used_rule_ids"]:
        rule = plan["rule_by_id"][rid]
        rule_cfg = _rule_to_config(rule, particle_density, motion_intensity)
        sample_ln = next(
            (ln for ln in lines if plan["assignments"][ln.index + 1]["id"] == rid),
            lines[0],
        )
        sample_st = styles.get(sample_ln.style) or next(iter(styles.values()))

        if use_ai and rule["prompt"]:
            try:
                fn, gen_code = generate_kfx_with_gemini(
                    rule["prompt"],
                    resolved_key,
                    gemini_model,
                    sample_ln,
                    sample_st,
                    meta,
                    thinking_level=thinking_level,
                    rule_cfg=rule_cfg,
                    shared_palette_mode=False,
                )
                rule_executors[rid] = (fn, rule_cfg, f"Gemini AI ({gemini_model})", gen_code)
                continue
            except Exception as exc:
                print(f"Gemini generation failed for '{rule['name']}' ({exc}); falling back to preset '{rule['preset']}'.")

        preset_fn = BUILTIN_PRESETS[rule["preset"]]
        rule_executors[rid] = (preset_fn, rule_cfg, f"Offline Preset ({rule['preset']})", None)

    ass_text, layer_counts, per_rule_counts = build_unified_ass_document(
        lines,
        styles,
        meta,
        plan["assignments"],
        rule_executors,
        seed=seed,
    )
    ass_path = _save_final_ass(ass_text, out_dir)

    return {
        "kfx_mode": MODE_B_LABEL,
        "ass_text": ass_text,
        "ass_path": ass_path,
        "line_assignments": plan["assignments"],
        "overlap_warnings": plan["overlap_warnings"],
        "unassigned_lines": plan["unassigned_lines"],
        "auto_fallback_used": plan["auto_fallback_used"],
        "used_rule_ids": plan["used_rule_ids"],
        "rule_by_id": plan["rule_by_id"],
        "rule_executors": rule_executors,
        "layer_counts": layer_counts,
        "per_rule_counts": per_rule_counts,
    }


def compile_active_kfx(
    kfx_mode: str,
    lines: list[KaraokeLine],
    styles: dict[str, StyleInfo],
    meta: ScriptMeta,
    master_prompt: str,
    master_preset: str,
    palette_rules: list[dict],
    section_rules: list[dict],
    generation_mode: str,
    gemini_api_key: str,
    gemini_model: str,
    thinking_level: str,
    particle_density: float,
    motion_intensity: float,
    seed: int = 42,
    out_dir: Path | None = None,
) -> dict:
    if str(kfx_mode).startswith("Mode A"):
        return compile_shared_animation_kfx(
            lines,
            styles,
            meta,
            master_prompt,
            master_preset,
            palette_rules,
            generation_mode,
            gemini_api_key,
            gemini_model,
            thinking_level,
            particle_density,
            motion_intensity,
            seed=seed,
            out_dir=out_dir,
        )
    return compile_multi_section_kfx(
        lines,
        styles,
        meta,
        section_rules,
        generation_mode,
        gemini_api_key,
        gemini_model,
        thinking_level,
        particle_density,
        motion_intensity,
        seed=seed,
        out_dir=out_dir,
    )


def parse_rules_json(json_str: str) -> list[dict]:
    data = json.loads(json_str)
    if not isinstance(data, list):
        raise ValueError("RULES_JSON_CONFIG must be a JSON array of rule objects.")
    return [normalize_rule(r, i) for i, r in enumerate(data)]


def parse_color_palettes_json(json_str: str) -> list[dict]:
    data = json.loads(json_str)
    if not isinstance(data, list):
        raise ValueError("COLOR_PALETTES_JSON_CONFIG must be a JSON array of color range objects.")
    return [normalize_color_palette_rule(r, i) for i, r in enumerate(data)]


# ==============================================================================
# Interactive ipywidgets KFX Rule & Palette Builder UI (Mode A & Mode B)
# ==============================================================================
class KFXRuleManagerUI:
    """Interactive Stage 2 KFX Studio with Mode A (Shared Animation + Color Ranges) & Mode B (Per-Section KFX)."""

    def __init__(
        self,
        initial_rules: list[dict],
        initial_palette_rules: list[dict] | None = None,
        initial_mode: str = MODE_A_LABEL,
        initial_master_preset: str = MASTER_PRESET,
        initial_master_prompt: str = MASTER_KFX_PROMPT,
        detected_styles: list[str] | None = None,
    ):
        import ipywidgets as widgets  # type: ignore

        self.widgets = widgets
        self.detected_styles = list(detected_styles or [])
        self._rule_seq = 0
        self._palette_seq = 0
        self.cards: list[dict] = []
        self.palette_cards: list[dict] = []

        start_mode = initial_mode if initial_mode in KFX_RULE_MODES else MODE_A_LABEL
        self.mode_toggle = widgets.ToggleButtons(
            options=KFX_RULE_MODES,
            value=start_mode,
            description="KFX Mode:",
            style={"button_width": "auto", "description_width": "80px"},
        )

        # ---- Mode A: Master KFX Prompt / Preset + Lightweight Color Range Cards ----
        self.master_preset_dd = widgets.Dropdown(
            options=list(BUILTIN_PRESETS.keys()),
            value=initial_master_preset if initial_master_preset in BUILTIN_PRESETS else "neon_glow_bounce",
            description="Master Preset (No-API / Fallback):",
            style={"description_width": "210px"},
            layout=widgets.Layout(width="430px"),
        )
        self.master_prompt_ta = widgets.Textarea(
            value=initial_master_prompt or MASTER_KFX_PROMPT,
            description="Master KFX Prompt:",
            placeholder="Describe ONE unified song-wide animation (colors automatically swap per line range)...",
            style={"description_width": "135px"},
            layout=widgets.Layout(width="99%", height="76px"),
        )
        master_box = widgets.VBox(
            [
                widgets.HTML(
                    "<div style='font-weight:700;font-size:13.5px;margin-bottom:4px;'>"
                    "Master KFX Animation (Compiled Once for All Lines — Dynamic <code>palette</code> Colors Swapped Per Range)</div>"
                ),
                self.master_preset_dd,
                self.master_prompt_ta,
            ],
            layout=widgets.Layout(
                border="1px solid #0284C7",
                border_radius="8px",
                padding="10px",
                width="100%",
                gap="6px",
                margin="0 0 8px 0",
            ),
        )
        self.palette_cards_box = widgets.VBox([], layout=widgets.Layout(width="100%", gap="10px"))
        self.mode_a_panel = widgets.VBox(
            [master_box, self.palette_cards_box],
            layout=widgets.Layout(width="100%", gap="6px"),
        )

        # ---- Mode B: Full Per-Section Rule Cards ----
        self.cards_box = widgets.VBox([], layout=widgets.Layout(width="100%", gap="10px"))
        self.mode_b_panel = widgets.VBox(
            [self.cards_box],
            layout=widgets.Layout(width="100%", gap="6px"),
        )

        # ---- Shared Action Toolbar ----
        self.add_palette_btn = widgets.Button(
            description="+ Add Color Range",
            button_style="info",
            icon="plus",
            layout=widgets.Layout(width="190px"),
        )
        self.add_btn = widgets.Button(
            description="+ Add KFX Section Rule",
            button_style="info",
            icon="plus",
            layout=widgets.Layout(width="210px"),
        )
        self.validate_btn = widgets.Button(
            description="Validate Line Mapping",
            button_style="",
            icon="check",
            layout=widgets.Layout(width="185px"),
        )
        self.export_json_btn = widgets.Button(
            description="Export Config JSON",
            button_style="",
            icon="code",
            layout=widgets.Layout(width="175px"),
        )
        self.generate_btn = widgets.Button(
            description="Generate Full KFX .ass",
            button_style="success",
            icon="magic",
            layout=widgets.Layout(width="210px"),
        )
        self.output_area = widgets.Output()

        self.mode_toggle.observe(self._on_mode_changed, names="value")
        self.add_palette_btn.on_click(self._on_add_palette_clicked)
        self.add_btn.on_click(self._on_add_clicked)
        self.validate_btn.on_click(self._on_validate_clicked)
        self.export_json_btn.on_click(self._on_export_json_clicked)
        self.generate_btn.on_click(self._on_generate_clicked)

        toolbar = widgets.HBox(
            [self.add_palette_btn, self.add_btn, self.validate_btn, self.export_json_btn, self.generate_btn],
            layout=widgets.Layout(gap="8px", margin="8px 0", flex_flow="row wrap"),
        )
        header = widgets.HTML(
            "<div style='font-weight:700;font-size:15px;margin-bottom:6px;'>"
            "Stage 2 KFX Studio — Mode A (Shared Animation + Color Ranges) &amp; Mode B (Unique Animation per Range)</div>"
        )
        self.container = widgets.VBox(
            [header, self.mode_toggle, self.mode_a_panel, self.mode_b_panel, toolbar, self.output_area]
        )

        pal_seed = initial_palette_rules if initial_palette_rules is not None else DEFAULT_COLOR_PALETTE_RULES
        for pr in pal_seed:
            self.add_palette_rule(pr, refresh=False)
        for r in initial_rules:
            self.add_rule(r, refresh=False)
        self._refresh_palette_cards()
        self._refresh_cards()
        self._sync_mode_visibility()

    def _build_card_common(self, rule: dict, default_name: str, rid: str, include_effect_fields: bool, on_delete) -> dict:
        w = self.widgets
        title_lbl = w.HTML("")
        name_input = w.Text(
            value=rule.get("name", default_name),
            description="Rule Name:",
            layout=w.Layout(width="360px"),
        )
        del_btn = w.Button(
            description="Delete Range" if not include_effect_fields else "Delete Rule",
            button_style="danger",
            icon="trash",
            layout=w.Layout(width="130px"),
        )

        mode_dd = w.Dropdown(
            options=TARGET_MODES,
            value=rule.get("target_mode", "Line Number Range")
            if rule.get("target_mode") in TARGET_MODES
            else "All Remaining Unassigned Lines",
            description="Target Mode:",
            layout=w.Layout(width="320px"),
        )
        target_input = w.Text(
            value=rule.get("target_value", ""),
            description="Target:",
            placeholder="e.g. 1-5 or Romaji_Verse",
            layout=w.Layout(width="270px"),
        )
        style_opts = ["(pick detected style)"] + self.detected_styles
        style_pick = w.Dropdown(
            options=style_opts,
            value="(pick detected style)",
            description="Styles:",
            layout=w.Layout(width="240px"),
        )

        p_init = _normalize_hex(rule.get("primary_hex", "#00F5FF"), "#00F5FF")
        h_init = _normalize_hex(rule.get("highlight_hex") or rule.get("secondary_hex") or "", "#FF4FD8")
        g_init = _normalize_hex(rule.get("glow_hex") or rule.get("outline_hex") or "", "#0284C7")
        pt_init = _normalize_hex(rule.get("particle_hex", ""), p_init)

        c1_pick = w.ColorPicker(
            description="Primary (\\1c):",
            value=p_init,
            concise=False,
            layout=w.Layout(width="215px"),
        )
        c2_pick = w.ColorPicker(
            description="Highlight (\\2c):",
            value=h_init,
            concise=False,
            layout=w.Layout(width="225px"),
        )
        c3_pick = w.ColorPicker(
            description="Glow/Border (\\3c):",
            value=g_init,
            concise=False,
            layout=w.Layout(width="235px"),
        )
        cp_pick = w.ColorPicker(
            description="Particle (\\p1):",
            value=pt_init,
            concise=False,
            layout=w.Layout(width="220px"),
        )
        ass_hex_lbl = w.HTML("")

        def _update_ass_hex(_change=None):
            a1 = KFXHelpers.ass_color(c1_pick.value)
            a2 = KFXHelpers.ass_color(c2_pick.value)
            a3 = KFXHelpers.ass_color(c3_pick.value)
            ap = KFXHelpers.ass_color(cp_pick.value)
            ass_hex_lbl.value = (
                f"<code style='font-size:12px;'>ASS BGR: \\1c{a1}  \\2c{a2}  \\3c{a3}  particle={ap}</code>"
            )

        def _update_mode(_change=None):
            m = mode_dd.value
            if m == "Line Number Range":
                target_input.disabled = False
                target_input.placeholder = "e.g. 1-5 or 1-3, 7-9"
                style_pick.layout.display = "none"
            elif m == "ASS Style Name":
                target_input.disabled = False
                target_input.placeholder = "e.g. Romaji_Chorus"
                style_pick.layout.display = "flex" if self.detected_styles else "none"
            else:
                target_input.disabled = True
                target_input.placeholder = "(auto-matches remaining lines)"
                style_pick.layout.display = "none"

        def _on_style_pick(change):
            val = change.get("new")
            if val and val != "(pick detected style)":
                target_input.value = val

        for cp in (c1_pick, c2_pick, c3_pick, cp_pick):
            cp.observe(_update_ass_hex, names="value")
        mode_dd.observe(_update_mode, names="value")
        style_pick.observe(_on_style_pick, names="value")

        _update_ass_hex()
        _update_mode()

        row_top = w.HBox(
            [title_lbl, name_input, del_btn],
            layout=w.Layout(justify_content="space-between", align_items="center", flex_flow="row wrap"),
        )
        row_target = w.HBox(
            [mode_dd, target_input, style_pick],
            layout=w.Layout(gap="8px", align_items="center", flex_flow="row wrap"),
        )
        row_colors = w.HBox(
            [c1_pick, c2_pick, c3_pick, cp_pick],
            layout=w.Layout(gap="8px", align_items="center", flex_flow="row wrap"),
        )
        children = [row_top, row_target, row_colors, ass_hex_lbl]

        preset_dd = None
        prompt_ta = None
        if include_effect_fields:
            preset_dd = w.Dropdown(
                options=list(BUILTIN_PRESETS.keys()),
                value=rule.get("preset", "neon_glow_bounce")
                if rule.get("preset") in BUILTIN_PRESETS
                else "neon_glow_bounce",
                description="Fallback Preset:",
                layout=w.Layout(width="320px"),
            )
            prompt_ta = w.Textarea(
                value=rule.get("prompt", ""),
                description="KFX Prompt:",
                placeholder="Describe unique animation, glow, bounce, and \\p1 particles for this section...",
                layout=w.Layout(width="98%", height="72px"),
            )
            children.extend([preset_dd, prompt_ta])

        card_box = w.VBox(
            children,
            layout=w.Layout(
                border="1px solid #475569",
                border_radius="8px",
                padding="10px",
                width="100%",
                gap="6px",
            ),
        )

        card = {
            "id": rid,
            "title_lbl": title_lbl,
            "name_input": name_input,
            "del_btn": del_btn,
            "mode_dd": mode_dd,
            "target_input": target_input,
            "style_pick": style_pick,
            "c1_pick": c1_pick,
            "c2_pick": c2_pick,
            "c3_pick": c3_pick,
            "cp_pick": cp_pick,
            "preset_dd": preset_dd,
            "prompt_ta": prompt_ta,
            "box": card_box,
        }
        del_btn.on_click(lambda _b: on_delete(card))
        return card

    def _sync_mode_visibility(self):
        is_a = self.get_active_mode().startswith("Mode A")
        self.mode_a_panel.layout.display = "flex" if is_a else "none"
        self.mode_b_panel.layout.display = "none" if is_a else "flex"
        self.add_palette_btn.layout.display = "inline-flex" if is_a else "none"
        self.add_btn.layout.display = "none" if is_a else "inline-flex"

    def _on_mode_changed(self, _change=None):
        self._sync_mode_visibility()

    def _refresh_cards_group(self, card_list: list[dict], box, label_prefix: str):
        for idx, c in enumerate(card_list):
            c["title_lbl"].value = f"<b>{label_prefix} #{idx + 1}</b>"
            c["del_btn"].disabled = len(card_list) <= 1
        box.children = tuple(c["box"] for c in card_list)

    def _refresh_palette_cards(self):
        self._refresh_cards_group(self.palette_cards, self.palette_cards_box, "Color Range")

    def _refresh_cards(self):
        self._refresh_cards_group(self.cards, self.cards_box, "Rule")

    def get_active_mode(self) -> str:
        val = self.mode_toggle.value
        return val if val in KFX_RULE_MODES else MODE_A_LABEL

    def get_master_config(self) -> dict:
        return {
            "master_preset": self.master_preset_dd.value,
            "master_prompt": (self.master_prompt_ta.value or "").strip(),
        }

    def add_palette_rule(self, rule: dict | None = None, refresh: bool = True):
        self._palette_seq += 1
        base = rule or {
            "name": f"Color Range #{len(self.palette_cards) + 1}",
            "target_mode": "Line Number Range",
            "target_value": "",
            "primary_hex": "#A855F7",
            "highlight_hex": "#F472B6",
            "glow_hex": "#4C1D95",
            "particle_hex": "#F9A8D4",
        }
        rid = str(base.get("id") or f"pal_{self._palette_seq}")
        card = self._build_card_common(
            base,
            f"Color Range #{self._palette_seq}",
            rid,
            include_effect_fields=False,
            on_delete=self.remove_palette_rule,
        )
        self.palette_cards.append(card)
        if refresh:
            self._refresh_palette_cards()

    def remove_palette_rule(self, card: dict):
        if len(self.palette_cards) <= 1:
            return
        self.palette_cards = [c for c in self.palette_cards if c is not card]
        self._refresh_palette_cards()

    def get_palette_rules(self) -> list[dict]:
        out = []
        master_cfg = self.get_master_config()
        for idx, c in enumerate(self.palette_cards):
            out.append(
                normalize_color_palette_rule(
                    {
                        "id": c["id"],
                        "name": c["name_input"].value,
                        "target_mode": c["mode_dd"].value,
                        "target_value": c["target_input"].value,
                        "primary_hex": c["c1_pick"].value,
                        "highlight_hex": c["c2_pick"].value,
                        "glow_hex": c["c3_pick"].value,
                        "particle_hex": c["cp_pick"].value,
                        "preset": master_cfg["master_preset"],
                        "prompt": master_cfg["master_prompt"],
                    },
                    idx,
                )
            )
        return out

    def add_rule(self, rule: dict | None = None, refresh: bool = True):
        self._rule_seq += 1
        base = rule or {
            "name": f"Section Rule #{len(self.cards) + 1}",
            "target_mode": "Line Number Range",
            "target_value": "",
            "primary_hex": "#A855F7",
            "secondary_hex": "#F472B6",
            "outline_hex": "#4C1D95",
            "particle_hex": "#F9A8D4",
            "preset": "sakura_petals",
            "prompt": "Glowing purple-pink bloom with drifting sakura petals \\p1",
        }
        rid = str(base.get("id") or f"rule_{self._rule_seq}")
        card = self._build_card_common(
            base,
            f"Section Rule #{self._rule_seq}",
            rid,
            include_effect_fields=True,
            on_delete=self.remove_rule,
        )
        self.cards.append(card)
        if refresh:
            self._refresh_cards()

    def remove_rule(self, card: dict):
        if len(self.cards) <= 1:
            return
        self.cards = [c for c in self.cards if c is not card]
        self._refresh_cards()

    def get_rules(self) -> list[dict]:
        out = []
        for idx, c in enumerate(self.cards):
            out.append(
                normalize_rule(
                    {
                        "id": c["id"],
                        "name": c["name_input"].value,
                        "target_mode": c["mode_dd"].value,
                        "target_value": c["target_input"].value,
                        "primary_hex": c["c1_pick"].value,
                        "secondary_hex": c["c2_pick"].value,
                        "outline_hex": c["c3_pick"].value,
                        "particle_hex": c["cp_pick"].value,
                        "preset": c["preset_dd"].value,
                        "prompt": c["prompt_ta"].value,
                    },
                    idx,
                )
            )
        return out

    def _on_add_palette_clicked(self, _btn):
        self.add_palette_rule(refresh=True)

    def _on_add_clicked(self, _btn):
        self.add_rule(refresh=True)

    def _on_validate_clicked(self, _btn):
        self.output_area.clear_output()
        with self.output_area:
            mode = self.get_active_mode()
            active_list = self.get_palette_rules() if mode.startswith("Mode A") else self.get_rules()
            plan = resolve_line_rule_assignments(KFX_LINES, active_list)
            print(f"Active Mode: {mode}")
            print_rule_assignment_summary(KFX_LINES, plan)

    def _on_export_json_clicked(self, _btn):
        self.output_area.clear_output()
        with self.output_area:
            if self.get_active_mode().startswith("Mode A"):
                mc = self.get_master_config()
                clean = [
                    {
                        k: r[k]
                        for k in ("name", "target_mode", "target_value", "primary_hex", "highlight_hex", "glow_hex", "particle_hex")
                    }
                    for r in self.get_palette_rules()
                ]
                print(f"MASTER_PRESET = {mc['master_preset']!r}")
                print(f"MASTER_KFX_PROMPT = {mc['master_prompt']!r}")
                print("\nCopy into COLOR_PALETTES_JSON_CONFIG:")
                print(json.dumps(clean, indent=2))
            else:
                clean = [
                    {
                        k: r[k]
                        for k in ("name", "target_mode", "target_value", "primary_hex", "secondary_hex", "outline_hex", "particle_hex", "preset", "prompt")
                    }
                    for r in self.get_rules()
                ]
                print("Copy into RULES_JSON_CONFIG:")
                print(json.dumps(clean, indent=2))

    def _on_generate_clicked(self, _btn):
        global MULTI_SECTION_RESULT, FINAL_ASS_TEXT, FINAL_ASS_PATH, LAYER_COUNTS, ACTIVE_RULES, ACTIVE_PALETTE_RULES, ACTIVE_MASTER, ACTIVE_MODE
        self.output_area.clear_output()
        with self.output_area:
            ACTIVE_MODE = self.get_active_mode()
            ACTIVE_MASTER = self.get_master_config()
            ACTIVE_PALETTE_RULES = self.get_palette_rules()
            ACTIVE_RULES = self.get_rules()
            MULTI_SECTION_RESULT = compile_active_kfx(
                ACTIVE_MODE,
                KFX_LINES,
                KFX_STYLES,
                KFX_META,
                ACTIVE_MASTER["master_prompt"],
                ACTIVE_MASTER["master_preset"],
                ACTIVE_PALETTE_RULES,
                ACTIVE_RULES,
                GENERATION_MODE,
                GEMINI_API_KEY,
                GEMINI_MODEL,
                GEMINI_THINKING_LEVEL,
                PARTICLE_DENSITY,
                MOTION_INTENSITY,
                seed=RANDOM_SEED,
                out_dir=WORK_DIR,
            )
            FINAL_ASS_TEXT = MULTI_SECTION_RESULT["ass_text"]
            FINAL_ASS_PATH = MULTI_SECTION_RESULT["ass_path"]
            LAYER_COUNTS = MULTI_SECTION_RESULT["layer_counts"]
            print_compilation_report(KFX_LINES, MULTI_SECTION_RESULT)
            try:
                dl_btn = self.widgets.Button(
                    description="Download final_kfx_karaoke.ass",
                    button_style="primary",
                    icon="download",
                    layout=self.widgets.Layout(width="250px"),
                )
                dl_btn.on_click(lambda _b: download_file_compat(FINAL_ASS_PATH))
                from IPython.display import display  # type: ignore
                display(dl_btn)
            except Exception:
                pass


def print_rule_assignment_summary(lines: list[KaraokeLine], plan: dict) -> None:
    for w in plan.get("overlap_warnings", []):
        print(f"[Rule Warning] {w}")
    if plan.get("auto_fallback_used"):
        print(
            f"[Auto Fallback] Unassigned lines {plan.get('unassigned_lines', [])} will use "
            "'Auto Fallback (Unassigned Lines)'."
        )
    assign_map = plan.get("assignments") or plan.get("line_assignments") or {}
    is_shared = str(plan.get("kfx_mode", "")).startswith("Mode A")
    print("\nLine-by-Line KFX Rule Mapping:")
    for ln in lines:
        n = ln.index + 1
        r = assign_map[n]
        tag = "Shared Master" if is_shared else r["preset"]
        pal = r["palette"]
        print(
            f"  Line #{n:>2} [{ln.style:<14}] -> {r['name']} "
            f"({tag}, \\1c{pal['primary_1c']} \\2c{pal['highlight_2c']} \\3c{pal['glow_3c']} \\p1={pal['particle_c']})"
        )


def print_compilation_report(lines: list[KaraokeLine], result: dict) -> None:
    print(f"KFX Mode              : {result.get('kfx_mode', MODE_B_LABEL)}")
    print_rule_assignment_summary(lines, result)
    print(f"\nSaved Unified KFX ASS : {result['ass_path']}")
    total_ev = sum(result["layer_counts"].values())
    print(f"Total FX Events       : {total_ev:,} across {len(lines)} lyric lines")
    print("Layer Breakdown       :", ", ".join(f"Layer {k}: {v}" for k, v in sorted(result["layer_counts"].items())))
    print("Per-Section Breakdown :", ", ".join(f"{k}: {v}" for k, v in result["per_rule_counts"].items()))
    if result.get("kfx_mode", "").startswith("Mode A"):
        print(f"Master Engine         : {result.get('master_engine')}")
        if result.get("master_code"):
            print("-" * 55)
            print(result["master_code"])
            print("-" * 55)
    else:
        for rid in result["used_rule_ids"]:
            r = result["rule_by_id"][rid]
            _fn, _cfg, src, gen_code = result["rule_executors"][rid]
            print(f"  [{r['name']}] Engine: {src}")
            if gen_code:
                print("-" * 55)
                print(gen_code)
                print("-" * 55)


KFX_RULE_MANAGER = None
MULTI_SECTION_RESULT = None


def get_active_kfx_mode() -> str:
    if CONFIG_MODE.startswith("Interactive UI") and KFX_RULE_MANAGER is not None:
        return KFX_RULE_MANAGER.get_active_mode()
    return KFX_RULE_MODE


def get_active_master_config() -> dict:
    if CONFIG_MODE.startswith("Interactive UI") and KFX_RULE_MANAGER is not None:
        return KFX_RULE_MANAGER.get_master_config()
    return {
        "master_preset": MASTER_PRESET,
        "master_prompt": (MASTER_KFX_PROMPT or "").strip(),
    }


def get_active_palette_rules() -> list[dict]:
    if CONFIG_MODE.startswith("Interactive UI") and KFX_RULE_MANAGER is not None:
        return KFX_RULE_MANAGER.get_palette_rules()
    return parse_color_palettes_json(COLOR_PALETTES_JSON_CONFIG)


def get_active_kfx_rules() -> list[dict]:
    if CONFIG_MODE.startswith("Interactive UI") and KFX_RULE_MANAGER is not None:
        return KFX_RULE_MANAGER.get_rules()
    return parse_rules_json(RULES_JSON_CONFIG)


if CONFIG_MODE.startswith("Interactive UI"):
    try:
        import ipywidgets as _ipyw  # type: ignore
        KFX_RULE_MANAGER = KFXRuleManagerUI(
            parse_rules_json(RULES_JSON_CONFIG),
            initial_palette_rules=parse_color_palettes_json(COLOR_PALETTES_JSON_CONFIG),
            initial_mode=KFX_RULE_MODE,
            initial_master_preset=MASTER_PRESET,
            initial_master_prompt=MASTER_KFX_PROMPT,
            detected_styles=globals().get("STAGE2_DETECTED_STYLES", []),
        )
        from IPython import get_ipython  # type: ignore
        if get_ipython() is not None or os.environ.get("KFX_FORCE_WIDGET_RENDER") == "1":
            from IPython.display import display  # type: ignore
            display(KFX_RULE_MANAGER.container)
        else:
            print("Rule Manager initialized (non-notebook mode: run Cell 2.4 or inspect get_active_kfx_mode()).")
    except Exception as exc:
        KFX_RULE_MANAGER = None
        print(f"Interactive UI unavailable ({exc}); using JSON configs (KFX_RULE_MODE={KFX_RULE_MODE}).")

ACTIVE_MODE = get_active_kfx_mode()
ACTIVE_MASTER = get_active_master_config()
ACTIVE_PALETTE_RULES = get_active_palette_rules()
ACTIVE_RULES = get_active_kfx_rules()
print(
    f"Active Stage 2 Mode: {ACTIVE_MODE} | Color Ranges: {len(ACTIVE_PALETTE_RULES)} | Section Rules: {len(ACTIVE_RULES)}. "
    "Click 'Generate Full KFX .ass' above or run Cell 2.4."
)

In [ ]:
#@title 2.4 Compile Unified final_kfx_karaoke.ass
def synthesize_full_ass(
    lines: list[KaraokeLine],
    styles: dict[str, StyleInfo],
    meta: ScriptMeta,
    kfx_fn,
    config: dict,
    seed: int = 42,
    palette: dict | None = None,
) -> tuple[str, dict[int, int]]:
    """Backwards-compatible single-rule wrapper delegating to build_unified_ass_document."""
    pal = resolve_palette(config, palette)
    cfg = dict(config or {})
    cfg["palette"] = pal
    single_rule = normalize_rule(
        {
            "id": "single",
            "name": cfg.get("rule_name", "Preview Rule"),
            "target_mode": "All Remaining Unassigned Lines",
            "primary_hex": pal["primary_hex"],
            "highlight_hex": pal["highlight_hex"],
            "glow_hex": pal["glow_hex"],
            "particle_hex": pal["particle_hex"],
            "preset": "neon_glow_bounce",
            "prompt": cfg.get("prompt", ""),
        },
        0,
    )
    assignments = {ln.index + 1: single_rule for ln in lines}
    executors = {"single": (kfx_fn, cfg, "custom", None)}
    ass_text, layer_counts, _ = build_unified_ass_document(
        lines, styles, meta, assignments, executors, seed=seed
    )
    return ass_text, layer_counts


if not KFX_LINES:
    raise ValueError("KFX_LINES is empty. Run Cells 2.1–2.3 first.")

ACTIVE_MODE = get_active_kfx_mode()
ACTIVE_MASTER = get_active_master_config()
ACTIVE_PALETTE_RULES = get_active_palette_rules()
ACTIVE_RULES = get_active_kfx_rules()

MULTI_SECTION_RESULT = compile_active_kfx(
    ACTIVE_MODE,
    KFX_LINES,
    KFX_STYLES,
    KFX_META,
    ACTIVE_MASTER["master_prompt"],
    ACTIVE_MASTER["master_preset"],
    ACTIVE_PALETTE_RULES,
    ACTIVE_RULES,
    GENERATION_MODE,
    GEMINI_API_KEY,
    GEMINI_MODEL,
    GEMINI_THINKING_LEVEL,
    PARTICLE_DENSITY,
    MOTION_INTENSITY,
    seed=RANDOM_SEED,
    out_dir=WORK_DIR,
)

FINAL_ASS_TEXT = MULTI_SECTION_RESULT["ass_text"]
FINAL_ASS_PATH = MULTI_SECTION_RESULT["ass_path"]
LAYER_COUNTS = MULTI_SECTION_RESULT["layer_counts"]

print_compilation_report(KFX_LINES, MULTI_SECTION_RESULT)
print("\nSample generated Dialogue lines:")
shown = 0
for ln in FINAL_ASS_TEXT.splitlines():
    if ln.startswith("Dialogue:"):
        print(" ", ln[:140] + ("..." if len(ln) > 140 else ""))
        shown += 1
        if shown >= 6:
            break

In [ ]:
#@title 2.5 Render Preview (FFmpeg + libass) & Download final_kfx_karaoke.ass
RENDER_PREVIEW = True #@param {type:"boolean"}
PREVIEW_LINE_NUMBER = 1 #@param {type:"integer"}
RENDER_ANIMATED_VIDEO = True #@param {type:"boolean"}
PREVIEW_WIDTH = 960 #@param {type:"integer"}
AUTO_DOWNLOAD = True #@param {type:"boolean"}


def build_zero_based_preview_ass(
    lines: list[KaraokeLine],
    styles: dict[str, StyleInfo],
    meta: ScriptMeta,
    kfx_fn,
    config: dict,
    line_index: int = 0,
    font_family_override: str | None = None,
    seed: int = 42,
    palette: dict | None = None,
) -> tuple[str, int, list[int]]:
    """Build a short 0-based single-line ASS clip for instant FFmpeg/libass frame & video preview."""
    idx = max(0, min(len(lines) - 1, int(line_index)))
    target = lines[idx]

    # Shift selected line so its start_ms sits cleanly at 400ms
    shift_ms = target.start_ms - 400
    preview_styles = {}
    for s_name, st in styles.items():
        st_copy = StyleInfo(**{**st.__dict__})
        if font_family_override:
            st_copy.fontname = font_family_override
            st_copy.raw_line = ""
        preview_styles[s_name] = st_copy

    line_copy = KaraokeLine(**{**target.__dict__})
    line_copy.start_ms = max(0, target.start_ms - shift_ms)
    line_copy.end_ms = max(line_copy.start_ms + 100, target.end_ms - shift_ms)
    shifted_syls = []
    for s in target.syllables:
        sc = Syllable(**{**s.__dict__})
        sc.start_ms = max(0, s.start_ms - shift_ms)
        sc.end_ms = max(sc.start_ms + 10, s.end_ms - shift_ms)
        shifted_syls.append(sc)
    line_copy.syllables = shifted_syls

    pal = resolve_palette(config, palette)
    preview_text, _ = synthesize_full_ass([line_copy], preview_styles, meta, kfx_fn, config, seed=seed, palette=pal)
    clip_dur_ms = line_copy.end_ms + 500
    non_sp = [s for s in shifted_syls if not s.is_space and s.text]
    if non_sp:
        mid_syl = non_sp[len(non_sp) // 2]
        t_pre = max(50, line_copy.start_ms + 40)
        t_mid = mid_syl.start_ms + min(90, max(20, mid_syl.duration_ms // 2))
        t_end = min(clip_dur_ms - 120, non_sp[-1].end_ms + 90)
    else:
        t_pre, t_mid, t_end = 250, clip_dur_ms // 2, clip_dur_ms - 150
    return preview_text, clip_dur_ms, [t_pre, t_mid, t_end]


def render_kfx_preview(
    lines: list[KaraokeLine],
    styles: dict[str, StyleInfo],
    meta: ScriptMeta,
    kfx_fn,
    config: dict,
    out_dir: Path,
    line_index: int = 0,
    font_path: str | None = None,
    font_family_override: str | None = None,
    width: int = 960,
    render_video: bool = True,
    seed: int = 42,
    palette: dict | None = None,
):
    from PIL import Image, ImageDraw  # type: ignore

    if not (HAS_FFMPEG and HAS_LIBASS):
        print("Preview skipped: ffmpeg with libass (`ass` filter) is not available.")
        return None, None

    prev_dir = out_dir / "preview"
    prev_dir.mkdir(parents=True, exist_ok=True)
    fonts_dir = prev_dir / "fonts"
    fonts_dir.mkdir(parents=True, exist_ok=True)
    if font_path and Path(font_path).is_file():
        shutil.copy2(font_path, fonts_dir / Path(font_path).name)

    pal = resolve_palette(config, palette)
    preview_ass, clip_dur_ms, sample_times_ms = build_zero_based_preview_ass(
        lines,
        styles,
        meta,
        kfx_fn,
        config,
        line_index=line_index,
        font_family_override=font_family_override,
        seed=seed,
        palette=pal,
    )
    ass_clip_path = prev_dir / "preview_clip.ass"
    ass_clip_path.write_text(preview_ass, encoding="utf-8")

    # Compute preview canvas height matching PlayRes aspect ratio
    aspect = meta.play_res_y / max(1, meta.play_res_x)
    h = max(240, int(round(width * aspect)) // 2 * 2)
    w = int(width) // 2 * 2
    dur_s = clip_dur_ms / 1000.0
    vf = "ass=preview_clip.ass:fontsdir=fonts"

    frame_paths = []
    labels = ["Pre-Hit / Entrance", "Mid-Line Peak + Particles", "Tail / Exit Fade"]
    for idx, t_ms in enumerate(sample_times_ms):
        png_path = prev_dir / f"frame_{idx}.png"
        cmd = [
            "ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
            "-f", "lavfi", "-i", f"color=c=0x101320:s={w}x{h}:r=30:d={dur_s:.2f}",
            "-ss", f"{t_ms / 1000.0:.3f}",
            "-vf", vf,
            "-frames:v", "1",
            str(png_path),
        ]
        subprocess.run(cmd, cwd=str(prev_dir), check=True)
        frame_paths.append(png_path)

    # Stitch 3 keyframes into one labeled contact sheet
    tile_w = w // 2 if w >= 960 else w
    tile_h = int(round(h * (tile_w / float(w))))
    pad = 12
    bar_h = 26
    sheet = Image.new("RGB", (tile_w * 3 + pad * 4, tile_h + bar_h + pad * 2), (18, 20, 32))
    draw = ImageDraw.Draw(sheet)
    for idx, (fp, lbl, t_ms) in enumerate(zip(frame_paths, labels, sample_times_ms)):
        im = Image.open(fp).convert("RGB").resize((tile_w, tile_h), Image.Resampling.LANCZOS)
        ox = pad + idx * (tile_w + pad)
        sheet.paste(im, (ox, pad))
        draw.text((ox + 4, pad + tile_h + 6), f"{lbl} ({t_ms}ms)", fill=(220, 232, 255))

    sheet_path = prev_dir / "kfx_contact_sheet.png"
    sheet.save(sheet_path)

    video_path = None
    if render_video:
        video_path = prev_dir / "kfx_preview.mp4"
        cmd_vid = [
            "ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
            "-f", "lavfi", "-i", f"color=c=0x101320:s={w}x{h}:r=30:d={dur_s:.2f}",
            "-vf", vf,
            "-c:v", "libx264", "-pix_fmt", "yuv420p", "-preset", "veryfast", "-crf", "20",
            str(video_path),
        ]
        subprocess.run(cmd_vid, cwd=str(prev_dir), check=True)

    return sheet_path, video_path


PREVIEW_SHEET_PATH = None
PREVIEW_VIDEO_PATH = None

if RENDER_PREVIEW:
    if not KFX_LINES:
        raise ValueError("KFX_LINES is empty.")
    _p_num = max(1, min(len(KFX_LINES), int(PREVIEW_LINE_NUMBER)))
    _p_idx = _p_num - 1
    if MULTI_SECTION_RESULT and _p_num in MULTI_SECTION_RESULT["line_assignments"]:
        _p_rule = MULTI_SECTION_RESULT["line_assignments"][_p_num]
        _p_fn, _p_cfg, _p_src, _ = MULTI_SECTION_RESULT["rule_executors"][_p_rule["id"]]
        _p_pal = _p_rule["palette"]
        print(
            f"Previewing Line #{_p_num} [{MULTI_SECTION_RESULT.get('kfx_mode', '')}] "
            f"using '{_p_rule['name']}' [{_p_src}] | "
            f"\\1c{_p_pal['primary_1c']} \\2c{_p_pal['highlight_2c']} \\3c{_p_pal['glow_3c']} \\p1={_p_pal['particle_c']}"
        )
    else:
        _r0 = normalize_rule(ACTIVE_RULES[0] if ACTIVE_RULES else DEFAULT_MULTI_SECTION_RULES[0], 0)
        _p_fn = BUILTIN_PRESETS[_r0["preset"]]
        _p_cfg = _rule_to_config(_r0, PARTICLE_DENSITY, MOTION_INTENSITY)
        _p_pal = _r0["palette"]

    PREVIEW_SHEET_PATH, PREVIEW_VIDEO_PATH = render_kfx_preview(
        KFX_LINES,
        KFX_STYLES,
        KFX_META,
        _p_fn,
        _p_cfg,
        WORK_DIR,
        line_index=_p_idx,
        font_path=STAGE2_FONT_PATH,
        font_family_override=STAGE2_UPLOADED_FONT_FAMILY if globals().get("STAGE2_FONT_IS_CUSTOM") else None,
        width=PREVIEW_WIDTH,
        render_video=RENDER_ANIMATED_VIDEO,
        seed=RANDOM_SEED,
        palette=_p_pal,
    )
    if PREVIEW_SHEET_PATH:
        print(f"Rendered Keyframe Contact Sheet : {PREVIEW_SHEET_PATH}")
        try:
            from IPython.display import HTML, Image as IPyImage, display  # type: ignore
            display(IPyImage(filename=str(PREVIEW_SHEET_PATH)))
            if PREVIEW_VIDEO_PATH and PREVIEW_VIDEO_PATH.is_file():
                import base64
                b64 = base64.b64encode(PREVIEW_VIDEO_PATH.read_bytes()).decode("ascii")
                display(
                    HTML(
                        f'<video width="720" controls autoplay loop muted>'
                        f'<source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'
                    )
                )
        except Exception:
            pass

if AUTO_DOWNLOAD:
    download_file_compat(FINAL_ASS_PATH)